In [203]:
import pandas as pd
import mysql.connector

print("Pandas:", pd.__version__)
print("MySQL Connector: Working")

Pandas: 3.0.5
MySQL Connector: Working


In [204]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.db_connection import get_connection

connection = get_connection()

print("DATABASE CONNECTION SUCCESSFUL")

DATABASE CONNECTION SUCCESSFUL


In [205]:
cursor = connection.cursor()

cursor.execute("SHOW TABLES")

tables = [table[0] for table in cursor.fetchall()]

print("Tables in database:")
for table in tables:
    print("-", table)

Tables in database:
- downtime
- employees
- inventory
- machines
- maintenance
- production
- production_logs
- production_targets
- quality
- sensors
- shifts
- suppliers


In [206]:
data = {}

for table in tables:
    query = f"SELECT * FROM `{table}`"
    data[table] = pd.read_sql(query, connection)

    print(f"{table}: {data[table].shape[0]} rows × {data[table].shape[1]} columns")

downtime: 431 rows × 6 columns
employees: 5 rows × 5 columns
inventory: 5 rows × 6 columns
machines: 5 rows × 6 columns
maintenance: 173 rows × 7 columns
production: 5475 rows × 7 columns
production_logs: 5475 rows × 6 columns
production_targets: 5475 rows × 4 columns
quality: 5475 rows × 6 columns


C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_245008\4262546910.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(query, connection)


sensors: 10950 rows × 6 columns
shifts: 3 rows × 5 columns
suppliers: 5 rows × 6 columns


In [207]:
for table, df in data.items():
    print("\n" + "=" * 60)
    print(f"TABLE: {table}")
    print("=" * 60)
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes)


TABLE: downtime
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

Data types:
downtime_id                 int64
machine_id                  int64
downtime_start     datetime64[us]
downtime_end       datetime64[us]
downtime_reason               str
downtime_hours            float64
dtype: object

TABLE: employees
Columns: ['employee_id', 'employee_name', 'department', 'role', 'shift']

Data types:
employee_id      int64
employee_name      str
department         str
role               str
shift              str
dtype: object

TABLE: inventory
Columns: ['inventory_id', 'material_name', 'quantity_available', 'reorder_level', 'unit', 'last_updated']

Data types:
inventory_id           int64
material_name            str
quantity_available     int64
reorder_level          int64
unit                     str
last_updated          object
dtype: object

TABLE: machines
Columns: ['machine_id', 'machine_name', 'machine_type', 'location',

In [208]:
summary = []

for table, df in data.items():
    summary.append({
        "table": table,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table,rows,columns,missing_values,duplicate_rows
0,downtime,431,6,0,0
1,employees,5,5,0,0
2,inventory,5,6,0,0
3,machines,5,6,0,0
4,maintenance,173,7,0,0
5,production,5475,7,0,0
6,production_logs,5475,6,0,0
7,production_targets,5475,4,0,0
8,quality,5475,6,0,0
9,sensors,10950,6,0,0


In [209]:
important_tables = [
    "machines",
    "production",
    "production_targets",
    "quality",
    "maintenance",
    "downtime",
    "sensors"
]

for table in important_tables:
    print("\n" + "=" * 70)
    print(f"TABLE: {table}")
    print("=" * 70)
    display(data[table])


TABLE: machines


,machine_id,machine_name,machine_type,location,status,installation_date
0,1,PCB Assembly Line 01,Assembly Line,Production Floor A,Operational,2022-05-10
1,2,PCB Assembly Line 02,Assembly Line,Production Floor A,Operational,2022-08-15
2,3,CNC Precision Unit,CNC Machine,Production Floor B,Maintenance,2021-03-20
3,4,Testing Station 01,Testing Equipment,Quality Floor,Operational,2023-01-12
4,5,Testing Station 02,Testing Equipment,Quality Floor,Operational,2023-04-18



TABLE: production


,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours
0,1,1,2025-08-24,Morning,969,59,8.0
1,2,1,2025-08-24,Afternoon,944,48,8.0
2,3,1,2025-08-24,Night,940,13,8.0
3,4,2,2025-08-24,Morning,890,62,8.0
4,5,2,2025-08-24,Afternoon,963,11,8.0
...,...,...,...,...,...,...,...
5470,5471,4,2026-08-23,Afternoon,921,61,8.0
5471,5472,4,2026-08-23,Night,987,63,8.0
5472,5473,5,2026-08-23,Morning,925,59,8.0
5473,5474,5,2026-08-23,Afternoon,883,29,8.0



TABLE: production_targets


...

,maintenance_id,equipment_id,maintenance_date,maintenance_type,maintenance_status,downtime_hours,maintenance_cost
0,1,4,2025-09-01,Corrective,Completed,5.03,12134.92
1,2,4,2025-09-05,Preventive,Completed,2.43,5757.39
2,3,3,2025-09-06,Routine Inspection,Completed,1.71,3505.76
3,4,2,2025-09-07,Preventive,Completed,2.97,2633.45
4,5,1,2025-09-10,Preventive,In Progress,2.46,3943.19
...,...,...,...,...,...,...,...
168,169,5,2026-08-14,Preventive,In Progress,2.48,3780.97
169,170,2,2026-08-15,Preventive,In Progress,1.69,4069.43
170,171,3,2026-08-15,Routine Inspection,Completed,1.20,2612.92
171,172,3,2026-08-20,Preventive,Completed,2.97,2876.86



TABLE: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_reason,downtime_hours
0,1,3,2025-08-25 06:00:00,2025-08-25 09:57:36,Material Shortage,3.96
1,2,1,2025-08-26 17:00:00,2025-08-26 18:59:24,Mechanical Failure,1.99
2,3,2,2025-08-27 19:00:00,2025-08-27 20:01:48,Material Shortage,1.03
3,4,3,2025-08-27 08:00:00,2025-08-27 11:22:48,Mechanical Failure,3.38
4,5,4,2025-08-27 15:00:00,2025-08-27 16:49:48,Machine Adjustment,1.83
...,...,...,...,...,...,...
426,427,3,2026-08-19 07:00:00,2026-08-19 09:29:24,Mechanical Failure,2.49
427,428,2,2026-08-20 17:00:00,2026-08-20 18:57:00,Material Shortage,1.95
428,429,5,2026-08-20 21:00:00,2026-08-20 23:28:12,Testing Error,2.47
429,430,3,2026-08-21 11:00:00,2026-08-21 11:54:00,Mechanical Failure,0.90



TABLE: sensors


,sensor_id,machine_id,sensor_type,sensor_value,unit,recorded_at
0,1,1,Temperature,66.49,°C,2025-08-24 08:00:00
1,2,1,Vibration,3.13,mm/s,2025-08-24 08:00:00
2,3,1,Temperature,62.62,°C,2025-08-24 14:00:00
3,4,1,Vibration,3.21,mm/s,2025-08-24 14:00:00
4,5,1,Temperature,65.77,°C,2025-08-24 22:00:00
...,...,...,...,...,...,...
10945,10946,5,Vibration,2.43,mm/s,2026-08-23 08:00:00
10946,10947,5,Temperature,62.15,°C,2026-08-23 14:00:00
10947,10948,5,Vibration,3.59,mm/s,2026-08-23 14:00:00
10948,10949,5,Temperature,67.43,°C,2026-08-23 22:00:00


In [210]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"NUMERIC SUMMARY: {table}")
    print("=" * 60)
    display(data[table].describe())


NUMERIC SUMMARY: machines


,machine_id
count,5.000000
mean,3.000000
std,1.581139
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000



NUMERIC SUMMARY: production


,production_id,machine_id,units_produced,units_rejected,production_time_hours
count,5475.000000,5475.000000,5475.000000,5475.000000,5475.0
mean,2738.000000,3.000000,912.130228,40.011872,8.0
std,1580.640693,1.414343,73.804539,18.788957,0.0
min,1.000000,1.000000,670.000000,7.000000,8.0
25%,1369.500000,2.000000,867.000000,24.000000,8.0
50%,2738.000000,3.000000,924.000000,39.000000,8.0
75%,4106.500000,4.000000,964.000000,55.000000,8.0
max,5475.000000,5.000000,1134.000000,84.000000,8.0



NUMERIC SUMMARY: production_targets


...

,maintenance_id,equipment_id,downtime_hours,maintenance_cost
count,173.000000,173.000000,173.000000,173.000000
mean,87.000000,2.907514,2.173526,5065.121098
std,50.084928,1.352260,1.239136,3205.217792
min,1.000000,1.000000,0.520000,1540.440000
25%,44.000000,2.000000,1.280000,2889.510000
50%,87.000000,3.000000,1.850000,3859.840000
75%,130.000000,4.000000,2.680000,5725.370000
max,173.000000,5.000000,5.870000,14899.450000



NUMERIC SUMMARY: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_hours
count,431.000000,431.000000,431,431,431.000000
mean,216.000000,2.883991,2026-02-27 08:21:17.958236,2026-02-27 10:35:35.443155,2.238190
min,1.000000,1.000000,2025-08-25 06:00:00,2025-08-25 09:57:36,0.500000
25%,108.500000,2.000000,2025-11-30 17:00:00,2025-11-30 19:34:48,1.395000
50%,216.000000,3.000000,2026-02-27 08:00:00,2026-02-27 09:15:36,2.210000
75%,323.500000,4.000000,2026-06-01 19:30:00,2026-06-01 21:26:06,3.110000
max,431.000000,5.000000,2026-08-22 07:00:00,2026-08-22 10:32:24,3.990000
std,124.563237,1.279716,NaN,NaN,0.994685



NUMERIC SUMMARY: sensors


,sensor_id,machine_id,sensor_value,recorded_at
count,10950.000000,10950.000000,10950.000000,10950
mean,5475.500000,3.000000,36.797256,2026-02-22 14:40:00
min,1.000000,1.000000,0.500000,2025-08-24 08:00:00
25%,2738.250000,2.000000,2.910000,2025-11-23 08:00:00
50%,5475.500000,3.000000,30.605000,2026-02-22 14:00:00
75%,8212.750000,4.000000,68.570000,2026-05-24 22:00:00
max,10950.000000,5.000000,91.030000,2026-08-23 22:00:00
std,3161.137058,1.414278,33.951788,NaN


In [211]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"CATEGORICAL VALUES: {table}")
    print("=" * 60)

    for column in data[table].select_dtypes(include=["object", "str"]).columns:
        print(f"\n{column}:")
        print(data[table][column].unique())


CATEGORICAL VALUES: machines

machine_name:
<StringArray>
['PCB Assembly Line 01', 'PCB Assembly Line 02',   'CNC Precision Unit',
   'Testing Station 01',   'Testing Station 02']
Length: 5, dtype: str

machine_type:
<StringArray>
['Assembly Line', 'CNC Machine', 'Testing Equipment']
Length: 3, dtype: str

location:
<StringArray>
['Production Floor A', 'Production Floor B', 'Quality Floor']
Length: 3, dtype: str

status:
<StringArray>
['Operational', 'Maintenance']
Length: 2, dtype: str

installation_date:
[datetime.date(2022, 5, 10) datetime.date(2022, 8, 15)
 datetime.date(2021, 3, 20) datetime.date(2023, 1, 12)
 datetime.date(2023, 4, 18)]

CATEGORICAL VALUES: production

production_date:
[datetime.date(2025, 8, 24) datetime.date(2025, 8, 25)
 datetime.date(2025, 8, 26) datetime.date(2025, 8, 27)
 datetime.date(2025, 8, 28) datetime.date(2025, 8, 29)
 datetime.date(2025, 8, 30) datetime.date(2025, 8, 31)
 datetime.date(2025, 9, 1) datetime.date(2025, 9, 2)
 datetime.date(2025, 9, 3

In [212]:
print("Machines referenced in production:")
print(sorted(data["production"]["machine_id"].unique()))

print("\nMachines in machines table:")
print(sorted(data["machines"]["machine_id"].unique()))

print("\nProduction IDs referenced in quality:")
print(sorted(data["quality"]["production_id"].unique()))

print("\nProduction IDs in production table:")
print(sorted(data["production"]["production_id"].unique()))

print("\nMachines referenced in downtime:")
print(sorted(data["downtime"]["machine_id"].unique()))

print("\nMachines referenced in sensors:")
print(sorted(data["sensors"]["machine_id"].unique()))

Machines referenced in production:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Machines in machines table:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Production IDs referenced in quality:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.

In [213]:
# ============================================================
# DATA CLEANING PIPELINE
# CHECK FOR  MISSING VALUES
# ============================================================

print("MISSING VALUE CHECK")
print("=" * 70)

for table, df in data.items():
    print(f"\nTABLE: {table}")
    print("-" * 50)

    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])

MISSING VALUE CHECK

TABLE: downtime
--------------------------------------------------
No missing values found.

TABLE: employees
--------------------------------------------------
No missing values found.

TABLE: inventory
--------------------------------------------------
No missing values found.

TABLE: machines
--------------------------------------------------
No missing values found.

TABLE: maintenance
--------------------------------------------------
No missing values found.

TABLE: production
--------------------------------------------------
No missing values found.

TABLE: production_logs
--------------------------------------------------
No missing values found.

TABLE: production_targets
--------------------------------------------------
No missing values found.

TABLE: quality
--------------------------------------------------
No missing values found.

TABLE: sensors
--------------------------------------------------
No missing values found.

TABLE: shifts
-------------

In [214]:
# ============================================================
# CHECK FOR DUPLICATE ROWS
# ============================================================

print("DUPLICATE VALUE CHECK")
print("=" * 70)

for table, df in data.items():
    duplicate_count = df.duplicated().sum()

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if duplicate_count == 0:
        print("No duplicate rows found.")
    else:
        print(f"Duplicate rows found: {duplicate_count}")

DUPLICATE VALUE CHECK

TABLE: downtime
--------------------------------------------------
No duplicate rows found.

TABLE: employees
--------------------------------------------------
No duplicate rows found.

TABLE: inventory
--------------------------------------------------
No duplicate rows found.

TABLE: machines
--------------------------------------------------
No duplicate rows found.

TABLE: maintenance
--------------------------------------------------
No duplicate rows found.

TABLE: production
--------------------------------------------------
No duplicate rows found.

TABLE: production_logs
--------------------------------------------------
No duplicate rows found.

TABLE: production_targets
--------------------------------------------------
No duplicate rows found.

TABLE: quality
--------------------------------------------------
No duplicate rows found.

TABLE: sensors
--------------------------------------------------
No duplicate rows found.

TABLE: shifts
-----------

In [215]:
# ============================================================
# CHECK FOR INVALID / NEGATIVE VALUES
# ============================================================

print("INVALID / NEGATIVE VALUE CHECK")
print("=" * 70)

numeric_columns = [
    "units_produced",
    "units_rejected",
    "production_time_hours",
    "target_quantity",
    "defect_count",
    "sensor_value",
    "downtime_hours",
    "maintenance_cost"
]

for table, df in data.items():
    print(f"\nTABLE: {table}")
    print("-" * 50)

    found_invalid = False

    for column in numeric_columns:
        if column in df.columns:
            invalid_count = (df[column] < 0).sum()

            if invalid_count > 0:
                print(f"{column}: {invalid_count} negative values")
                found_invalid = True

    if not found_invalid:
        print("No negative values found.")

INVALID / NEGATIVE VALUE CHECK

TABLE: downtime
--------------------------------------------------
No negative values found.

TABLE: employees
--------------------------------------------------
No negative values found.

TABLE: inventory
--------------------------------------------------
No negative values found.

TABLE: machines
--------------------------------------------------
No negative values found.

TABLE: maintenance
--------------------------------------------------
No negative values found.

TABLE: production
--------------------------------------------------
No negative values found.

TABLE: production_logs
--------------------------------------------------
No negative values found.

TABLE: production_targets
--------------------------------------------------
No negative values found.

TABLE: quality
--------------------------------------------------
No negative values found.

TABLE: sensors
--------------------------------------------------
No negative values found.

TABLE:

In [216]:
print("Production rows:", len(data["production"]))
print("Production targets rows:", len(data["production_targets"]))
print("Quality rows:", len(data["quality"]))
print("Sensors rows:", len(data["sensors"]))

Production rows: 5475
Production targets rows: 5475
Quality rows: 5475
Sensors rows: 10950


In [217]:




# CHECK INVALID NUMERIC VALUES
# ============================================================

print("INVALID NUMERIC VALUE CHECK")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    found_invalid = False

    for col in df.columns:

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Skip timedelta columns
        if pd.api.types.is_timedelta64_dtype(df[col]):
            continue

        # Check only actual numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            negative_count = (df[col] < 0).sum()

            if negative_count > 0:
                print(f"{col}: {negative_count} negative values")
                found_invalid = True

    if not found_invalid:
        print("No negative numeric values found.")

INVALID NUMERIC VALUE CHECK

TABLE: downtime
--------------------------------------------------
No negative numeric values found.

TABLE: employees
--------------------------------------------------
No negative numeric values found.

TABLE: inventory
--------------------------------------------------
No negative numeric values found.

TABLE: machines
--------------------------------------------------
No negative numeric values found.

TABLE: maintenance
--------------------------------------------------
No negative numeric values found.

TABLE: production
--------------------------------------------------
No negative numeric values found.

TABLE: production_logs
--------------------------------------------------
No negative numeric values found.

TABLE: production_targets
--------------------------------------------------
No negative numeric values found.

TABLE: quality
--------------------------------------------------
No negative numeric values found.

TABLE: sensors
---------------

In [218]:

# # ============================================================
# CHECK DATE AND TIME VALUES
# ============================================================

print("DATE / TIME VALIDATION")
print("=" * 70)

date_columns = [
    "production_date",
    "target_date",
    "inspection_date",
    "recorded_at",
    "downtime_start",
    "downtime_end",
    "maintenance_date"
]

for table, df in data.items():
    print(f"\nTABLE: {table}")
    print("-" * 50)

    found_date = False

    for column in date_columns:
        if column in df.columns:
            found_date = True

            dates = pd.to_datetime(df[column], errors="coerce")

            invalid_dates = dates.isna().sum()
            future_dates = (dates > pd.Timestamp.today()).sum()

            print(f"{column}:")
            print(f"  Invalid dates: {invalid_dates}")
            print(f"  Future dates: {future_dates}")

    if not found_date:
        print("No date/time columns.")

DATE / TIME VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_start:
  Invalid dates: 0
  Future dates: 0
downtime_end:
  Invalid dates: 0
  Future dates: 0

TABLE: employees
--------------------------------------------------
No date/time columns.

TABLE: inventory
--------------------------------------------------
No date/time columns.

TABLE: machines
--------------------------------------------------
No date/time columns.

TABLE: maintenance
--------------------------------------------------
maintenance_date:
  Invalid dates: 0
  Future dates: 0

TABLE: production
--------------------------------------------------
production_date:
  Invalid dates: 0
  Future dates: 0

TABLE: production_logs
--------------------------------------------------
No date/time columns.

TABLE: production_targets
--------------------------------------------------
target_date:
  Invalid dates: 0
  Future dates: 0

TABLE: quality
------------------------------------------

In [219]:
print("BUSINESS RULE VALIDATION")
print("=" * 70)

# Production checks
production = data["production"]

print("\nPRODUCTION")
print("-" * 50)

if "units_produced" in production.columns:
    print("Negative production:",
          (production["units_produced"] < 0).sum())

if "target_quantity" in production.columns:
    print("Negative target quantity:",
          (production["target_quantity"] < 0).sum())

# Quality checks
quality = data["quality"]

print("\nQUALITY")
print("-" * 50)

if "defect_quantity" in quality.columns:
    print("Negative defects:",
          (quality["defect_quantity"] < 0).sum())

# Downtime checks
downtime = data["downtime"]

print("\nDOWNTIME")
print("-" * 50)

if "downtime_start" in downtime.columns and "downtime_end" in downtime.columns:
    invalid_downtime = (
        downtime["downtime_end"] < downtime["downtime_start"]
    ).sum()

    print("Invalid downtime periods:", invalid_downtime)

# Production target checks
targets = data["production_targets"]

print("\nPRODUCTION TARGETS")
print("-" * 50)

numeric_target_cols = targets.select_dtypes(include="number").columns

for col in numeric_target_cols:
    print(f"{col} negative values:", (targets[col] < 0).sum())

BUSINESS RULE VALIDATION

PRODUCTION
--------------------------------------------------
Negative production: 0

QUALITY
--------------------------------------------------

DOWNTIME
--------------------------------------------------
Invalid downtime periods: 0

PRODUCTION TARGETS
--------------------------------------------------
target_id negative values: 0
machine_id negative values: 0
target_quantity negative values: 0


In [220]:
print("QUALITY DEFECT VALIDATION")
print("=" * 70)

quality = data["quality"]

print("Negative defect counts:",
      (quality["defect_count"] < 0).sum())

QUALITY DEFECT VALIDATION
Negative defect counts: 0


In [221]:
print("DATA TYPE VALIDATION")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    for column in df.columns:
        print(f"{column}: {df[column].dtype}")

DATA TYPE VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_id: int64
machine_id: int64
downtime_start: datetime64[us]
downtime_end: datetime64[us]
downtime_reason: str
downtime_hours: float64

TABLE: employees
--------------------------------------------------
employee_id: int64
employee_name: str
department: str
role: str
shift: str

TABLE: inventory
--------------------------------------------------
inventory_id: int64
material_name: str
quantity_available: int64
reorder_level: int64
unit: str
last_updated: object

TABLE: machines
--------------------------------------------------
machine_id: int64
machine_name: str
machine_type: str
location: str
status: str
installation_date: object

TABLE: maintenance
--------------------------------------------------
maintenance_id: int64
equipment_id: int64
maintenance_date: object
maintenance_type: str
maintenance_status: str
downtime_hours: float64
maintenance_cost: float64

TABLE: production
------------

In [222]:
print("OUTLIER CHECK")
print("=" * 70)

for table, df in data.items():

    numeric_cols = df.select_dtypes(include=["number"]).columns

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        continue

    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = (
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ).sum()

        print(f"{col}: {outliers} potential outliers")

OUTLIER CHECK

TABLE: downtime
--------------------------------------------------
downtime_id: 0 potential outliers
machine_id: 0 potential outliers
downtime_hours: 0 potential outliers

TABLE: employees
--------------------------------------------------
employee_id: 0 potential outliers

TABLE: inventory
--------------------------------------------------
inventory_id: 0 potential outliers
quantity_available: 0 potential outliers
reorder_level: 0 potential outliers

TABLE: machines
--------------------------------------------------
machine_id: 0 potential outliers

TABLE: maintenance
--------------------------------------------------
maintenance_id: 0 potential outliers
equipment_id: 0 potential outliers
downtime_hours: 13 potential outliers
maintenance_cost: 19 potential outliers

TABLE: production
--------------------------------------------------
production_id: 0 potential outliers
machine_id: 0 potential outliers
units_produced: 38 potential outliers
units_rejected: 0 potential out

In [223]:
 #PRODUCTION EDA

production = data["production"]

print("PRODUCTION DATA")
print("=" * 70)

print("\nShape:")
print(production.shape)

print("\nColumns:")
print(production.columns.tolist())

print("\nFirst 5 rows:")
display(production.head())

print("\nStatistical Summary:")
display(production.describe())

PRODUCTION DATA

Shape:
(5475, 7)

Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

First 5 rows:


,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours
0,1,1,2025-08-24,Morning,969,59,8.0
1,2,1,2025-08-24,Afternoon,944,48,8.0
2,3,1,2025-08-24,Night,940,13,8.0
3,4,2,2025-08-24,Morning,890,62,8.0
4,5,2,2025-08-24,Afternoon,963,11,8.0



Statistical Summary:


,production_id,machine_id,units_produced,units_rejected,production_time_hours
count,5475.000000,5475.000000,5475.000000,5475.000000,5475.0
mean,2738.000000,3.000000,912.130228,40.011872,8.0
std,1580.640693,1.414343,73.804539,18.788957,0.0
min,1.000000,1.000000,670.000000,7.000000,8.0
25%,1369.500000,2.000000,867.000000,24.000000,8.0
50%,2738.000000,3.000000,924.000000,39.000000,8.0
75%,4106.500000,4.000000,964.000000,55.000000,8.0
max,5475.000000,5.000000,1134.000000,84.000000,8.0


In [224]:
# PRODUCTION ANALYSIS

print("PRODUCTION ANALYSIS")
print("=" * 70)

# Total production
print("\nTotal Units Produced:")
print(production["units_produced"].sum())

# Average production
print("\nAverage Units Produced:")
print(production["units_produced"].mean())

# Minimum production
print("\nMinimum Units Produced:")
print(production["units_produced"].min())

# Maximum production
print("\nMaximum Units Produced:")
print(production["units_produced"].max())

# Machine-wise production
print("\nMachine-wise Production:")
machine_production = (
    production.groupby("machine_id")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(machine_production)

PRODUCTION ANALYSIS

Total Units Produced:
4993913

Average Units Produced:
912.1302283105023

Minimum Units Produced:
670

Maximum Units Produced:
1134

Machine-wise Production:


,machine_id,sum,mean,min,max
0,1,1041452,951.097717,830,1091
1,2,985611,900.101370,784,1029
2,3,875771,799.790868,670,917
3,4,1074295,981.091324,859,1134
4,5,1016784,928.569863,785,1087


In [225]:
# SHIFT-WISE PRODUCTION EDA

print("SHIFT-WISE PRODUCTION ANALYSIS")
print("=" * 70)

shift_production = (
    production.groupby("shift")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(shift_production)

SHIFT-WISE PRODUCTION ANALYSIS


,shift,sum,mean,min,max
0,Afternoon,1665500,912.602740,682,1097
1,Morning,1663924,911.739178,683,1134
2,Night,1664489,912.048767,670,1070


In [226]:
# TARGET VS ACTUAL PRODUCTION

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

targets = data["production_targets"]

print("\nTarget Data:")
print("Shape:", targets.shape)
print("Columns:", targets.columns.tolist())

display(targets.head())

TARGET VS ACTUAL PRODUCTION

Target Data:
Shape: (5475, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']


,target_id,machine_id,target_date,target_quantity
0,1,1,2025-08-24,1002
1,2,1,2025-08-24,1005
2,3,1,2025-08-24,994
3,4,2,2025-08-24,873
4,5,2,2025-08-24,882


In [227]:
# TARGET VS ACTUAL PRODUCTION

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

# Production and target tables have the same row order.
# So we match each production row with the target row at the same position.

target_vs_actual = production.copy()

target_vs_actual["target_id"] = targets["target_id"].values
target_vs_actual["target_quantity"] = targets["target_quantity"].values

# Calculate target achievement
target_vs_actual["achievement_percent"] = (
    target_vs_actual["units_produced"]
    / target_vs_actual["target_quantity"]
) * 100

# Calculate difference between actual and target
target_vs_actual["difference"] = (
    target_vs_actual["units_produced"]
    - target_vs_actual["target_quantity"]
)

print("\nTarget vs Actual Dataset Shape:")
print(target_vs_actual.shape)

print("\nFirst 5 Rows:")
display(
    target_vs_actual[
        [
            "production_id",
            "machine_id",
            "production_date",
            "shift",
            "units_produced",
            "target_quantity",
            "achievement_percent",
            "difference"
        ]
    ].head()
)

print("\nOverall Target Quantity:")
print(target_vs_actual["target_quantity"].sum())

print("\nOverall Actual Production:")
print(target_vs_actual["units_produced"].sum())

print("\nOverall Achievement %:")
overall_achievement = (
    target_vs_actual["units_produced"].sum()
    / target_vs_actual["target_quantity"].sum()
) * 100

print(overall_achievement)

print("\nMachine-wise Target vs Actual:")

machine_target_actual = (
    target_vs_actual
    .groupby("machine_id")
    .agg(
        target_quantity=("target_quantity", "sum"),
        actual_production=("units_produced", "sum")
    )
    .reset_index()
)

machine_target_actual["achievement_percent"] = (
    machine_target_actual["actual_production"]
    / machine_target_actual["target_quantity"]
) * 100

machine_target_actual["difference"] = (
    machine_target_actual["actual_production"]
    - machine_target_actual["target_quantity"]
)

display(machine_target_actual)

TARGET VS ACTUAL PRODUCTION

Target vs Actual Dataset Shape:
(5475, 11)

First 5 Rows:


,production_id,machine_id,production_date,shift,units_produced,target_quantity,achievement_percent,difference
0,1,1,2025-08-24,Morning,969,1002,96.706587,-33
1,2,1,2025-08-24,Afternoon,944,1005,93.930348,-61
2,3,1,2025-08-24,Night,940,994,94.567404,-54
3,4,2,2025-08-24,Morning,890,873,101.947308,17
4,5,2,2025-08-24,Afternoon,963,882,109.183673,81



Overall Target Quantity:
5089338

Overall Actual Production:
4993913

Overall Achievement %:
98.12500171928058

Machine-wise Target vs Actual:


,machine_id,target_quantity,actual_production,achievement_percent,difference
0,1,1093419,1041452,95.247293,-51967
1,2,985483,985611,100.012989,128
2,3,875751,875771,100.002284,20
3,4,1095218,1074295,98.089604,-20923
4,5,1039467,1016784,97.817824,-22683


In [228]:
# QUALITY / DEFECT EDA

# QUALITY / DEFECT EDA

quality = data["quality"]

print("QUALITY DATA")
print("=" * 70)

print("Shape:", quality.shape)

print("Columns:", quality.columns.tolist())

print("\nFirst 5 rows:")
print(quality.head().to_string())

print("\nStatistical Summary:")
print(quality.describe().to_string())

QUALITY DATA
Shape: (5475, 6)
Columns: ['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

First 5 rows:
   quality_id  production_id inspection_date             defect_type  defect_count quality_status
0           1              1      2025-08-24           Solder Defect            20           Fail
1           2              2      2025-08-24           Solder Defect            16           Fail
2           3              3      2025-08-24       Missing Component             3           Pass
3           4              4      2025-08-24  Component Misplacement            21           Fail
4           5              5      2025-08-24             Overheating             2           Pass

Statistical Summary:
        quality_id  production_id  defect_count
count  5475.000000    5475.000000   5475.000000
mean   2738.000000    2738.000000     10.552329
std    1580.640693    1580.640693      6.127314
min       1.000000       1.000000      1.000

In [229]:
# QUALITY / DEFECT ANALYSIS

print("QUALITY / DEFECT ANALYSIS")
print("=" * 70)

# Quality status count
print("\nQuality Status:")
quality_status = quality["quality_status"].value_counts()

display(quality_status.to_frame("count"))

# Defect type count
print("\nDefect Type Distribution:")
defect_types = quality["defect_type"].value_counts()

display(defect_types.to_frame("count"))

# Average defect count by quality status
print("\nAverage Defect Count by Quality Status:")

avg_defects = (
    quality.groupby("quality_status")["defect_count"]
    .agg(["count", "mean", "min", "max"])
    .reset_index()
)

display(avg_defects)

QUALITY / DEFECT ANALYSIS

Quality Status:


,count
quality_status,
Pass,4070
Fail,1405



Defect Type Distribution:


,count
defect_type,
Overheating,1166
Component Misplacement,1107
PCB Damage,1094
Missing Component,1081
Solder Defect,1027



Average Defect Count by Quality Status:


,quality_status,count,mean,min,max
0,Fail,1405,19.027046,15,32
1,Pass,4070,7.626781,1,14


In [230]:
targets = data["production_targets"]

print("TARGET TABLE")
print("=" * 70)

print("Shape:", targets.shape)

print("Columns:", targets.columns.tolist())

print("\nFirst 5 rows:")

print(targets.head().to_string())

TARGET TABLE
Shape: (5475, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']

First 5 rows:
   target_id  machine_id target_date  target_quantity
0          1           1  2025-08-24             1002
1          2           1  2025-08-24             1005
2          3           1  2025-08-24              994
3          4           2  2025-08-24              873
4          5           2  2025-08-24              882


In [231]:
#  QUALITY STATUS ANALYSIS

print("QUALITY STATUS ANALYSIS")
print("=" * 70)

print("\nQuality Status Counts:")
print(quality["quality_status"].value_counts().to_string())

print("\nQuality Status Percentage:")
print(
    (quality["quality_status"].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

QUALITY STATUS ANALYSIS

Quality Status Counts:
quality_status
Pass    4070
Fail    1405

Quality Status Percentage:
quality_status
Pass    74.34
Fail    25.66


In [232]:
# DEFECT TYPE ANALYSIS

print("DEFECT TYPE ANALYSIS")
print("=" * 70)

defect_analysis = (
    quality.groupby("defect_type")["defect_count"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDefect Type Summary:")
print(defect_analysis.to_string(index=False))

DEFECT TYPE ANALYSIS

Defect Type Summary:
           defect_type  count   sum      mean  min  max
           Overheating   1166 12419 10.650943    1   30
            PCB Damage   1094 11749 10.739488    1   32
Component Misplacement   1107 11557 10.439928    1   29
     Missing Component   1081 11454 10.595745    1   28
         Solder Defect   1027 10595 10.316456    1   28


In [233]:
# — MACHINE-WISE QUALITY ANALYSIS

print("MACHINE-WISE QUALITY ANALYSIS")
print("=" * 70)

# Connect quality data with production data to get machine_id
quality_machine = quality.merge(
    production[["production_id", "machine_id"]],
    on="production_id",
    how="left"
)

machine_quality = (
    quality_machine.groupby("machine_id")
    .agg(
        total_defects=("defect_count", "sum"),
        average_defects=("defect_count", "mean"),
        inspections=("quality_id", "count")
    )
    .reset_index()
)

machine_quality["defects_per_inspection"] = (
    machine_quality["total_defects"]
    / machine_quality["inspections"]
)

print("\nMachine-wise Quality Summary:")
print(machine_quality.to_string(index=False))

MACHINE-WISE QUALITY ANALYSIS



Machine-wise Quality Summary:
 machine_id  total_defects  average_defects  inspections  defects_per_inspection
          1          11987        10.947032         1095               10.947032
          2          11645        10.634703         1095               10.634703
          3           9949         9.085845         1095                9.085845
          4          12626        11.530594         1095               11.530594
          5          11567        10.563470         1095               10.563470


In [234]:
#   SENSOR EDA

sensors = data["sensors"]

print("SENSOR DATA")
print("=" * 70)

print("Shape:", sensors.shape)
print("Columns:", sensors.columns.tolist())

print("\nFirst 5 rows:")
print(sensors.head().to_string())

print("\nStatistical Summary:")
print(sensors.describe().to_string())

SENSOR DATA
Shape: (10950, 6)
Columns: ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

First 5 rows:
   sensor_id  machine_id  sensor_type  sensor_value  unit         recorded_at
0          1           1  Temperature         66.49    °C 2025-08-24 08:00:00
1          2           1    Vibration          3.13  mm/s 2025-08-24 08:00:00
2          3           1  Temperature         62.62    °C 2025-08-24 14:00:00
3          4           1    Vibration          3.21  mm/s 2025-08-24 14:00:00
4          5           1  Temperature         65.77    °C 2025-08-24 22:00:00

Statistical Summary:


          sensor_id    machine_id  sensor_value          recorded_at
count  10950.000000  10950.000000  10950.000000                10950
mean    5475.500000      3.000000     36.797256  2026-02-22 14:40:00
min        1.000000      1.000000      0.500000  2025-08-24 08:00:00
25%     2738.250000      2.000000      2.910000  2025-11-23 08:00:00
50%     5475.500000      3.000000     30.605000  2026-02-22 14:00:00
75%     8212.750000      4.000000     68.570000  2026-05-24 22:00:00
max    10950.000000      5.000000     91.030000  2026-08-23 22:00:00
std     3161.137058      1.414278     33.951788                  NaN


In [235]:
#  SENSOR TYPE ANALYSIS

print("SENSOR TYPE ANALYSIS")
print("=" * 70)

sensor_type_summary = (
    sensors.groupby("sensor_type")["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nSensor Type Summary:")
print(sensor_type_summary.to_string(index=False))

SENSOR TYPE ANALYSIS

Sensor Type Summary:
sensor_type  count      mean   min   max      std
Temperature   5475 70.400424 54.64 91.03 6.768219
  Vibration   5475  3.194088  0.50  6.57 1.044645


In [236]:
# MACHINE-WISE SENSOR ANALYSIS

print("MACHINE-WISE SENSOR ANALYSIS")
print("=" * 70)

machine_sensor_summary = (
    sensors.groupby(["machine_id", "sensor_type"])["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nMachine-wise Sensor Summary:")
print(machine_sensor_summary.to_string(index=False))

MACHINE-WISE SENSOR ANALYSIS

Machine-wise Sensor Summary:
 machine_id sensor_type  count      mean   min   max      std
          1 Temperature   1095 70.153726 61.62 79.79 3.082530
          1   Vibration   1095  3.001361  1.28  4.48 0.497661
          2 Temperature   1095 67.907342 58.83 76.69 3.006898
          2   Vibration   1095  2.802393  1.05  4.26 0.476934
          3 Temperature   1095 82.032420 71.40 91.03 3.032493
          3   Vibration   1095  4.998219  3.50  6.57 0.514230
          4 Temperature   1095 64.878685 54.64 76.18 3.004166
          4   Vibration   1095  2.461041  0.50  3.79 0.498736
          5 Temperature   1095 67.029945 57.94 77.81 2.974885
          5   Vibration   1095  2.707425  1.03  4.37 0.498583


In [237]:
# SENSOR TREND ANALYSIS

print("SENSOR TREND ANALYSIS")
print("=" * 70)

sensor_trends = (
    sensors.groupby(
        [sensors["recorded_at"].dt.date, "sensor_type"]
    )["sensor_value"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

print("\nDaily Sensor Trends:")
print(sensor_trends.to_string(index=False))

SENSOR TREND ANALYSIS

Daily Sensor Trends:
recorded_at sensor_type      mean   min   max
 2025-08-24 Temperature 69.137333 56.64 84.28
 2025-08-24   Vibration  3.206000  2.12  4.88
 2025-08-25 Temperature 70.766000 60.77 82.15
 2025-08-25   Vibration  3.077333  1.37  5.14
 2025-08-26 Temperature 69.531333 60.46 83.35
 2025-08-26   Vibration  3.213333  1.89  5.69
 2025-08-27 Temperature 71.707333 60.44 82.76
 2025-08-27   Vibration  3.216667  1.88  5.23
 2025-08-28 Temperature 71.698667 65.35 86.21
 2025-08-28   Vibration  3.031333  1.51  6.47
 2025-08-29 Temperature 69.512667 61.19 87.56
 2025-08-29   Vibration  3.156667  1.77  5.82
 2025-08-30 Temperature 70.028000 60.27 83.44
 2025-08-30   Vibration  3.228000  1.84  5.97
 2025-08-31 Temperature 70.904667 62.66 82.12
 2025-08-31   Vibration  3.323333  1.81  5.66
 2025-09-01 Temperature 70.827333 63.80 87.89
 2025-09-01   Vibration  3.043333  2.11  5.56
 2025-09-02 Temperature 71.474000 61.30 86.70
 2025-09-02   Vibration  3.112000  1

In [238]:
# DOWNTIME EDA

downtime = data["downtime"]

print("DOWNTIME DATA")
print("=" * 70)

print("Shape:", downtime.shape)
print("Columns:", downtime.columns.tolist())

print("\nFirst 5 rows:")
print(downtime.head().to_string())

print("\nStatistical Summary:")
print(downtime.describe().to_string())

DOWNTIME DATA
Shape: (431, 6)
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

First 5 rows:
   downtime_id  machine_id      downtime_start        downtime_end     downtime_reason  downtime_hours
0            1           3 2025-08-25 06:00:00 2025-08-25 09:57:36   Material Shortage            3.96
1            2           1 2025-08-26 17:00:00 2025-08-26 18:59:24  Mechanical Failure            1.99
2            3           2 2025-08-27 19:00:00 2025-08-27 20:01:48   Material Shortage            1.03
3            4           3 2025-08-27 08:00:00 2025-08-27 11:22:48  Mechanical Failure            3.38
4            5           4 2025-08-27 15:00:00 2025-08-27 16:49:48  Machine Adjustment            1.83

Statistical Summary:
       downtime_id  machine_id              downtime_start                downtime_end  downtime_hours
count   431.000000  431.000000                         431                         431      431.000000

In [239]:
# DOWNTIME REASON ANALYSIS

print("DOWNTIME REASON ANALYSIS")
print("=" * 70)

downtime_reason = (
    downtime.groupby("downtime_reason")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDowntime Reason Summary:")
print(downtime_reason.to_string(index=False))

DOWNTIME REASON ANALYSIS

Downtime Reason Summary:
   downtime_reason  count    sum     mean  min  max
Mechanical Failure    145 306.52 2.113931 0.53 3.98
 Material Shortage     66 147.43 2.233788 0.52 3.96
  Electrical Issue     61 138.61 2.272295 0.62 3.97
     Testing Error     58 133.18 2.296207 0.51 3.99
Machine Adjustment     50 125.60 2.512000 0.57 3.95
       Calibration     51 113.32 2.221961 0.50 3.84


In [240]:
# MACHINE-WISE DOWNTIME ANALYSIS

print("MACHINE-WISE DOWNTIME ANALYSIS")
print("=" * 70)

machine_downtime = (
    downtime.groupby("machine_id")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .reset_index()
)

machine_downtime.columns = [
    "machine_id",
    "downtime_events",
    "total_downtime_hours",
    "average_downtime_hours",
    "minimum_downtime_hours",
    "maximum_downtime_hours"
]

print("\nMachine-wise Downtime Summary:")
print(machine_downtime.to_string(index=False))

MACHINE-WISE DOWNTIME ANALYSIS

Machine-wise Downtime Summary:
 machine_id  downtime_events  total_downtime_hours  average_downtime_hours  minimum_downtime_hours  maximum_downtime_hours
          1               74                176.94                2.391081                    0.52                    3.95
          2               90                209.15                2.323889                    0.50                    3.99
          3              147                315.06                2.143265                    0.53                    3.98
          4               52                115.73                2.225577                    0.53                    3.95
          5               68                147.78                2.173235                    0.51                    3.97


In [241]:
# MACHINE-WISE SENSOR ANALYSIS

print("MACHINE-WISE SENSOR ANALYSIS")
print("=" * 70)

machine_sensor_summary = (
    sensors.groupby(["machine_id", "sensor_type"])["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nMachine-wise Sensor Summary:")

print(machine_sensor_summary.to_string(index=False))

MACHINE-WISE SENSOR ANALYSIS

Machine-wise Sensor Summary:
 machine_id sensor_type  count      mean   min   max      std
          1 Temperature   1095 70.153726 61.62 79.79 3.082530
          1   Vibration   1095  3.001361  1.28  4.48 0.497661
          2 Temperature   1095 67.907342 58.83 76.69 3.006898
          2   Vibration   1095  2.802393  1.05  4.26 0.476934
          3 Temperature   1095 82.032420 71.40 91.03 3.032493
          3   Vibration   1095  4.998219  3.50  6.57 0.514230
          4 Temperature   1095 64.878685 54.64 76.18 3.004166
          4   Vibration   1095  2.461041  0.50  3.79 0.498736
          5 Temperature   1095 67.029945 57.94 77.81 2.974885
          5   Vibration   1095  2.707425  1.03  4.37 0.498583


In [242]:
# DOWNTIME EDA

downtime = data["downtime"]

print("DOWNTIME DATA")
print("=" * 70)

print("Shape:", downtime.shape)

print("Columns:", downtime.columns.tolist())

print("\nFirst 5 rows:")
print(downtime.head().to_string())

print("\nStatistical Summary:")
print(downtime.describe().to_string())

DOWNTIME DATA
Shape: (431, 6)
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

First 5 rows:
   downtime_id  machine_id      downtime_start        downtime_end     downtime_reason  downtime_hours
0            1           3 2025-08-25 06:00:00 2025-08-25 09:57:36   Material Shortage            3.96
1            2           1 2025-08-26 17:00:00 2025-08-26 18:59:24  Mechanical Failure            1.99
2            3           2 2025-08-27 19:00:00 2025-08-27 20:01:48   Material Shortage            1.03
3            4           3 2025-08-27 08:00:00 2025-08-27 11:22:48  Mechanical Failure            3.38
4            5           4 2025-08-27 15:00:00 2025-08-27 16:49:48  Machine Adjustment            1.83

Statistical Summary:
       downtime_id  machine_id              downtime_start                downtime_end  downtime_hours
count   431.000000  431.000000                         431                         431      431.000000

In [243]:
# DOWNTIME REASON ANALYSIS

print("DOWNTIME REASON ANALYSIS")
print("=" * 70)

downtime_reason = (
    downtime.groupby("downtime_reason")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDowntime Reason Summary:")

print(downtime_reason.to_string(index=False))

DOWNTIME REASON ANALYSIS

Downtime Reason Summary:
   downtime_reason  count    sum     mean  min  max
Mechanical Failure    145 306.52 2.113931 0.53 3.98
 Material Shortage     66 147.43 2.233788 0.52 3.96
  Electrical Issue     61 138.61 2.272295 0.62 3.97
     Testing Error     58 133.18 2.296207 0.51 3.99
Machine Adjustment     50 125.60 2.512000 0.57 3.95
       Calibration     51 113.32 2.221961 0.50 3.84


In [244]:
#  MAINTENANCE EDA

maintenance = data["maintenance"]

print("MAINTENANCE DATA")
print("=" * 70)

print("Shape:", maintenance.shape)
print("Columns:", maintenance.columns.tolist())

print("\nFirst 5 rows:")
print(maintenance.head().to_string())

print("\nStatistical Summary:")
print(maintenance.describe().to_string())

MAINTENANCE DATA
Shape: (173, 7)
Columns: ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']

First 5 rows:
   maintenance_id  equipment_id maintenance_date    maintenance_type maintenance_status  downtime_hours  maintenance_cost
0               1             4       2025-09-01          Corrective          Completed            5.03          12134.92
1               2             4       2025-09-05          Preventive          Completed            2.43           5757.39
2               3             3       2025-09-06  Routine Inspection          Completed            1.71           3505.76
3               4             2       2025-09-07          Preventive          Completed            2.97           2633.45
4               5             1       2025-09-10          Preventive        In Progress            2.46           3943.19

Statistical Summary:
       maintenance_id  equipment_id  downtime_hours  m

In [245]:
#  MAINTENANCE TYPE ANALYSIS

print("MAINTENANCE TYPE ANALYSIS")
print("=" * 70)

maintenance_type_summary = (
    maintenance.groupby("maintenance_type")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        avg_downtime=("downtime_hours", "mean"),
        total_cost=("maintenance_cost", "sum"),
        avg_cost=("maintenance_cost", "mean")
    )
    .reset_index()
)

print(maintenance_type_summary.to_string(index=False))

MAINTENANCE TYPE ANALYSIS
  maintenance_type  maintenance_count  total_downtime  avg_downtime  total_cost     avg_cost
        Corrective                 32          139.51      4.359687   354892.94 11090.404375
        Preventive                 84          167.90      1.998810   360523.78  4291.949762
Routine Inspection                 57           68.61      1.203684   160849.23  2821.916316


In [246]:
# MACHINE-WISE MAINTENANCE ANALYSIS

print("MACHINE-WISE MAINTENANCE ANALYSIS")
print("=" * 70)

machine_maintenance = (
    maintenance.groupby("equipment_id")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        total_cost=("maintenance_cost", "sum")
    )
    .reset_index()
)

print(machine_maintenance.to_string(index=False))

MACHINE-WISE MAINTENANCE ANALYSIS
 equipment_id  maintenance_count  total_downtime  total_cost
            1                 33           67.62   178225.28
            2                 34           81.60   190862.46
            3                 55          126.92   284903.62
            4                 18           34.41    85705.11
            5                 33           65.47   136569.48


In [247]:
#  CORRELATION ANALYSIS

print("CORRELATION ANALYSIS")
print("=" * 70)

# Production + quality + machine information
analysis_df = production.merge(
    quality[["production_id", "defect_count"]],
    on="production_id",
    how="left"
)

# Correlation between important numerical variables
correlation = analysis_df[
    ["units_produced", "units_rejected", "production_time_hours", "defect_count"]
].corr()

print("\nCorrelation Matrix:")
print(correlation.round(2).to_string())

CORRELATION ANALYSIS

Correlation Matrix:
                       units_produced  units_rejected  production_time_hours  defect_count
units_produced                   1.00            0.19                    NaN          0.15
units_rejected                   0.19            1.00                    NaN          0.85
production_time_hours             NaN             NaN                    NaN           NaN
defect_count                     0.15            0.85                    NaN          1.00


In [248]:
# EDA CONCLUSIONS

print("EDA CONCLUSIONS")
print("=" * 70)

print("""
1. PRODUCTION:
   - The dataset contains 5,475 production records covering 365 days.
   - Average production is approximately 912 units per record.
   - Machine 4 has the highest average production.
   - Machine 3 has the lowest average production.
   - Production time is constant at 8 hours.

2. TARGET ACHIEVEMENT:
   - Actual production is compared with machine-wise production targets.
   - Overall target achievement is approximately 98.13%.
   - Machine 2 and Machine 3 are approximately at or slightly above their targets.
   - Machine 1 has the lowest target achievement.

3. QUALITY:
   - Quality data contains 5,475 inspection records.
   - Defect counts vary across machines and defect types.
   - Machine 4 has the highest total and average defect count.
   - Defect types show different levels of occurrence across the production process.

4. SENSORS:
   - The dataset contains temperature and vibration readings.
   - Temperature and vibration values vary across machines.
   - Machine 3 has the highest average temperature and vibration.
   - Sensor readings also vary over time.

5. DOWNTIME:
   - There are 431 downtime events.
   - Average downtime per event is approximately 2.24 hours.
   - Mechanical Failure contributes the highest total downtime.
   - Machine 3 has the highest total downtime.

6. MAINTENANCE:
   - There are 173 maintenance records.
   - Preventive maintenance is the most common maintenance type.
   - Corrective maintenance has the highest average downtime and average cost.
   - Machine 3 has the highest maintenance activity, maintenance downtime and maintenance cost.

7. CORRELATION:
   - Units rejected and defect count have a strong positive correlation of approximately 0.85.
   - Units produced and defect count have a weak positive correlation of approximately 0.15.
   - Production time is constant at 8 hours, so it has no meaningful correlation.

8. ML RELEVANCE:
   - Machine-wise production, target achievement, sensor readings,
     quality information, downtime and maintenance history provide
     useful information for ML modeling.
   - These features can be used for production prediction,
     quality/defect prediction, failure-risk prediction and anomaly detection.
""")

EDA CONCLUSIONS

1. PRODUCTION:
   - The dataset contains 5,475 production records covering 365 days.
   - Average production is approximately 912 units per record.
   - Machine 4 has the highest average production.
   - Machine 3 has the lowest average production.
   - Production time is constant at 8 hours.

2. TARGET ACHIEVEMENT:
   - Actual production is compared with machine-wise production targets.
   - Overall target achievement is approximately 98.13%.
   - Machine 2 and Machine 3 are approximately at or slightly above their targets.
   - Machine 1 has the lowest target achievement.

3. QUALITY:
   - Quality data contains 5,475 inspection records.
   - Defect counts vary across machines and defect types.
   - Machine 4 has the highest total and average defect count.
   - Defect types show different levels of occurrence across the production process.

4. SENSORS:
   - The dataset contains temperature and vibration readings.
   - Temperature and vibration values vary across machi

In [249]:
#  PRODUCTION KPIs

print("PRODUCTION KPIs")
print("=" * 70)

total_production = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

total_output = total_production + total_rejected

production_rejection_rate = (
    total_rejected / total_output
) * 100

print("\nTotal Units Produced:")
print(total_production)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nTotal Production Output:")
print(total_output)

print("\nRejection Rate (%):")
print(round(production_rejection_rate, 2))

PRODUCTION KPIs

Total Units Produced:
4993913

Total Units Rejected:
219065

Total Production Output:
5212978

Rejection Rate (%):
4.2


In [250]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = production_targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs


NameError: name 'production_targets' is not defined

In [ ]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'pd', 'mysql', 'sys', 'os', 'get_connection', 'connection', 'cursor', 'tables', 'table', 'data', 'query', 'df', 'summary', 'summary_df', 'important_tables', 'column', 'missing', 'duplicate_count', 'found_invalid', 'col', 'negative_count', 'datetime_cols', 'production', 'quality', 'downtime', 'invalid_downtime', 'targets', 'numeric_target_cols', 'numeric_cols', 'Q1', 'Q3', 'IQR', 'lower_bound', 'upper_bound', 'outliers', 'defect_analysis', 'quality_machine', 'machine_quality', 'sensors', 'sensor_type_summary', 'machine_sensor_summary', 'sensor_trends', 'downtime_reason', 'machine_downtime', 'maintenance', 'maintenance_type_summary', 'machine_maintenance', 'analysis_df', 'correlation', 'total_production', 'total_rejected', 'total_output', 'production_rejection_rate', 'total_produced']


In [ ]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs

Total Target Quantity:
5089338

Total Units Produced:
4993913

Overall Target Achievement (%):
98.13


In [ ]:
print("PRODUCTION COLUMNS:")
print(production.columns.tolist())

print("\nTARGET COLUMNS:")
print(targets.columns.tolist())

PRODUCTION COLUMNS:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

TARGET COLUMNS:
['target_id', 'machine_id', 'target_date', 'target_quantity']


In [ ]:
print("ACTUAL PRODUCTION TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production")

for row in cursor.fetchall():
    print(row)

ACTUAL PRODUCTION TABLE COLUMNS
('production_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('production_date', 'date', 'NO', '', None, '')
('shift', 'varchar(20)', 'YES', '', None, '')
('units_produced', 'int', 'YES', '', None, '')
('units_rejected', 'int', 'YES', '', None, '')
('production_time_hours', 'decimal(5,2)', 'YES', '', None, '')


In [ ]:
print("\nACTUAL PRODUCTION TARGETS TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production_targets")

for row in cursor.fetchall():
    print(row)


ACTUAL PRODUCTION TARGETS TABLE COLUMNS
('target_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('target_date', 'date', 'NO', '', None, '')
('target_quantity', 'int', 'NO', '', None, '')


In [ ]:
#  MACHINE-WISE TARGET ACHIEVEMENT
# ============================================================

print("MACHINE-WISE TARGET ACHIEVEMENT")
print("=" * 70)

machine_production = production.groupby("machine_id")["units_produced"].sum()
machine_target = targets.groupby("machine_id")["target_quantity"].sum()

machine_target_kpi = pd.DataFrame({
    "total_produced": machine_production,
    "total_target": machine_target
})

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["total_produced"] /
    machine_target_kpi["total_target"]
) * 100

machine_target_kpi["performance_status"] = machine_target_kpi[
    "target_achievement_percent"
].apply(
    lambda x: "Above Target" if x > 100
    else "On Target" if x >= 95
    else "Below Target"
)

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["target_achievement_percent"].round(2)
)

print(machine_target_kpi)

MACHINE-WISE TARGET ACHIEVEMENT
            total_produced  total_target  target_achievement_percent  \
machine_id                                                             
1                  1041452       1093419                       95.25   
2                   985611        985483                      100.01   
3                   875771        875751                      100.00   
4                  1074295       1095218                       98.09   
5                  1016784       1039467                       97.82   

           performance_status  
machine_id                     
1                   On Target  
2                Above Target  
3                Above Target  
4                   On Target  
5                   On Target  


In [ ]:
# ============================================================
#  QUALITY KPIs


print("QUALITY KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

quality_defect_rate = (
    total_rejected / total_produced
) * 100

quality_pass_rate = (
    (total_produced - total_rejected) / total_produced
) * 100

print("\nTotal Units Produced:")
print(total_produced)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nDefect/Rejection Rate (%):")
print(round(quality_defect_rate, 2))

print("\nQuality Pass Rate (%):")
print(round(quality_pass_rate, 2))

QUALITY KPIs

Total Units Produced:
4993913

Total Units Rejected:
219065

Defect/Rejection Rate (%):
4.39

Quality Pass Rate (%):
95.61


In [ ]:
#  MACHINE-WISE DEFECT RATE
# ============================================================

print("MACHINE-WISE DEFECT RATE")
print("=" * 70)

machine_quality_kpi = production.groupby("machine_id").agg(
    total_produced=("units_produced", "sum"),
    total_rejected=("units_rejected", "sum")
)

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["total_rejected"] /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["quality_pass_rate_percent"] = (
    (machine_quality_kpi["total_produced"] -
     machine_quality_kpi["total_rejected"]) /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["defect_rate_percent"].round(2)
)

machine_quality_kpi["quality_pass_rate_percent"] = (
    machine_quality_kpi["quality_pass_rate_percent"].round(2)
)

print(machine_quality_kpi)

MACHINE-WISE DEFECT RATE
            total_produced  total_rejected  defect_rate_percent  \
machine_id                                                        
1                  1041452           45804                 4.40   
2                   985611           43754                 4.44   
3                   875771           37822                 4.32   
4                  1074295           47507                 4.42   
5                  1016784           44178                 4.34   

            quality_pass_rate_percent  
machine_id                             
1                               95.60  
2                               95.56  
3                               95.68  
4                               95.58  
5                               95.66  


In [ ]:

#  CHECK DOWNTIME DATA
# ============================================================

print("DOWNTIME COLUMNS")
print("=" * 70)

print(downtime.columns.tolist())

DOWNTIME COLUMNS
['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']


In [ ]:
#  DOWNTIME KPIs
# ============================================================

print("DOWNTIME KPIs")
print("=" * 70)

total_downtime_hours = downtime["downtime_hours"].sum()

average_downtime_hours = downtime["downtime_hours"].mean()

total_downtime_events = downtime["downtime_id"].count()

print("\nTotal Downtime (Hours):")
print(round(total_downtime_hours, 2))

print("\nAverage Downtime per Event (Hours):")
print(round(average_downtime_hours, 2))

print("\nTotal Downtime Events:")
print(total_downtime_events)

DOWNTIME KPIs

Total Downtime (Hours):
964.66

Average Downtime per Event (Hours):
2.24

Total Downtime Events:
431


In [ ]:

# MACHINE-WISE DOWNTIME
# ============================================================

print("MACHINE-WISE DOWNTIME")
print("=" * 70)

machine_downtime_kpi = downtime.groupby("machine_id").agg(
    total_downtime_hours=("downtime_hours", "sum"),
    downtime_events=("downtime_id", "count")
)

machine_downtime_kpi["average_downtime_hours"] = (
    machine_downtime_kpi["total_downtime_hours"] /
    machine_downtime_kpi["downtime_events"]
)

machine_downtime_kpi = machine_downtime_kpi.round(2)

print(machine_downtime_kpi)

MACHINE-WISE DOWNTIME
            total_downtime_hours  downtime_events  average_downtime_hours
machine_id                                                               
1                         176.94               74                    2.39
2                         209.15               90                    2.32
3                         315.06              147                    2.14
4                         115.73               52                    2.23
5                         147.78               68                    2.17


In [ ]:

# CHECK PRODUCTION TIME DATA
# ============================================================

print("PRODUCTION TIME SUMMARY")
print("=" * 70)

print("\nProduction Time Statistics:")
print(production["production_time_hours"].describe())

print("\nTotal Production Time (Hours):")
print(round(production["production_time_hours"].sum(), 2))

PRODUCTION TIME SUMMARY

Production Time Statistics:
count    5475.0
mean        8.0
std         0.0
min         8.0
25%         8.0
50%         8.0
75%         8.0
max         8.0
Name: production_time_hours, dtype: float64

Total Production Time (Hours):
43800.0


In [ ]:
# ============================================================
# MACHINE UTILIZATION
# ============================================================

print("MACHINE UTILIZATION KPIs")
print("=" * 70)

# Total available production hours for each machine
machine_available_hours = production.groupby("machine_id")[
    "production_time_hours"
].sum()

# Total downtime hours for each machine
machine_downtime_hours = downtime.groupby("machine_id")[
    "downtime_hours"
].sum()

# Combine available hours and downtime
machine_utilization_kpi = pd.DataFrame({
    "available_hours": machine_available_hours,
    "downtime_hours": machine_downtime_hours
}).fillna(0)

# Calculate actual operating hours
machine_utilization_kpi["operating_hours"] = (
    machine_utilization_kpi["available_hours"] -
    machine_utilization_kpi["downtime_hours"]
)

# Calculate utilization percentage
machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["operating_hours"] /
    machine_utilization_kpi["available_hours"]
) * 100

machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["utilization_percent"].round(2)
)

print(machine_utilization_kpi)

MACHINE UTILIZATION KPIs
            available_hours  downtime_hours  operating_hours  \
machine_id                                                     
1                    8760.0          176.94          8583.06   
2                    8760.0          209.15          8550.85   
3                    8760.0          315.06          8444.94   
4                    8760.0          115.73          8644.27   
5                    8760.0          147.78          8612.22   

            utilization_percent  
machine_id                       
1                         97.98  
2                         97.61  
3                         96.40  
4                         98.68  
5                         98.31  


In [ ]:

#  CHECK AVAILABLE DATA
# ============================================================

print("AVAILABLE DATAFRAMES")
print("=" * 70)

dataframes = {
    "production": production,
    "targets": targets,
    "quality": quality,
    "downtime": downtime,
    "sensors": sensors,
    "maintenance": maintenance
}

for name, dataframe in dataframes.items():
    print(f"\n{name}:")
    print(f"Rows    : {dataframe.shape[0]}")
    print(f"Columns : {dataframe.shape[1]}")
    print(f"Columns : {dataframe.columns.tolist()}")

AVAILABLE DATAFRAMES

production:
Rows    : 5475
Columns : 7
Columns : ['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

targets:
Rows    : 5475
Columns : 4
Columns : ['target_id', 'machine_id', 'target_date', 'target_quantity']

quality:
Rows    : 5475
Columns : 6
Columns : ['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

downtime:
Rows    : 431
Columns : 6
Columns : ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

sensors:
Rows    : 10950
Columns : 6
Columns : ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

maintenance:
Rows    : 173
Columns : 7
Columns : ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']


In [ ]:
# ============================================================
# CHECK LOADED DATAFRAMES
# ============================================================

print("LOADED DATAFRAMES")
print("=" * 70)

for name in ["production", "targets", "quality", "downtime", "sensors", "maintenance"]:
    print(f"{name}: {name in globals()}")

LOADED DATAFRAMES
production: True
targets: True
quality: True
downtime: True
sensors: True
maintenance: True


In [ ]:
# ============================================================
# RELOAD MISSING DATAFRAMES
# ============================================================

print("RELOADING SENSOR AND MAINTENANCE DATA")
print("=" * 70)

# Load sensors table
query = "SELECT * FROM sensors"
sensors = pd.read_sql(query, connection)

# Load maintenance table
query = "SELECT * FROM maintenance"
maintenance = pd.read_sql(query, connection)

print("\nSensors:")
print("Rows:", sensors.shape[0])
print("Columns:", sensors.columns.tolist())

print("\nMaintenance:")
print("Rows:", maintenance.shape[0])
print("Columns:", maintenance.columns.tolist())

RELOADING SENSOR AND MAINTENANCE DATA

Sensors:
Rows: 10950
Columns: ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

Maintenance:
Rows: 173
Columns: ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']


C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_232728\4204890061.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sensors = pd.read_sql(query, connection)
C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_232728\4204890061.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  maintenance = pd.read_sql(query, connection)


In [ ]:

 #table relationships


print("TABLE RELATIONSHIPS")
print("=" * 70)

print("\nPRODUCTION:")
print(production.columns.tolist())

print("\nTARGETS:")
print(targets.columns.tolist())

print("\nQUALITY:")
print(quality.columns.tolist())

print("\nDOWNTIME:")
print(downtime.columns.tolist())

print("\nSENSORS:")
print(sensors.columns.tolist())

print("\nMAINTENANCE:")
print(maintenance.columns.tolist())

TABLE RELATIONSHIPS

PRODUCTION:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

TARGETS:
['target_id', 'machine_id', 'target_date', 'target_quantity']

QUALITY:
['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

DOWNTIME:
['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

SENSORS:
['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

MAINTENANCE:
['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']


In [ ]:
# ============================================================
#  PRODUCTION + TARGETS


print("BUILDING PRODUCTION-TARGET DATASET")
print("=" * 70)

# Merge production with production targets
master_df = production.merge(
    targets,
    left_on=["machine_id", "production_date"],
    right_on=["machine_id", "target_date"],
    how="left"
)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMaster Dataset Columns:")
print(master_df.columns.tolist())

print("\nMissing Target Values:")
print(master_df["target_quantity"].isna().sum())

BUILDING PRODUCTION-TARGET DATASET

Master Dataset Shape:
(16425, 10)

Master Dataset Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity']

Missing Target Values:
0


In [ ]:
# CHECKING TARGET DUPLICATES
# ============================================================

print("CHECKING TARGET DUPLICATES")
print("=" * 70)

target_duplicates = targets[
    targets.duplicated(
        subset=["machine_id", "target_date"],
        keep=False
    )
].sort_values(["machine_id", "target_date"])

print("\nDuplicate machine-date target records:")
print(target_duplicates)

print("\nNumber of duplicate rows:")
print(len(target_duplicates))

CHECKING TARGET DUPLICATES

Duplicate machine-date target records:
      target_id  machine_id target_date  target_quantity
0             1           1  2025-08-24             1002
1             2           1  2025-08-24             1005
2             3           1  2025-08-24              994
15           16           1  2025-08-25              939
16           17           1  2025-08-25              985
...         ...         ...         ...              ...
5458       5459           5  2026-08-22              973
5459       5460           5  2026-08-22              969
5472       5473           5  2026-08-23              954
5473       5474           5  2026-08-23              968
5474       5475           5  2026-08-23              947

[5475 rows x 4 columns]

Number of duplicate rows:
5475


In [ ]:
# ============================================================
# INSPECTING TARGET-SHIFT STRUCTURE
# ============================================================

print("TARGET AND PRODUCTION STRUCTURE")
print("=" * 70)

print("\nProduction records per machine/date:")
print(
    production.groupby(
        ["machine_id", "production_date"]
    ).size().value_counts()
)

print("\nTarget records per machine/date:")
print(
    targets.groupby(
        ["machine_id", "target_date"]
    ).size().value_counts()
)

print("\nSample production records:")
print(
    production[
        ["machine_id", "production_date", "shift", "units_produced"]
    ].head(15)
)

print("\nSample target records:")
print(
    targets[
        ["machine_id", "target_date", "target_quantity"]
    ].head(15)
)

TARGET AND PRODUCTION STRUCTURE

Production records per machine/date:
3    1825
Name: count, dtype: int64

Target records per machine/date:
3    1825
Name: count, dtype: int64

Sample production records:
    machine_id production_date      shift  units_produced
0            1      2025-08-24    Morning             969
1            1      2025-08-24  Afternoon             944
2            1      2025-08-24      Night             940
3            2      2025-08-24    Morning             890
4            2      2025-08-24  Afternoon             963
5            2      2025-08-24      Night             930
6            3      2025-08-24    Morning             781
7            3      2025-08-24  Afternoon             821
8            3      2025-08-24      Night             809
9            4      2025-08-24    Morning             903
10           4      2025-08-24  Afternoon             939
11           4      2025-08-24      Night             992
12           5      2025-08-24    Morning 

In [ ]:
#CORRECT TARGET MAPPING
# ============================================================

print("CREATING CORRECT PRODUCTION-TARGET DATASET")
print("=" * 70)

# Create an order number within each machine/date group
production_temp = production.copy()
targets_temp = targets.copy()

production_temp["shift_order"] = (
    production_temp
    .groupby(["machine_id", "production_date"])
    .cumcount()
)

targets_temp["shift_order"] = (
    targets_temp
    .groupby(["machine_id", "target_date"])
    .cumcount()
)

# Merge using machine + date + order
master_df = production_temp.merge(
    targets_temp,
    left_on=["machine_id", "production_date", "shift_order"],
    right_on=["machine_id", "target_date", "shift_order"],
    how="left"
)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMissing Target Values:")
print(master_df["target_quantity"].isna().sum())

print("\nShift Distribution:")
print(master_df["shift"].value_counts())

print("\nMaster Dataset Preview:")
print(
    master_df[
        ["production_id", "machine_id", "production_date",
         "shift", "units_produced", "target_quantity"]
    ].head(15).to_string(index=False)
)

CREATING CORRECT PRODUCTION-TARGET DATASET

Master Dataset Shape:
(5475, 11)

Missing Target Values:
0

Shift Distribution:
shift
Morning      1825
Afternoon    1825
Night        1825
Name: count, dtype: int64

Master Dataset Preview:
 production_id  machine_id production_date     shift  units_produced  target_quantity
             1           1      2025-08-24   Morning             969             1002
             2           1      2025-08-24 Afternoon             944             1005
             3           1      2025-08-24     Night             940              994
             4           2      2025-08-24   Morning             890              873
             5           2      2025-08-24 Afternoon             963              882
             6           2      2025-08-24     Night             930              867
             7           3      2025-08-24   Morning             781              773
             8           3      2025-08-24 Afternoon             821         

In [ ]:
# ============================================================
#  PRODUCTION FEATURES

print("CREATING PRODUCTION FEATURES")
print("=" * 70)

# Target achievement
master_df["target_achievement_percent"] = (
    master_df["units_produced"] /
    master_df["target_quantity"]
) * 100

# Total production output
master_df["total_output"] = (
    master_df["units_produced"] +
    master_df["units_rejected"]
)

# Rejection rate
master_df["rejection_rate_percent"] = (
    master_df["units_rejected"] /
    master_df["total_output"]
) * 100

# Production efficiency
master_df["production_rate_per_hour"] = (
    master_df["units_produced"] /
    master_df["production_time_hours"]
)

# Round calculated features
master_df["target_achievement_percent"] = (
    master_df["target_achievement_percent"].round(2)
)

master_df["rejection_rate_percent"] = (
    master_df["rejection_rate_percent"].round(2)
)

master_df["production_rate_per_hour"] = (
    master_df["production_rate_per_hour"].round(2)
)

print("\nProduction Features Created:")
print([
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
])

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "shift",
            "units_produced",
            "target_quantity",
            "target_achievement_percent",
            "units_rejected",
            "rejection_rate_percent",
            "production_rate_per_hour"
        ]
    ].head(10)
)

CREATING PRODUCTION FEATURES

Production Features Created:
['target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour']

Feature Preview:
   machine_id      shift  units_produced  target_quantity  \
0           1    Morning             969             1002   
1           1  Afternoon             944             1005   
2           1      Night             940              994   
3           2    Morning             890              873   
4           2  Afternoon             963              882   
5           2      Night             930              867   
6           3    Morning             781              773   
7           3  Afternoon             821              809   
8           3      Night             809              794   
9           4    Morning             903             1004   

   target_achievement_percent  units_rejected  rejection_rate_percent  \
0                       96.71              59                    5.74   
1   

In [ ]:
# QUALITY FEATURES
# ============================================================

print("CREATING QUALITY FEATURES")
print("=" * 70)

# Aggregate quality data by production record
quality_features = quality.groupby("production_id").agg(
    total_defects=("defect_count", "sum"),
    defect_type_count=("defect_type", "nunique")
).reset_index()

# Merge quality features into master dataset
master_df = master_df.merge(
    quality_features,
    on="production_id",
    how="left"
)

# Production records without quality defects
master_df["total_defects"] = (
    master_df["total_defects"].fillna(0)
)

master_df["defect_type_count"] = (
    master_df["defect_type_count"].fillna(0)
)

# Quality defect rate
master_df["quality_defect_rate_percent"] = (
    master_df["total_defects"] /
    master_df["total_output"]
) * 100

master_df["quality_defect_rate_percent"] = (
    master_df["quality_defect_rate_percent"].round(2)
)

print("\nQuality Features Created:")
print([
    "total_defects",
    "defect_type_count",
    "quality_defect_rate_percent"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMissing Quality Values:")
print(
    master_df[
        ["total_defects", "defect_type_count"]
    ].isna().sum()
)

print("\nFeature Preview:")
print(
    master_df[
        [
            "production_id",
            "total_output",
            "total_defects",
            "defect_type_count",
            "quality_defect_rate_percent"
        ]
    ].head(10)
)

CREATING QUALITY FEATURES

Quality Features Created:
['total_defects', 'defect_type_count', 'quality_defect_rate_percent']

Master Dataset Shape:
(5475, 18)

Missing Quality Values:
total_defects        0
defect_type_count    0
dtype: int64

Feature Preview:
   production_id  total_output  total_defects  defect_type_count  \
0              1          1028             20                  1   
1              2           992             16                  1   
2              3           953              3                  1   
3              4           952             21                  1   
4              5           974              2                  1   
5              6          1002             19                  1   
6              7           798              3                  1   
7              8           839              6                  1   
8              9           841             11                  1   
9             10           930              7                

In [ ]:

#  DOWNTIME FEATURES
# ============================================================

print("CREATING DOWNTIME FEATURES")
print("=" * 70)

# Aggregate downtime by machine
downtime_features = downtime.groupby("machine_id").agg(
    total_downtime_hours=("downtime_hours", "sum"),
    downtime_event_count=("downtime_id", "count"),
    average_downtime_hours=("downtime_hours", "mean")
).reset_index()

# Merge downtime features into master dataset
master_df = master_df.merge(
    downtime_features,
    on="machine_id",
    how="left"
)

# Fill missing downtime values
master_df["total_downtime_hours"] = (
    master_df["total_downtime_hours"].fillna(0)
)

master_df["downtime_event_count"] = (
    master_df["downtime_event_count"].fillna(0)
)

master_df["average_downtime_hours"] = (
    master_df["average_downtime_hours"].fillna(0)
)

# Total available production hours per machine
available_hours = (
    production.groupby("machine_id")["production_time_hours"]
    .sum()
    .reset_index()
    .rename(columns={
        "production_time_hours": "available_hours"
    })
)

# Merge available hours
master_df = master_df.merge(
    available_hours,
    on="machine_id",
    how="left"
)

# Operating hours
master_df["operating_hours"] = (
    master_df["available_hours"] -
    master_df["total_downtime_hours"]
)

# Utilization
master_df["utilization_percent"] = (
    master_df["operating_hours"] /
    master_df["available_hours"]
) * 100

# Round values
master_df["average_downtime_hours"] = (
    master_df["average_downtime_hours"].round(2)
)

master_df["operating_hours"] = (
    master_df["operating_hours"].round(2)
)

master_df["utilization_percent"] = (
    master_df["utilization_percent"].round(2)
)

print("\nDowntime Features Created:")
print([
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "total_downtime_hours",
            "downtime_event_count",
            "average_downtime_hours",
            "available_hours",
            "operating_hours",
            "utilization_percent"
        ]
    ].drop_duplicates("machine_id")
)

CREATING DOWNTIME FEATURES

Downtime Features Created:
['total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent']

Master Dataset Shape:
(5475, 24)

Feature Preview:
    machine_id  total_downtime_hours  downtime_event_count  \
0            1                176.94                    74   
3            2                209.15                    90   
6            3                315.06                   147   
9            4                115.73                    52   
12           5                147.78                    68   

    average_downtime_hours  available_hours  operating_hours  \
0                     2.39           8760.0          8583.06   
3                     2.32           8760.0          8550.85   
6                     2.14           8760.0          8444.94   
9                     2.23           8760.0          8644.27   
12                    2.17           8760.0          8612.22   


In [ ]:
# SENSOR FEATURES
# ============================================================

print("CREATING SENSOR FEATURES")
print("=" * 70)

# Aggregate sensor readings by machine
sensor_features = sensors.groupby("machine_id").agg(
    sensor_value_mean=("sensor_value", "mean"),
    sensor_value_min=("sensor_value", "min"),
    sensor_value_max=("sensor_value", "max"),
    sensor_value_std=("sensor_value", "std"),
    sensor_reading_count=("sensor_id", "count")
).reset_index()

# Merge sensor features into master dataset
master_df = master_df.merge(
    sensor_features,
    on="machine_id",
    how="left"
)

# Fill missing values
sensor_columns = [
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count"
]

master_df[sensor_columns] = (
    master_df[sensor_columns].fillna(0)
)

# Round sensor values
master_df["sensor_value_mean"] = (
    master_df["sensor_value_mean"].round(2)
)

master_df["sensor_value_min"] = (
    master_df["sensor_value_min"].round(2)
)

master_df["sensor_value_max"] = (
    master_df["sensor_value_max"].round(2)
)

master_df["sensor_value_std"] = (
    master_df["sensor_value_std"].round(2)
)

print("\nSensor Features Created:")
print(sensor_columns)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "sensor_value_mean",
            "sensor_value_min",
            "sensor_value_max",
            "sensor_value_std",
            "sensor_reading_count"
        ]
    ].drop_duplicates("machine_id")
)

CREATING SENSOR FEATURES

Sensor Features Created:
['sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count']

Master Dataset Shape:
(5475, 29)

Feature Preview:
    machine_id  sensor_value_mean  sensor_value_min  sensor_value_max  \
0            1              36.58              1.28             79.79   
3            2              35.35              1.05             76.69   
6            3              43.52              3.50             91.03   
9            4              33.67              0.50             76.18   
12           5              34.87              1.03             77.81   

    sensor_value_std  sensor_reading_count  
0              33.66                  2190  
3              32.63                  2190  
6              38.59                  2190  
9              31.29                  2190  
12             32.24                  2190  


In [ ]:
#MAINTENANCE FEATURES
# ============================================================

print("CREATING MAINTENANCE FEATURES")
print("=" * 70)

# Aggregate maintenance data by equipment
maintenance_features = maintenance.groupby("equipment_id").agg(
    maintenance_event_count=("maintenance_id", "count"),
    total_maintenance_downtime=("downtime_hours", "sum"),
    average_maintenance_downtime=("downtime_hours", "mean"),
    total_maintenance_cost=("maintenance_cost", "sum")
).reset_index()

# Rename equipment_id to machine_id for merging
maintenance_features = maintenance_features.rename(
    columns={"equipment_id": "machine_id"}
)

# Count completed maintenance events
completed_maintenance = (
    maintenance[maintenance["maintenance_status"].str.lower() == "completed"]
    .groupby("equipment_id")
    .size()
    .reset_index(name="completed_maintenance_count")
)

completed_maintenance = completed_maintenance.rename(
    columns={"equipment_id": "machine_id"}
)

# Merge completed maintenance count
maintenance_features = maintenance_features.merge(
    completed_maintenance,
    on="machine_id",
    how="left"
)

# Merge maintenance features into master dataset
master_df = master_df.merge(
    maintenance_features,
    on="machine_id",
    how="left"
)

# Fill missing maintenance values
maintenance_columns = [
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
]

master_df[maintenance_columns] = (
    master_df[maintenance_columns].fillna(0)
)

# Round values
master_df["total_maintenance_downtime"] = (
    master_df["total_maintenance_downtime"].round(2)
)

master_df["average_maintenance_downtime"] = (
    master_df["average_maintenance_downtime"].round(2)
)

master_df["total_maintenance_cost"] = (
    master_df["total_maintenance_cost"].round(2)
)

print("\nMaintenance Features Created:")
print(maintenance_columns)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "maintenance_event_count",
            "total_maintenance_downtime",
            "average_maintenance_downtime",
            "total_maintenance_cost",
            "completed_maintenance_count"
        ]
    ].drop_duplicates("machine_id")
)

CREATING MAINTENANCE FEATURES

Maintenance Features Created:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

Master Dataset Shape:
(5475, 34)

Feature Preview:
    machine_id  maintenance_event_count  total_maintenance_downtime  \
0            1                       33                       67.62   
3            2                       34                       81.60   
6            3                       55                      126.92   
9            4                       18                       34.41   
12           5                       33                       65.47   

    average_maintenance_downtime  total_maintenance_cost  \
0                           2.05               178225.28   
3                           2.40               190862.46   
6                           2.31               284903.62   
9                           1.91                85705.11   
12           

In [ ]:
# ============================================================
#  TIME AND SHIFT FEATURES
# ============================================================

print("CREATING TIME AND SHIFT FEATURES")
print("=" * 70)

# Make sure production date is datetime
master_df["production_date"] = pd.to_datetime(
    master_df["production_date"]
)

# Date-based features
master_df["production_day"] = (
    master_df["production_date"].dt.day
)

master_df["production_month"] = (
    master_df["production_date"].dt.month
)

master_df["production_day_of_week"] = (
    master_df["production_date"].dt.dayofweek
)

# Weekend indicator
master_df["is_weekend"] = (
    master_df["production_day_of_week"] >= 5
).astype(int)

# Shift encoding
shift_mapping = {
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
}

master_df["shift_encoded"] = (
    master_df["shift"].map(shift_mapping)
)

print("\nTime and Shift Features Created:")
print([
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "production_date",
            "shift",
            "production_day",
            "production_month",
            "production_day_of_week",
            "is_weekend",
            "shift_encoded"
        ]
    ].head(10)
)

CREATING TIME AND SHIFT FEATURES

Time and Shift Features Created:
['production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded']

Master Dataset Shape:
(5475, 39)

Feature Preview:
  production_date      shift  production_day  production_month  \
0      2025-08-24    Morning              24                 8   
1      2025-08-24  Afternoon              24                 8   
2      2025-08-24      Night              24                 8   
3      2025-08-24    Morning              24                 8   
4      2025-08-24  Afternoon              24                 8   
5      2025-08-24      Night              24                 8   
6      2025-08-24    Morning              24                 8   
7      2025-08-24  Afternoon              24                 8   
8      2025-08-24      Night              24                 8   
9      2025-08-24    Morning              24                 8   

   production_day_of_week  is_weekend  shift_encoded  
0  

In [ ]:

#  HISTORICAL FEATURES


print("CREATING HISTORICAL PRODUCTION FEATURES")
print("=" * 70)

# Define shift order
shift_order = {
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
}

# Create temporary shift order for sorting
master_df["shift_order"] = (
    master_df["shift"].map(shift_order)
)

# Sort chronologically within each machine
master_df = master_df.sort_values(
    ["machine_id", "production_date", "shift_order"]
).reset_index(drop=True)

# Previous production
master_df["previous_units_produced"] = (
    master_df
    .groupby("machine_id")["units_produced"]
    .shift(1)
)

# Previous rejection rate
master_df["previous_rejection_rate"] = (
    master_df
    .groupby("machine_id")["rejection_rate_percent"]
    .shift(1)
)

# Rolling average production over previous 3 records
master_df["rolling_3_production_avg"] = (
    master_df
    .groupby("machine_id")["units_produced"]
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

# Rolling average rejection rate over previous 3 records
master_df["rolling_3_rejection_avg"] = (
    master_df
    .groupby("machine_id")["rejection_rate_percent"]
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

# Round values
master_df["rolling_3_production_avg"] = (
    master_df["rolling_3_production_avg"].round(2)
)

master_df["previous_rejection_rate"] = (
    master_df["previous_rejection_rate"].round(2)
)

master_df["rolling_3_rejection_avg"] = (
    master_df["rolling_3_rejection_avg"].round(2)
)

print("\nHistorical Features Created:")
print([
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nHistorical Feature Preview:")
print(
    master_df[
        [
            "machine_id",
            "production_date",
            "shift",
            "units_produced",
            "previous_units_produced",
            "previous_rejection_rate",
            "rolling_3_production_avg",
            "rolling_3_rejection_avg"
        ]
    ].head(15).to_string(index=False)
)

CREATING HISTORICAL PRODUCTION FEATURES

Historical Features Created:
['previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Master Dataset Shape:
(5475, 43)

Historical Feature Preview:
 machine_id production_date     shift  units_produced  previous_units_produced  previous_rejection_rate  rolling_3_production_avg  rolling_3_rejection_avg
          1      2025-08-24   Morning             969                      NaN                      NaN                       NaN                      NaN
          1      2025-08-24 Afternoon             944                    969.0                     5.74                    969.00                     5.74
          1      2025-08-24     Night             940                    944.0                     4.84                    956.50                     5.29
          1      2025-08-25   Morning             954                    940.0                     1.36                    951.00        

In [ ]:
#FINAL VALIDATION
# ============================================================

print("FINAL FEATURE DATASET VALIDATION")
print("=" * 70)

# Remove temporary sorting column
master_df = master_df.drop(columns=["shift_order"])

# Check shape
print("\nDataset Shape:")
print(master_df.shape)

# Check duplicate production records
duplicate_production_ids = (
    master_df["production_id"].duplicated().sum()
)

print("\nDuplicate Production IDs:")
print(duplicate_production_ids)

# Check missing values
missing_values = master_df.isnull().sum()

print("\nColumns With Missing Values:")
print(missing_values[missing_values > 0])

# Check infinite values
numeric_data = master_df.select_dtypes(include="number")

infinite_values = (
    numeric_data.isin([float("inf"), float("-inf")])
    .sum()
    .sum()
)

print("\nInfinite Values:")
print(infinite_values)

# Number of numeric features
print("\nNumeric Columns:")
print(len(numeric_data.columns))

# Final columns
print("\nFinal Feature Columns:")
print(master_df.columns.tolist())

FINAL FEATURE DATASET VALIDATION

Dataset Shape:
(5475, 42)

Duplicate Production IDs:
0

Columns With Missing Values:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Infinite Values:
0

Numeric Columns:
39

Final Feature Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenanc

In [ ]:

#  SAVE FEATURE DATASET

print("SAVING FEATURE-ENGINEERED DATASET")
print("=" * 70)

# Create output directory
import os

os.makedirs("../data/processed", exist_ok=True)

# Save feature-engineered dataset
feature_file = "../data/processed/feature_engineered_data.csv"

master_df.to_csv(
    feature_file,
    index=False
)

print("\nFeature dataset saved successfully!")
print("File:", feature_file)
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

SAVING FEATURE-ENGINEERED DATASET

Feature dataset saved successfully!
File: ../data/processed/feature_engineered_data.csv
Rows: 5475
Columns: 42


In [ ]:
# ============================================================
# CHECK FEATURE-ENGINEERED DATASET
# ============================================================

print("FEATURE-ENGINEERED DATASET CHECK")
print("=" * 70)

print("\nShape:")
print(master_df.shape)

print("\nColumns:")
for i, column in enumerate(master_df.columns, start=1):
    print(f"{i}. {column}")

print("\nMissing Values:")
print(master_df.isnull().sum().sum())

print("\nFirst 5 Rows:")
display(master_df.head())

FEATURE-ENGINEERED DATASET CHECK

Shape:
(5475, 42)

Columns:
1. production_id
2. machine_id
3. production_date
4. shift
5. units_produced
6. units_rejected
7. production_time_hours
8. target_id
9. target_date
10. target_quantity
11. target_achievement_percent
12. total_output
13. rejection_rate_percent
14. production_rate_per_hour
15. total_defects
16. defect_type_count
17. quality_defect_rate_percent
18. total_downtime_hours
19. downtime_event_count
20. average_downtime_hours
21. available_hours
22. operating_hours
23. utilization_percent
24. sensor_value_mean
25. sensor_value_min
26. sensor_value_max
27. sensor_value_std
28. sensor_reading_count
29. maintenance_event_count
30. total_maintenance_downtime
31. average_maintenance_downtime
32. total_maintenance_cost
33. completed_maintenance_count
34. production_day
35. production_month
36. production_day_of_week
37. is_weekend
38. shift_encoded
39. previous_units_produced
40. previous_rejection_rate
41. rolling_3_production_avg
42. rol

,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours,target_id,target_date,target_quantity,...,completed_maintenance_count,production_day,production_month,production_day_of_week,is_weekend,shift_encoded,previous_units_produced,previous_rejection_rate,rolling_3_production_avg,rolling_3_rejection_avg
0,1,1,2025-08-24,Morning,969,59,8.0,1,2025-08-24,1002,...,26,24,8,6,1,0,NaN,NaN,NaN,NaN
1,2,1,2025-08-24,Afternoon,944,48,8.0,2,2025-08-24,1005,...,26,24,8,6,1,1,969.0,5.74,969.0,5.74
2,3,1,2025-08-24,Night,940,13,8.0,3,2025-08-24,994,...,26,24,8,6,1,2,944.0,4.84,956.5,5.29
3,16,1,2025-08-25,Morning,954,20,8.0,16,2025-08-25,939,...,26,25,8,0,0,0,940.0,1.36,951.0,3.98
4,17,1,2025-08-25,Afternoon,925,53,8.0,17,2025-08-25,985,...,26,25,8,0,0,1,954.0,2.05,946.0,2.75


In [ ]:
# ============================================================
# CHECK MISSING VALUES IN FEATURE DATASET
# ============================================================

print("MISSING VALUES IN FEATURE DATASET")
print("=" * 70)

missing_features = master_df.isnull().sum()

missing_features = missing_features[missing_features > 0]

if len(missing_features) == 0:
    print("No missing values found.")
else:
    print(missing_features)

MISSING VALUES IN FEATURE DATASET
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64


In [ ]:
# ============================================================
# HANDLE MISSING TIME-BASED FEATURES
# ============================================================

print("HANDLING MISSING FEATURE VALUES")
print("=" * 70)

master_df["previous_units_produced"] = (
    master_df["previous_units_produced"]
    .fillna(master_df["units_produced"])
)

master_df["previous_rejection_rate"] = (
    master_df["previous_rejection_rate"]
    .fillna(master_df["rejection_rate_percent"])
)

master_df["rolling_3_production_avg"] = (
    master_df["rolling_3_production_avg"]
    .fillna(master_df["units_produced"])
)

master_df["rolling_3_rejection_avg"] = (
    master_df["rolling_3_rejection_avg"]
    .fillna(master_df["rejection_rate_percent"])
)

print("\nMissing values after handling:")

remaining_missing = master_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("No missing values found.")

else:
    print(remaining_missing)

print("\nDataset Shape:")
print(master_df.shape)

HANDLING MISSING FEATURE VALUES

Missing values after handling:
No missing values found.

Dataset Shape:
(5475, 42)


In [ ]:
# ============================================================
# SAVE UPDATED FEATURE DATASET
# ============================================================

feature_file = "../data/processed/feature_engineered_data.csv"

master_df.to_csv(
    feature_file,
    index=False
)

print("UPDATED FEATURE DATASET SAVED")
print("=" * 70)

print("File:", feature_file)
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])
print("Missing Values:", master_df.isnull().sum().sum())

UPDATED FEATURE DATASET SAVED
File: ../data/processed/feature_engineered_data.csv
Rows: 5475
Columns: 42
Missing Values: 0


In [ ]:
# LOAD DATA
# ============================================================

import pandas as pd

feature_data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("ML DATASET LOADED")
print("=" * 70)

print("Shape:", feature_data.shape)

print("\nColumns:")
print(feature_data.columns.tolist())

print("\nMissing Values:")
print(feature_data.isnull().sum()[feature_data.isnull().sum() > 0])

ML DATASET LOADED
Shape: (5475, 42)

Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Missing Values:
Series([],

In [ ]:
# ============================================================
# RELOAD AND VERIFY FEATURE DATASET
# ============================================================

feature_data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("FEATURE DATASET RELOADED")
print("=" * 70)

print("Shape:", feature_data.shape)

print("\nMissing Values:")
print(feature_data.isnull().sum().sum())

print("\nDuplicate Rows:")
print(feature_data.duplicated().sum())

FEATURE DATASET RELOADED
Shape: (5475, 42)

Missing Values:
0

Duplicate Rows:
0


In [ ]:
# COLUMN CATEGORIZATION
# ============================================================

print("COLUMN CATEGORIZATION")
print("=" * 70)

print("\nIDENTIFIER COLUMNS:")
print([
    "production_id",
    "target_id"
])

print("\nDATE / TIME COLUMNS:")
print([
    "production_date",
    "target_date"
])

print("\nPRODUCTION COLUMNS:")
print([
    "units_produced",
    "units_rejected",
    "production_time_hours"
])

print("\nTARGET / PERFORMANCE COLUMNS:")
print([
    "target_quantity",
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
])

print("\nQUALITY COLUMNS:")
print([
    "total_defects",
    "defect_type_count",
    "quality_defect_rate_percent"
])

print("\nDOWNTIME / UTILIZATION COLUMNS:")
print([
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent"
])

print("\nSENSOR COLUMNS:")
print([
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count"
])

print("\nMAINTENANCE COLUMNS:")
print([
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
])

print("\nTIME / SHIFT FEATURES:")
print([
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded"
])

print("\nHISTORICAL FEATURES:")
print([
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
])

COLUMN CATEGORIZATION

IDENTIFIER COLUMNS:
['production_id', 'target_id']

DATE / TIME COLUMNS:
['production_date', 'target_date']

PRODUCTION COLUMNS:
['units_produced', 'units_rejected', 'production_time_hours']

TARGET / PERFORMANCE COLUMNS:
['target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour']

QUALITY COLUMNS:
['total_defects', 'defect_type_count', 'quality_defect_rate_percent']

DOWNTIME / UTILIZATION COLUMNS:
['total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent']

SENSOR COLUMNS:
['sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count']

MAINTENANCE COLUMNS:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

TIME / SHIFT FEATURES:
['production_day', 'production_month', 'production_day_of_week

In [ ]:
# ============================================================
# PRODUCTION PREDICTION DATASET
# ============================================================

print("PRODUCTION PREDICTION DATASET")
print("=" * 70)

# Target variable
production_target = "units_produced"

# Features available before/current production outcome
production_features = [
    "machine_id",
    "target_quantity",
    "production_time_hours",
    "total_defects",
    "defect_type_count",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

# Check that all required columns exist
missing_columns = [
    column for column in production_features + [production_target]
    if column not in feature_data.columns
]

print("\nMissing required columns:")
print(missing_columns)

print("\nNumber of features:", len(production_features))
print("Target:", production_target)

PRODUCTION PREDICTION DATASET

Missing required columns:
[]

Number of features: 30
Target: units_produced


In [ ]:
# ============================================================
# CREATE PRODUCTION X AND y
# ============================================================

print("CREATING PRODUCTION X AND y")
print("=" * 70)

X_production = feature_data[production_features].copy()
y_production = feature_data[production_target].copy()

print("\nX shape:")
print(X_production.shape)

print("\ny shape:")
print(y_production.shape)

print("\nX missing values:")
print(X_production.isnull().sum().sum())

print("\ny missing values:")
print(y_production.isnull().sum())

print("\nFirst 5 X rows:")
display(X_production.head())

print("\nFirst 5 y values:")
display(y_production.head())

CREATING PRODUCTION X AND y

X shape:
(5475, 30)

y shape:
(5475,)

X missing values:
0

y missing values:
0

First 5 X rows:


,machine_id,target_quantity,production_time_hours,total_defects,defect_type_count,total_downtime_hours,downtime_event_count,average_downtime_hours,available_hours,operating_hours,...,completed_maintenance_count,production_day,production_month,production_day_of_week,is_weekend,shift_encoded,previous_units_produced,previous_rejection_rate,rolling_3_production_avg,rolling_3_rejection_avg
0,1,1002,8.0,20,1,176.94,74,2.39,8760.0,8583.06,...,26,24,8,6,1,0,969.0,5.74,969.0,5.74
1,1,1005,8.0,16,1,176.94,74,2.39,8760.0,8583.06,...,26,24,8,6,1,1,969.0,5.74,969.0,5.74
2,1,994,8.0,3,1,176.94,74,2.39,8760.0,8583.06,...,26,24,8,6,1,2,944.0,4.84,956.5,5.29
3,1,939,8.0,3,1,176.94,74,2.39,8760.0,8583.06,...,26,25,8,0,0,0,940.0,1.36,951.0,3.98
4,1,985,8.0,20,1,176.94,74,2.39,8760.0,8583.06,...,26,25,8,0,0,1,954.0,2.05,946.0,2.75



First 5 y values:


0    969
1    944
2    940
3    954
4    925
Name: units_produced, dtype: int64

In [ ]:
# DATA TYPE & MISSING CHECK
# ============================================================

print("ML DATASET PRE-CHECK")
print("=" * 70)

print("\nDATA TYPES:")
print(feature_data.dtypes)

print("\nMISSING VALUES:")
missing = feature_data.isnull().sum()
print(missing[missing > 0])

print("\nDUPLICATE PRODUCTION IDs:")
print(feature_data["production_id"].duplicated().sum())

print("\nTARGET STATISTICS:")
print(feature_data["units_produced"].describe())

ML DATASET PRE-CHECK

DATA TYPES:
production_id                     int64
machine_id                        int64
production_date                     str
shift                               str
units_produced                    int64
units_rejected                    int64
production_time_hours           float64
target_id                         int64
target_date                         str
target_quantity                   int64
target_achievement_percent      float64
total_output                      int64
rejection_rate_percent          float64
production_rate_per_hour        float64
total_defects                     int64
defect_type_count                 int64
quality_defect_rate_percent     float64
total_downtime_hours            float64
downtime_event_count              int64
average_downtime_hours          float64
available_hours                 float64
operating_hours                 float64
utilization_percent             float64
sensor_value_mean               float64
sensor

In [ ]:
# ============================================================

# CREATING PRODUCTION PREDICTION DATASET
# ============================================================

# Target variable
target_column = "units_produced"

# Features that should NOT be used for production prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    
    # Target itself
    "units_produced",
    
    # Features calculated using the target
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
]

# Create X
X_production = feature_data.drop(
    columns=columns_to_remove
)

# Create y
y_production = feature_data[target_column]

print("PRODUCTION ML DATASET")
print("=" * 70)

print("\nX Shape:")
print(X_production.shape)

print("\ny Shape:")
print(y_production.shape)

print("\nX Columns:")
print(X_production.columns.tolist())

print("\nTarget:")
print(target_column)

print("\nTarget Statistics:")
print(y_production.describe())

PRODUCTION ML DATASET

X Shape:
(450, 33)

y Shape:
(450,)

X Columns:
['machine_id', 'shift', 'units_rejected', 'production_time_hours', 'target_quantity', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Target:
units_produced

Target Statistics:
count     450.000000
mean      911.237778
std        73.708261
min       670.000000
25%       868.000000
50%       921.000000
75%       961.

In [ ]:
# ============================================================
# TIME-BASED TRAIN TEST SPLIT
# ============================================================

print("TIME-BASED TRAIN TEST SPLIT")
print("=" * 70)

# Sort data by date, machine and shift
feature_data = feature_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

# Use 80% for training
split_index = int(len(feature_data) * 0.80)

train_data = feature_data.iloc[:split_index].copy()
test_data = feature_data.iloc[split_index:].copy()

# Create training and testing data
X_train_production = train_data[production_features]
X_test_production = test_data[production_features]

y_train_production = train_data[production_target]
y_test_production = test_data[production_target]

print("\nTraining data shape:")
print(X_train_production.shape)

print("\nTesting data shape:")
print(X_test_production.shape)

print("\nTraining date range:")
print(
    train_data["production_date"].min(),
    "to",
    train_data["production_date"].max()
)

print("\nTesting date range:")
print(
    test_data["production_date"].min(),
    "to",
    test_data["production_date"].max()
)

print("\nTraining target shape:")
print(y_train_production.shape)

print("\nTesting target shape:")
print(y_test_production.shape)

TIME-BASED TRAIN TEST SPLIT

Training data shape:
(4380, 30)

Testing data shape:
(1095, 30)

Training date range:
2025-08-24 to 2026-06-11

Testing date range:
2026-06-12 to 2026-08-23

Training target shape:
(4380,)

Testing target shape:
(1095,)


In [ ]:
# 
# CREATING PRODUCTION PREDICTION DATASET
# ============================================================

# Target variable
target_column = "units_produced"

# Features that should NOT be used for production prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    
    # Target itself
    "units_produced",
    
    # Features calculated using the target
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
]

# Create X
X_production = feature_data.drop(
    columns=columns_to_remove
)

# Create y
y_production = feature_data[target_column]

print("PRODUCTION ML DATASET")
print("=" * 70)

print("\nX Shape:")
print(X_production.shape)

print("\ny Shape:")
print(y_production.shape)

print("\nX Columns:")
print(X_production.columns.tolist())

print("\nTarget:")
print(target_column)

print("\nTarget Statistics:")
print(y_production.describe())

PRODUCTION ML DATASET

X Shape:
(450, 33)

y Shape:
(450,)

X Columns:
['machine_id', 'shift', 'units_rejected', 'production_time_hours', 'target_quantity', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Target:
units_produced

Target Statistics:
count     450.000000
mean      911.237778
std        73.708261
min       670.000000
25%       868.000000
50%       921.000000
75%       961.

In [ ]:

# CLEAN X FOR MACHINE LEARNING
# ============================================================

# Make a copy so the original X remains unchanged
X_production_clean = X_production.copy()

# Remove string-based shift column
# shift_encoded is already available
X_production_clean = X_production_clean.drop(columns=["shift"])

# Check missing values before handling
print("MISSING VALUES BEFORE HANDLING")
print("=" * 70)

missing_before = X_production_clean.isnull().sum()
print(missing_before[missing_before > 0])

# Fill historical missing values using 0
# These occur because the first record of each machine
# has no previous production history.
historical_columns = [
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

X_production_clean[historical_columns] = (
    X_production_clean[historical_columns].fillna(0)
)

print("\nMISSING VALUES AFTER HANDLING")
print("=" * 70)

missing_after = X_production_clean.isnull().sum()
print(missing_after[missing_after > 0])

print("\nFINAL X SHAPE:")
print(X_production_clean.shape)

print("\nDATA TYPES:")
print(X_production_clean.dtypes)

MISSING VALUES BEFORE HANDLING
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

MISSING VALUES AFTER HANDLING
Series([], dtype: int64)

FINAL X SHAPE:
(450, 32)

DATA TYPES:
machine_id                        int64
units_rejected                    int64
production_time_hours           float64
target_quantity                   int64
total_defects                     int64
defect_type_count                 int64
quality_defect_rate_percent     float64
total_downtime_hours            float64
downtime_event_count              int64
average_downtime_hours          float64
available_hours                 float64
operating_hours                 float64
utilization_percent             float64
sensor_value_mean               float64
sensor_value_min                float64
sensor_value_max                float64
sensor_value_std                float64
sensor_reading_count              int64
maintenance_event_cou

In [ ]:
# ============================================================

# TRAIN AND TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_production_clean,
    y_production,
    test_size=0.20,
    random_state=42
)

print("TRAIN / TEST SPLIT")
print("=" * 70)

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("\nTraining records:", len(X_train))
print("Testing records :", len(X_test))

ImportError: DLL load failed while importing _uarray: An Application Control policy has blocked this file.

In [ ]:

# ============================================================
# DATA LEAKAGE CHECK
# ============================================================

print("DATA LEAKAGE CHECK")
print("=" * 70)

# Check whether target exists inside features
print("\n1. Target column inside X:")
print("units_produced" in X_train_production.columns)

# Check whether train and test have overlapping indexes
print("\n2. Train/Test index overlap:")
print(len(set(X_train_production.index) & set(X_test_production.index)))

# Check missing values
print("\n3. Missing values in X_train:")
print(X_train_production.isnull().sum().sum())

print("\n4. Missing values in X_test:")
print(X_test_production.isnull().sum().sum())

# Check shapes
print("\n5. Dataset shapes:")
print("X_train:", X_train_production.shape)
print("X_test :", X_test_production.shape)
print("y_train:", y_train_production.shape)
print("y_test :", y_test_production.shape)

DATA LEAKAGE CHECK

1. Target column inside X:
False

2. Train/Test index overlap:
0

3. Missing values in X_train:
0

4. Missing values in X_test:
0

5. Dataset shapes:
X_train: (360, 32)
X_test : (90, 32)
y_train: (360,)
y_test : (90,)


In [ ]:
# ============================================================
# NEW PRODUCTION DATA LEAKAGE CHECK
# ============================================================

print("NEW PRODUCTION DATA LEAKAGE CHECK")
print("=" * 70)

# Check whether target is inside the new features
print("\n1. Target column inside X:")
print("units_produced" in X_train_production.columns)

# Check train/test overlap
print("\n2. Train/Test index overlap:")
print(
    len(
        set(X_train_production.index)
        & set(X_test_production.index)
    )
)

# Check missing values
print("\n3. Missing values in X_train:")
print(X_train_production.isnull().sum().sum())

print("\n4. Missing values in X_test:")
print(X_test_production.isnull().sum().sum())

# Check shapes
print("\n5. Dataset shapes:")
print("X_train:", X_train_production.shape)
print("X_test :", X_test_production.shape)
print("y_train:", y_train_production.shape)
print("y_test :", y_test_production.shape)

NEW PRODUCTION DATA LEAKAGE CHECK

1. Target column inside X:
False

2. Train/Test index overlap:
0

3. Missing values in X_train:
0

4. Missing values in X_test:
0

5. Dataset shapes:
X_train: (4380, 30)
X_test : (1095, 30)
y_train: (4380,)
y_test : (1095,)


In [ ]:
# ============================================================
# FEATURE LEVEL LEAKAGE REVIEW
# ============================================================

print("FEATURE LEVEL LEAKAGE REVIEW")
print("=" * 70)

# Features that are safe or mostly safe to know before production
safe_features = [
    "machine_id",
    "target_quantity",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

# Features that may contain information from the current
# production period or information known after production
risky_features = [
    "total_defects",
    "defect_type_count",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count",
    "production_time_hours"
]

print("\nSAFE FEATURES")
print("-" * 50)

for feature in safe_features:
    print(feature)

print("\nNumber of safe features:", len(safe_features))

print("\nRISKY FEATURES - NEED REVIEW")
print("-" * 50)

for feature in risky_features:
    print(feature)

print("\nNumber of risky features:", len(risky_features))

print("\nTotal features checked:", len(safe_features) + len(risky_features))

FEATURE LEVEL LEAKAGE REVIEW

SAFE FEATURES
--------------------------------------------------
machine_id
target_quantity
production_day
production_month
production_day_of_week
is_weekend
shift_encoded
previous_units_produced
previous_rejection_rate
rolling_3_production_avg
rolling_3_rejection_avg

Number of safe features: 11

RISKY FEATURES - NEED REVIEW
--------------------------------------------------
total_defects
defect_type_count
total_downtime_hours
downtime_event_count
average_downtime_hours
available_hours
operating_hours
utilization_percent
sensor_value_mean
sensor_value_min
sensor_value_max
sensor_value_std
sensor_reading_count
maintenance_event_count
total_maintenance_downtime
average_maintenance_downtime
total_maintenance_cost
completed_maintenance_count
production_time_hours

Number of risky features: 19

Total features checked: 30


In [ ]:
# ============================================================
# CREATE CLEAN PRODUCTION FEATURES
# ============================================================

print("CREATING CLEAN PRODUCTION FEATURES")
print("=" * 70)

# Keep only features that are safe for production prediction
production_safe_features = [
    "machine_id",
    "target_quantity",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

# Create clean training and testing datasets
X_train_production_clean = train_data[production_safe_features].copy()
X_test_production_clean = test_data[production_safe_features].copy()

y_train_production_clean = train_data[production_target].copy()
y_test_production_clean = test_data[production_target].copy()

print("\nClean training shape:")
print(X_train_production_clean.shape)

print("\nClean testing shape:")
print(X_test_production_clean.shape)

print("\nNumber of features:")
print(len(production_safe_features))

print("\nFeatures used:")
for feature in production_safe_features:
    print("-", feature)

print("\nMissing values in training:")
print(X_train_production_clean.isnull().sum().sum())

print("\nMissing values in testing:")
print(X_test_production_clean.isnull().sum().sum())

CREATING CLEAN PRODUCTION FEATURES

Clean training shape:
(4380, 11)

Clean testing shape:
(1095, 11)

Number of features:
11

Features used:
- machine_id
- target_quantity
- production_day
- production_month
- production_day_of_week
- is_weekend
- shift_encoded
- previous_units_produced
- previous_rejection_rate
- rolling_3_production_avg
- rolling_3_rejection_avg

Missing values in training:
0

Missing values in testing:
0


In [ ]:

# LOAD ML DATASETS
# ============================================================

import pandas as pd
import os

X_train = pd.read_csv("../data/processed/X_train_production.csv")
X_test = pd.read_csv("../data/processed/X_test_production.csv")

y_train = pd.read_csv("../data/processed/y_train_production.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test_production.csv").squeeze()

print("PRODUCTION ML DATASETS LOADED")
print("=" * 70)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nMissing values:")
print("X_train:", X_train.isnull().sum().sum())
print("X_test :", X_test.isnull().sum().sum())

print("\nTarget:")
print("units_produced")

print("\nTraining target statistics:")
print(y_train.describe())

PRODUCTION ML DATASETS LOADED

X_train: (360, 32)
X_test : (90, 32)
y_train: (360,)
y_test : (90,)

Missing values:
X_train: 0
X_test : 0

Target:
units_produced

Training target statistics:
count     360.000000
mean      908.686111
std        75.737185
min       670.000000
25%       863.750000
50%       920.000000
75%       959.250000
max      1134.000000
Name: units_produced, dtype: float64


In [ ]:
# ============================================================
# SAVE CLEAN PRODUCTION ML DATASETS
# ============================================================

print("SAVING CLEAN PRODUCTION ML DATASETS")
print("=" * 70)

X_train_production_clean.to_csv(
    "../data/processed/X_train_production_clean.csv",
    index=False
)

X_test_production_clean.to_csv(
    "../data/processed/X_test_production_clean.csv",
    index=False
)

y_train_production_clean.to_csv(
    "../data/processed/y_train_production_clean.csv",
    index=False
)

y_test_production_clean.to_csv(
    "../data/processed/y_test_production_clean.csv",
    index=False
)

print("\nDatasets saved successfully.")

print("\nX_train:", X_train_production_clean.shape)
print("X_test :", X_test_production_clean.shape)
print("y_train:", y_train_production_clean.shape)
print("y_test :", y_test_production_clean.shape)

SAVING CLEAN PRODUCTION ML DATASETS

Datasets saved successfully.

X_train: (4380, 11)
X_test : (1095, 11)
y_train: (4380,)
y_test : (1095,)


In [ ]:

# BASELINE MODEL

from sklearn.linear_model import LinearRegression

# Create the model
model = LinearRegression()

# Train the model using our training data
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

print("Baseline model trained successfully!")
print("Number of predictions:", len(y_pred))

print("\nFirst 10 predictions:")
print(y_pred[:10])

Baseline model trained successfully!
Number of predictions: 90

First 10 predictions:
[940.18044146 926.42833272 874.51631373 928.58180235 905.74486972
 896.29530735 998.19171487 915.52977171 832.26755549 963.5269672 ]


In [ ]:

# CHECKING HOW GOOD OUR BASELINE MODEL IS
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate the errors
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("BASELINE MODEL RESULTS")
print("=" * 70)

print(f"\nMAE  : {mae:.2f} units")
print(f"RMSE : {rmse:.2f} units")
print(f"R²   : {r2:.4f}")

BASELINE MODEL RESULTS

MAE  : 23.92 units
RMSE : 30.56 units
R²   : 0.7719


In [ ]:
# ==================================
# TRYING RANDOM FOREST


from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest model
random_forest = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the model
random_forest.fit(X_train, y_train)

# Predict production for the test data
rf_predictions = random_forest.predict(X_test)

print("Random Forest model trained successfully!")
print("Number of predictions:", len(rf_predictions))

print("\nFirst 10 predictions:")
print(rf_predictions[:10])

Random Forest model trained successfully!
Number of predictions: 90

First 10 predictions:
[933.205 925.07  898.925 948.63  891.67  912.215 990.455 921.625 797.235
 966.775]


In [ ]:

# EVALUATION OF RANDOM FOREST


# Calculate Random Forest errors
rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

rf_r2 = r2_score(y_test, rf_predictions)

print("RANDOM FOREST RESULTS")
print("=" * 70)

print(f"\nMAE  : {rf_mae:.2f} units")
print(f"RMSE : {rf_rmse:.2f} units")
print(f"R²   : {rf_r2:.4f}")

RANDOM FOREST RESULTS

MAE  : 31.28 units
RMSE : 39.73 units
R²   : 0.6143


In [ ]:

# TRYING GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor

# Create the Gradient Boosting model
gradient_boosting = GradientBoostingRegressor
(
    n_estimators=100,
    random_state=42
)

# Train the model
gradient_boosting.fit(X_train, y_train)

# Predict production for the test data
gb_predictions = gradient_boosting.predict(X_test)

print("Gradient Boosting model trained successfully!")
print("Number of predictions:", len(gb_predictions))

print("\nFirst 10 predictions:")
print(gb_predictions[:10])

Gradient Boosting model trained successfully!
Number of predictions: 90

First 10 predictions:
[945.83330549 910.29675154 899.2943974  930.5894455  898.12943067
 898.45597506 993.93157048 917.84737917 797.32885573 968.37750159]


In [ ]:

# CHECK GRADIENT BOOSTING PERFORMANCE
# ============================================================

# Calculate how far the predictions are from the actual values
gb_mae = mean_absolute_error(y_test, gb_predictions)

# Calculate the RMSE
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))

# Calculate the R² score
gb_r2 = r2_score(y_test, gb_predictions)

print("GRADIENT BOOSTING RESULTS")
print("=" * 70)

print(f"\nMAE  : {gb_mae:.2f} units")
print(f"RMSE : {gb_rmse:.2f} units")
print(f"R²   : {gb_r2:.4f}")

GRADIENT BOOSTING RESULTS

MAE  : 30.59 units
RMSE : 37.58 units
R²   : 0.6549


In [ ]:
# VALIDATING THE BEST MODEL

from sklearn.model_selection import cross_val_score

# Check the model using 5 different splits
validation_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("R² scores from 5 validation runs:")
print(validation_scores)

print(f"\nAverage R²: {validation_scores.mean():.4f}")
print(f"Standard deviation: {validation_scores.std():.4f}")

R² scores from 5 validation runs:
[0.81873921 0.86650132 0.85997447 0.80226233 0.79297371]

Average R²: 0.8281
Standard deviation: 0.0299


In [ ]:
# Training the final Linear Regression model again

final_model = LinearRegression()

final_model.fit(X_train, y_train)

print("Final Linear Regression model trained successfully.")

Final Linear Regression model trained successfully.


In [ ]:
# Using the final model to predict production for the test data

final_predictions = final_model.predict(X_test)

print("Final production predictions generated successfully.")
print("Number of predictions:", len(final_predictions))

print("\nFirst 10 predictions:")
print(final_predictions[:10])

Final production predictions generated successfully.
Number of predictions: 90

First 10 predictions:
[940.18044146 926.42833272 874.51631373 928.58180235 905.74486972
 896.29530735 998.19171487 915.52977171 832.26755549 963.5269672 ]


In [ ]:
# Created a file with actual and predicted production values

production_predictions = pd.DataFrame({
    "actual_production": y_test.values,
    "predicted_production": final_predictions
})

# Calculate the difference between actual and predicted values
production_predictions["prediction_error"] = (
    production_predictions["actual_production"]
    - production_predictions["predicted_production"]
)

# Save the predictions
production_predictions.to_csv(
    "../data/processed/production_predictions.csv",
    index=False
)

print("Production predictions saved successfully.")
print("File: ../data/processed/production_predictions.csv")
print("Rows:", len(production_predictions))
print("Columns:", production_predictions.columns.tolist())

print("\nFirst 10 rows:")
print(production_predictions.head(10))

Production predictions saved successfully.
File: ../data/processed/production_predictions.csv
Rows: 90
Columns: ['actual_production', 'predicted_production', 'prediction_error']

First 10 rows:
   actual_production  predicted_production  prediction_error
0                931            940.180441         -9.180441
1                950            926.428333         23.571667
2                871            874.516314         -3.516314
3                899            928.581802        -29.581802
4                891            905.744870        -14.744870
5                900            896.295307          3.704693
6               1003            998.191715          4.808285
7                907            915.529772         -8.529772
8                845            832.267555         12.732445
9                948            963.526967        -15.526967


In [ ]:
# Load the feature engineered file

import pandas as pd

# Load the feature engineered file
quality_data = pd.read_csv("../data/processed/feature_engineered_data.csv")

print("Quality data loaded")
print("Shape:", quality_data.shape)

# Check the target column
print("\nTarget column: quality_defect_rate_percent")

# Check if the target has any missing values
print("Missing values:", quality_data["quality_defect_rate_percent"].isnull().sum())

Quality data loaded
Shape: (450, 42)

Target column: quality_defect_rate_percent
Missing values: 0


In [ ]:
# Set the column we want to predict
target_column = "quality_defect_rate_percent"

# Columns that are not useful for prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    "quality_defect_rate_percent"
]

# Create X by removing the columns above
X_quality = quality_data.drop(columns=columns_to_remove)

# Create y using the quality defect rate
y_quality = quality_data[target_column]

print("Quality prediction dataset created")

print("\nX shape:", X_quality.shape)
print("y shape:", y_quality.shape)

print("\nTarget:")
print(target_column)

print("\nFeatures:")
print(X_quality.columns.tolist())

Quality prediction dataset created

X shape: (450, 37)
y shape: (450,)

Target:
quality_defect_rate_percent

Features:
['machine_id', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# Check if anything is missing
print("Missing values before handling:")
print(X_quality.isnull().sum()[X_quality.isnull().sum() > 0])

# Convert shift into numbers
X_quality["shift"] = X_quality["shift"].map({
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
})

# Check again after converting shift
print("\nMissing values after handling:")
print(X_quality.isnull().sum()[X_quality.isnull().sum() > 0])

print("\nFinal X shape:", X_quality.shape)
print("Target shape:", y_quality.shape)

Missing values before handling:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Missing values after handling:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Final X shape: (450, 37)
Target shape: (450,)


In [ ]:
# Fill the missing historical values with the median
history_columns = [
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

for column in history_columns:
    X_quality[column] = X_quality[column].fillna(X_quality[column].median())

# Check if any missing values are left
print("Missing values after handling:")
print(X_quality.isnull().sum().sum())

Missing values after handling:
0


In [ ]:
# Spliting training and testing data

from sklearn.model_selection import train_test_split

# Split the data into training and testing parts
X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality,
    y_quality,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Training data: (360, 37)
Testing data: (90, 37)
Training target: (360,)
Testing target: (90,)


In [ ]:
# Make sure the target is not inside our features
print("Target inside X:", target_column in X_quality.columns)

# Check for missing values
print("Missing values in training data:", X_train_quality.isnull().sum().sum())
print("Missing values in testing data:", X_test_quality.isnull().sum().sum())

# Check if training and testing rows overlap
print("Training/testing overlap:", len(
    set(X_train_quality.index) & set(X_test_quality.index)
))

Target inside X: False
Missing values in training data: 0
Missing values in testing data: 0
Training/testing overlap: 0


In [ ]:
# Train our first model


from sklearn.linear_model import LinearRegression

# Create the model
quality_model = LinearRegression()

# Train it using the training data
quality_model.fit(X_train_quality, y_train_quality)

# Predict the defect rate for the test data
quality_predictions = quality_model.predict(X_test_quality)

print("Quality prediction model trained successfully.")
print("Number of predictions:", len(quality_predictions))

print("\nFirst 10 predictions:")
print(quality_predictions[:10])


Quality prediction model trained successfully.
Number of predictions: 90

First 10 predictions:
[1.49101427 0.51753036 1.82275276 1.44424995 0.96451381 0.84885585
 1.46005733 2.07510161 0.35091246 0.20689628]


In [ ]:
# Evaluate the quality model


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Check how far our predictions are from the actual values
quality_mae = mean_absolute_error(y_test_quality, quality_predictions)

quality_rmse = np.sqrt

(


    mean_squared_error(y_test_quality, quality_predictions)
)

quality_r2 = r2_score(y_test_quality, quality_predictions)

print("Quality prediction results")
print("=" * 50)

print(f"MAE  : {quality_mae:.4f}%")
print(f"RMSE : {quality_rmse:.4f}%")
print(f"R²   : {quality_r2:.4f}")

Quality prediction results
MAE  : 0.0186%
RMSE : 0.0299%
R²   : 0.9979


In [ ]:
# Removed columns that directly reveal the quality result
quality_leakage_columns = [
    "total_defects",
    "total_output"
]

X_quality_clean = X_quality.drop(columns=quality_leakage_columns)

print("Leakage columns removed")
print("New X shape:", X_quality_clean.shape)

Leakage columns removed
New X shape: (450, 35)


In [ ]:
# Spliting the clean data into training and testing parts
X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality_clean,
    y_quality,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Training data: (360, 35)
Testing data: (90, 35)
Training target: (360,)
Testing target: (90,)


In [ ]:
# Create a fresh model
quality_model = LinearRegression()

# Train the model
quality_model.fit(X_train_quality, y_train_quality)

# Predict the defect rate
quality_predictions = quality_model.predict(X_test_quality)

print("Quality model trained successfully")
print("Number of predictions:", len(quality_predictions))

print("\nFirst 10 predictions:")
print(quality_predictions[:10])

Quality model trained successfully
Number of predictions: 90

First 10 predictions:
[1.79803207 0.29326821 1.31467828 1.89426956 0.65344151 0.63210098
 1.18336358 1.39335174 0.44170308 0.21127675]


In [ ]:
# Check how accurate the new predictions are
quality_mae = mean_absolute_error(y_test_quality, quality_predictions)

quality_rmse = np.sqrt(
    mean_squared_error(y_test_quality, quality_predictions)
)

quality_r2 = r2_score(y_test_quality, quality_predictions)

print("Clean quality model results")
print("=" * 50)

print(f"MAE  : {quality_mae:.4f}%")
print(f"RMSE : {quality_rmse:.4f}%")
print(f"R²   : {quality_r2:.4f}")

Clean quality model results
MAE  : 0.2921%
RMSE : 0.3747%
R²   : 0.6762


In [ ]:
# Random Forest try

from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest model
quality_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the model
quality_rf.fit(X_train_quality, y_train_quality)

# Predict the defect rate
quality_rf_predictions = quality_rf.predict(X_test_quality)

print("Random Forest quality model trained successfully")
print("Number of predictions:", len(quality_rf_predictions))

print("\nFirst 10 predictions:")
print(quality_rf_predictions[:10])

Random Forest quality model trained successfully
Number of predictions: 90

First 10 predictions:
[1.84265 0.3851  1.13745 1.4385  0.7303  0.65795 1.03645 1.5874  0.38615
 0.24915]


In [ ]:
# Check how accurate the Random Forest predictions are
quality_rf_mae = mean_absolute_error(
    y_test_quality,
    quality_rf_predictions
)

quality_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_quality,
        quality_rf_predictions
    )
)

quality_rf_r2 = r2_score(
    y_test_quality,
    quality_rf_predictions
)

print("Random Forest quality results")
print("=" * 50)

print(f"MAE  : {quality_rf_mae:.4f}%")
print(f"RMSE : {quality_rf_rmse:.4f}%")
print(f"R²   : {quality_rf_r2:.4f}")

Random Forest quality results
MAE  : 0.2859%
RMSE : 0.3728%
R²   : 0.6794


In [ ]:
# Validate Random Forest
from sklearn.model_selection import cross_val_score

# Check how the model performs on different parts of the data
quality_validation_scores = cross_val_score(
    quality_rf,
    X_train_quality,
    y_train_quality,
    cv=5,
    scoring="r2"
)

print("Random Forest validation results")
print("=" * 50)

print("\nR² scores:")
print(quality_validation_scores)

print(f"\nAverage R²: {quality_validation_scores.mean():.4f}")
print(f"Standard deviation: {quality_validation_scores.std():.4f}")

Random Forest validation results

R² scores:
[0.72091587 0.69546918 0.60359698 0.60219383 0.7079197 ]

Average R²: 0.6660
Standard deviation: 0.0522


In [ ]:
# Create the final quality model
final_quality_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the final model
final_quality_model.fit(
    X_train_quality,
    y_train_quality
)

print("Final quality model trained successfully")

Final quality model trained successfully


In [ ]:
# Generate the final defect rate predictions
final_quality_predictions = final_quality_model.predict(X_test_quality)

print("Final quality predictions generated")
print("Number of predictions:", len(final_quality_predictions))

print("\nFirst 10 predictions:")
print(final_quality_predictions[:10])

Final quality predictions generated
Number of predictions: 90

First 10 predictions:
[1.84265 0.3851  1.13745 1.4385  0.7303  0.65795 1.03645 1.5874  0.38615
 0.24915]


In [ ]:
# Create a file with actual and predicted defect rates
quality_predictions_data = pd.DataFrame({
    "actual_defect_rate": y_test_quality.values,
    "predicted_defect_rate": final_quality_predictions
})

# Calculate the prediction error
quality_predictions_data["prediction_error"] = (
    quality_predictions_data["actual_defect_rate"]
    - quality_predictions_data["predicted_defect_rate"]
)

# Save the predictions
quality_predictions_data.to_csv(
    "../data/processed/quality_predictions.csv",
    index=False
)

print("Quality predictions saved successfully")
print("File: ../data/processed/quality_predictions.csv")
print("Rows:", len(quality_predictions_data))
print("Columns:", quality_predictions_data.columns.tolist())

print("\nFirst 10 rows:")
print(quality_predictions_data.head(10))

Quality predictions saved successfully
File: ../data/processed/quality_predictions.csv
Rows: 90
Columns: ['actual_defect_rate', 'predicted_defect_rate', 'prediction_error']

First 10 rows:
   actual_defect_rate  predicted_defect_rate  prediction_error
0                1.50                1.84265          -0.34265
1                0.52                0.38510           0.13490
2                1.85                1.13745           0.71255
3                1.45                1.43850           0.01150
4                0.98                0.73030           0.24970
5                0.87                0.65795           0.21205
6                1.43                1.03645           0.39355
7                2.08                1.58740           0.49260
8                0.35                0.38615          -0.03615
9                0.21                0.24915          -0.03915


In [ ]:
# Loaded the feature engineered data
failure_data = pd.read_csv("../data/processed/feature_engineered_data.csv")

print("Failure prediction data loaded")
print("Shape:", failure_data.shape)

# Check the maintenance related columns
print("\nMaintenance columns:")
print([
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
])

# Check some machine related data
print("\nSample data:")
print(failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "utilization_percent",
        "sensor_value_mean",
        "maintenance_event_count",
        "completed_maintenance_count"
    ]
].head(10))

Failure prediction data loaded
Shape: (450, 42)

Maintenance columns:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

Sample data:
   machine_id  total_downtime_hours  utilization_percent  sensor_value_mean  \
0           1                  7.06                99.02              36.48   
1           1                  7.06                99.02              36.48   
2           1                  7.06                99.02              36.48   
3           1                  7.06                99.02              36.48   
4           1                  7.06                99.02              36.48   
5           1                  7.06                99.02              36.48   
6           1                  7.06                99.02              36.48   
7           1                  7.06                99.02              36.48   
8           1                  7.06                99.02  

In [ ]:
# Checking the machine related values

machine_data = failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "utilization_percent",
        "sensor_value_mean",
        "sensor_value_max",
        "sensor_value_std",
        "maintenance_event_count",
        "completed_maintenance_count"
    ]
].drop_duplicates()

print("Machine data:")
print(machine_data)

print("\nNumber of unique machines:", machine_data["machine_id"].nunique())

Machine data:
     machine_id  total_downtime_hours  utilization_percent  sensor_value_mean  \
0             1                  7.06                99.02              36.48   
90            2                 15.49                97.85              35.47   
180           3                 26.39                96.33              43.64   
270           4                 16.30                97.74              33.98   
360           5                  5.62                99.22              34.90   

     sensor_value_max  sensor_value_std  maintenance_event_count  \
0               76.56             33.65                        2   
90              75.40             32.86                        2   
180             88.87             38.73                        3   
270             75.49             31.59                        2   
360             75.02             32.34                        4   

     completed_maintenance_count  
0                              1  
90                  

In [ ]:
# Checking the columns available in the maintenance data

print("Columns in failure data:")
print(failure_data.columns.tolist())

Columns in failure data:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# Checking the maintenance data columns
print(maintenance_df.columns.tolist())

# See a few maintenance records
print(maintenance_df.head())

NameError: name 'maintenance_df' is not defined

In [ ]:

print(failure_data.columns.tolist())

['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
import os

# Check the files available in the data folders
print("Raw data files:")
print(os.listdir("../data/raw"))

print("\nProcessed data files:")
print(os.listdir("../data/processed"))

Raw data files:
[]

Processed data files:
['feature_engineered_data.csv', 'production_predictions.csv', 'quality_predictions.csv', 'X_test_production.csv', 'X_train_production.csv', 'y_test_production.csv', 'y_train_production.csv']


In [ ]:
# Check the values of important features for failure risk

risk_columns = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_mean",
    "sensor_value_max",
    "sensor_value_std",
    "utilization_percent",
    "maintenance_event_count",
    "total_maintenance_downtime"
]

print(failure_data[risk_columns].describe())

       total_downtime_hours  downtime_event_count  sensor_value_mean  \
count            450.000000            450.000000         450.000000   
mean              14.172000              6.400000          36.894000   
std                7.481768              3.502465           3.473014   
min                5.620000              3.000000          33.980000   
25%                7.060000              4.000000          34.900000   
50%               15.490000              6.000000          35.470000   
75%               16.300000              6.000000          36.480000   
max               26.390000             13.000000          43.640000   

       sensor_value_max  sensor_value_std  utilization_percent  \
count        450.000000        450.000000           450.000000   
mean          78.268000         33.834000            98.032000   
std            5.331561          2.541365             1.040645   
min           75.020000         31.590000            96.330000   
25%           75.4000

In [ ]:
# Keep only one row for each machine

machine_risk_data = failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "downtime_event_count",
        "sensor_value_mean",
        "sensor_value_max",
        "sensor_value_std",
        "utilization_percent",
        "maintenance_event_count",
        "total_maintenance_downtime"
    ]
].drop_duplicates()

print("Machine level risk data")
print(machine_risk_data)

Machine level risk data
     machine_id  total_downtime_hours  downtime_event_count  \
0             1                  7.06                     4   
90            2                 15.49                     6   
180           3                 26.39                    13   
270           4                 16.30                     6   
360           5                  5.62                     3   

     sensor_value_mean  sensor_value_max  sensor_value_std  \
0                36.48             76.56             33.65   
90               35.47             75.40             32.86   
180              43.64             88.87             38.73   
270              33.98             75.49             31.59   
360              34.90             75.02             32.34   

     utilization_percent  maintenance_event_count  total_maintenance_downtime  
0                  99.02                        2                        6.74  
90                 97.85                        2               

In [ ]:
# Selectin the important values used for failure risk

risk_data = machine_risk_data[
    [
        "machine_id",
        "total_downtime_hours",
        "downtime_event_count",
        "sensor_value_max",
        "sensor_value_std",
        "maintenance_event_count"
    ]
].copy()

print("Risk data prepared")
print(risk_data)

Risk data prepared
     machine_id  total_downtime_hours  downtime_event_count  sensor_value_max  \
0             1                  7.06                     4             76.56   
90            2                 15.49                     6             75.40   
180           3                 26.39                    13             88.87   
270           4                 16.30                     6             75.49   
360           5                  5.62                     3             75.02   

     sensor_value_std  maintenance_event_count  
0               33.65                        2  
90              32.86                        2  
180             38.73                        3  
270             31.59                        2  
360             32.34                        4  


In [ ]:
# Import MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# These are the columns used to calculate failure risk
risk_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Create the scaler
scaler = MinMaxScaler()

# Convert the values into a scale from 0 to 1
risk_data[risk_features] = scaler.fit_transform(
    risk_data[risk_features]
)

print("Risk data after normalization")
print(risk_data)

ImportError: DLL load failed while importing _group_columns: An Application Control policy has blocked this file.

In [ ]:
# Columns used for failure risk calculation

risk_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Normalize each column between 0 and 1
for column in risk_features:
    minimum = risk_data[column].min()
    maximum = risk_data[column].max()

    risk_data[column] = (
        (risk_data[column] - minimum)
        / (maximum - minimum)
    )

print("Risk data after normalization")
print(risk_data)

Risk data after normalization
     machine_id  total_downtime_hours  downtime_event_count  sensor_value_max  \
0             1              0.069331                   0.1          0.111191   
90            2              0.475205                   0.3          0.027437   
180           3              1.000000                   1.0          1.000000   
270           4              0.514203                   0.3          0.033935   
360           5              0.000000                   0.0          0.000000   

     sensor_value_std  maintenance_event_count  
0            0.288515                      0.0  
90           0.177871                      0.0  
180          1.000000                      0.5  
270          0.000000                      0.0  
360          0.105042                      1.0  


In [ ]:
# Calculate the average risk score for each machine

risk_data["failure_risk_score"] = risk_data[
    risk_features
].mean(axis=1)

print("Failure risk scores")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score"
        ]
    ]
)

Failure risk scores
     machine_id  failure_risk_score
0             1            0.113808
90            2            0.196103
180           3            0.900000
270           4            0.169628
360           5            0.221008


In [ ]:
# Classification of each machine based on its failure risk score

def get_risk_level(score):
    if score < 0.20:
        return "Low Risk"
    elif score < 0.50:
        return "Medium Risk"
    else:
        return "High Risk"


# Add the risk level to the data
risk_data["failure_risk"] = risk_data[
    "failure_risk_score"
].apply(get_risk_level)

print("Failure risk classification")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk"
        ]
    ]
)

Failure risk classification
     machine_id  failure_risk_score failure_risk
0             1            0.113808     Low Risk
90            2            0.196103     Low Risk
180           3            0.900000    High Risk
270           4            0.169628     Low Risk
360           5            0.221008  Medium Risk


In [ ]:
# Finding the main reason for the machine risk

def get_risk_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "Normal operating conditions"

    return ", ".join(reasons)


risk_data["risk_reason"] = risk_data.apply(
    get_risk_reason,
    axis=1
)

print("Failure risk results")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk",
            "risk_reason"
        ]
    ]
)

Failure risk results
     machine_id  failure_risk_score failure_risk                  risk_reason
0             1            0.113808     Low Risk  Normal operating conditions
90            2            0.196103     Low Risk  Normal operating conditions
180           3            0.900000    High Risk  Normal operating conditions
270           4            0.169628     Low Risk  Normal operating conditions
360           5            0.221008  Medium Risk  Normal operating conditions


In [ ]:
# Adding the  reasons based on the original machine values

def get_risk_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "Normal operating conditions"

    return ", ".join(reasons)


# Get the reasons from the original values
machine_risk_data["risk_reason"] = machine_risk_data.apply(
    get_risk_reason,
    axis=1
)

# Add the reasons to our risk results
risk_data["risk_reason"] = machine_risk_data[
    "risk_reason"
].values

print("Final failure risk results")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk",
            "risk_reason"
        ]
    ]
)

Final failure risk results
     machine_id  failure_risk_score failure_risk  \
0             1            0.113808     Low Risk   
90            2            0.196103     Low Risk   
180           3            0.900000    High Risk   
270           4            0.169628     Low Risk   
360           5            0.221008  Medium Risk   

                                           risk_reason  
0                          Normal operating conditions  
90                         Normal operating conditions  
180  High downtime, Frequent downtime, High sensor ...  
270                        Normal operating conditions  
360                               Frequent maintenance  


In [ ]:
# Selecting the final columns we need

final_risk_results = risk_data[
    [
        "machine_id",
        "failure_risk_score",
        "failure_risk",
        "risk_reason"
    ]
].copy()

# Save the results
final_risk_results.to_csv(
    "../data/processed/failure_risk_predictions.csv",
    index=False
)

print("Failure risk results saved successfully")
print("File: ../data/processed/failure_risk_predictions.csv")
print("Rows:", len(final_risk_results))
print("Columns:", final_risk_results.columns.tolist())

print("\nFinal results:")
print(final_risk_results)

Failure risk results saved successfully
File: ../data/processed/failure_risk_predictions.csv
Rows: 5
Columns: ['machine_id', 'failure_risk_score', 'failure_risk', 'risk_reason']

Final results:
     machine_id  failure_risk_score failure_risk  \
0             1            0.113808     Low Risk   
90            2            0.196103     Low Risk   
180           3            0.900000    High Risk   
270           4            0.169628     Low Risk   
360           5            0.221008  Medium Risk   

                                           risk_reason  
0                          Normal operating conditions  
90                         Normal operating conditions  
180  High downtime, Frequent downtime, High sensor ...  
270                        Normal operating conditions  
360                               Frequent maintenance  


In [ ]:
# Checking all the final prediction files

production_results = pd.read_csv(
    "../data/processed/production_predictions.csv"
)

quality_results = pd.read_csv(
    "../data/processed/quality_predictions.csv"
)

failure_results = pd.read_csv(
    "../data/processed/failure_risk_predictions.csv"
)

print("Production results:", production_results.shape)
print("Quality results:", quality_results.shape)
print("Failure risk results:", failure_results.shape)

print("\nProduction columns:")
print(production_results.columns.tolist())

print("\nQuality columns:")
print(quality_results.columns.tolist())

print("\nFailure risk columns:")
print(failure_results.columns.tolist())

Production results: (90, 3)
Quality results: (90, 3)
Failure risk results: (5, 4)

Production columns:
['actual_production', 'predicted_production', 'prediction_error']

Quality columns:
['actual_defect_rate', 'predicted_defect_rate', 'prediction_error']

Failure risk columns:
['machine_id', 'failure_risk_score', 'failure_risk', 'risk_reason']


In [ ]:
# Checking the variables currently available in the notebook

print("Variables containing 'model':")

for name in globals():
    if "model" in name.lower():
        print(name)

Variables containing 'model':


In [ ]:
# Re-Loading linear regression model

X_train_production = pd.read_csv(
    "../data/processed/X_train_production.csv"
)

X_test_production = pd.read_csv(
    "../data/processed/X_test_production.csv"
)

y_train_production = pd.read_csv(
    "../data/processed/y_train_production.csv"
).squeeze()

y_test_production = pd.read_csv(
    "../data/processed/y_test_production.csv"
).squeeze()

print("Production data loaded")

print("X_train:", X_train_production.shape)
print("X_test:", X_test_production.shape)
print("y_train:", y_train_production.shape)
print("y_test:", y_test_production.shape)

Production data loaded
X_train: (360, 32)
X_test: (90, 32)
y_train: (360,)
y_test: (90,)


In [ ]:
# Train the final production model

from sklearn.linear_model import LinearRegression

final_production_model = LinearRegression()

final_production_model.fit(
    X_train_production,
    y_train_production
)

print("Final production model trained successfully")

Final production model trained successfully


In [ ]:
# Save the final production model

import joblib

joblib.dump(
    final_production_model,
    "../data/processed/production_model.pkl"
)

print("Production model saved successfully")
print("File: ../data/processed/production_model.pkl")

Production model saved successfully
File: ../data/processed/production_model.pkl


In [ ]:
# Loading the saved quality training and testing data

X_train_quality = pd.read_csv(
    "../data/processed/X_train_quality.csv"
)

X_test_quality = pd.read_csv(
    "../data/processed/X_test_quality.csv"
)

y_train_quality = pd.read_csv(
    "../data/processed/y_train_quality.csv"
).squeeze()

y_test_quality = pd.read_csv(
    "../data/processed/y_test_quality.csv"
).squeeze()

print("Quality data loaded")

print("X_train:", X_train_quality.shape)
print("X_test:", X_test_quality.shape)
print("y_train:", y_train_quality.shape)
print("y_test:", y_test_quality.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/X_train_quality.csv'

In [ ]:
# Checking the quality data variables available right now

print("Quality variables:")

for name in globals():
    if "quality" in name.lower() or "X_train" in name:
        print(name)

Quality variables:
quality
quality_results
X_train_production


In [ ]:
# Check for the quality dataset

print("Quality data shape:", quality.shape)

print("\nQuality columns:")
print(quality.columns.tolist())

Quality data shape: (450, 6)

Quality columns:
['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']


In [ ]:
# Loading the feature engineered data

data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("Feature engineered data loaded")
print("Shape:", data.shape)

print("\nChecking quality target:")
print("quality_defect_rate_percent" in data.columns)

Feature engineered data loaded
Shape: (450, 42)

Checking quality target:
True


In [ ]:
# Seting the target column


quality_target = "quality_defect_rate_percent"

# Columns that should not be used for prediction
columns_to_remove = [
    "quality_defect_rate_percent",
    "production_id",
    "target_id",
    "target_date",
    "production_date"
]

# Create the input data and target
X_quality = data.drop(columns=columns_to_remove)
y_quality = data[quality_target]

print("Quality dataset prepared")
print("X shape:", X_quality.shape)
print("y shape:", y_quality.shape)

print("\nTarget:", quality_target)

Quality dataset prepared
X shape: (450, 37)
y shape: (450,)

Target: quality_defect_rate_percent


In [ ]:
# Removing the columns that can cause data leakage

leakage_columns = [
    "total_defects",
    "defect_type_count"
]

X_quality = X_quality.drop(columns=leakage_columns)

print("Leakage columns removed")
print("New X shape:", X_quality.shape)

Leakage columns removed
New X shape: (450, 35)


In [ ]:
# Split the quality data into training and testing data

from sklearn.model_selection import train_test_split

X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality,
    y_quality,
    test_size=0.20,
    random_state=42
)

print("Quality data split completed")

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)
print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Quality data split completed
Training data: (360, 35)
Testing data: (90, 35)
Training target: (360,)
Testing target: (90,)


In [ ]:


X_train_quality = X_train_quality.drop(columns=["shift"])
X_test_quality = X_test_quality.drop(columns=["shift"])

print("Text column removed")
print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

Text column removed
Training data: (360, 34)
Testing data: (90, 34)


In [ ]:
# Train the final quality model

from sklearn.ensemble import RandomForestRegressor

final_quality_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

final_quality_model.fit(
    X_train_quality,
    y_train_quality
)

print("Final quality model trained successfully")

Final quality model trained successfully


In [ ]:
# Saving

import joblib

joblib.dump(
    final_quality_model,
    "../data/processed/quality_model.pkl"
)

print("Quality model saved successfully")
print("File: ../data/processed/quality_model.pkl")

Quality model saved successfully
File: ../data/processed/quality_model.pkl


In [ ]:
# Checking the final files in the processed folder

import os

files = os.listdir("../data/processed")

print("Final processed files:")

for file in files:
    print(file)

Final processed files:
failure_risk_predictions.csv
feature_engineered_data.csv
production_model.pkl
production_predictions.csv
quality_model.pkl
quality_predictions.csv
X_test_production.csv
X_train_production.csv
y_test_production.csv
y_train_production.csv


In [ ]:
# Checking the current notebook variables

print("Data + ML work completed")
print("Production model:", type(final_production_model).__name__)
print("Quality model:", type(final_quality_model).__name__)
print("Failure risk results:", len(final_risk_results))

Data + ML work completed
Production model: LinearRegression
Quality model: RandomForestRegressor
Failure risk results: 5


In [ ]:
# ANOMOLY DETECTION
# Selecting the values we want to use for anomaly detection

anomaly_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Create a separate dataframe for anomaly detection
anomaly_data = machine_risk_data[
    ["machine_id"] + anomaly_features
].copy()

print("Anomaly detection data prepared")
print("Shape:", anomaly_data.shape)

print("\nAnomaly features:")
print(anomaly_features)

print("\nSample data:")
print(anomaly_data)

Anomaly detection data prepared
Shape: (5, 8)

Anomaly features:
['total_downtime_hours', 'downtime_event_count', 'utilization_percent', 'sensor_value_mean', 'sensor_value_max', 'sensor_value_std', 'maintenance_event_count']

Sample data:
     machine_id  total_downtime_hours  downtime_event_count  \
0             1                  7.06                     4   
90            2                 15.49                     6   
180           3                 26.39                    13   
270           4                 16.30                     6   
360           5                  5.62                     3   

     utilization_percent  sensor_value_mean  sensor_value_max  \
0                  99.02              36.48             76.56   
90                 97.85              35.47             75.40   
180                96.33              43.64             88.87   
270                97.74              33.98             75.49   
360                99.22              34.90             7

In [ ]:
# Training the anomaly detection model

from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(
    contamination=0.20,
    random_state=42
)

anomaly_model.fit(
    anomaly_data[anomaly_features]
)

print("Anomaly detection model trained successfully")

Anomaly detection model trained successfully


In [ ]:
# Prediction of  whether each machine is normal or abnormal

anomaly_data["anomaly_prediction"] = anomaly_model.predict(
    anomaly_data[anomaly_features]
)

# Get the anomaly score
anomaly_data["anomaly_score"] = anomaly_model.decision_function(
    anomaly_data[anomaly_features]
)

# Convert the prediction into an easy-to-read result
anomaly_data["anomaly_status"] = anomaly_data[
    "anomaly_prediction"
].map({
    1: "Normal",
    -1: "Anomaly"
})

print("Anomaly results generated")
print(
    anomaly_data[
        [
            "machine_id",
            "anomaly_score",
            "anomaly_status"
        ]
    ]
)

Anomaly results generated
     machine_id  anomaly_score anomaly_status
0             1       0.048116         Normal
90            2       0.148483         Normal
180           3      -0.125763        Anomaly
270           4       0.103202         Normal
360           5       0.031441         Normal


In [ ]:
# simple reason for the anomaly

def get_anomaly_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "No unusual behaviour detected"

    return ", ".join(reasons)


anomaly_data["anomaly_reason"] = anomaly_data.apply(
    get_anomaly_reason,
    axis=1
)

print("Anomaly explanations added")
print(
    anomaly_data[
        [
            "machine_id",
            "anomaly_score",
            "anomaly_status",
            "anomaly_reason"
        ]
    ]
)

Anomaly explanations added
     machine_id  anomaly_score anomaly_status  \
0             1       0.048116         Normal   
90            2       0.148483         Normal   
180           3      -0.125763        Anomaly   
270           4       0.103202         Normal   
360           5       0.031441         Normal   

                                        anomaly_reason  
0                        No unusual behaviour detected  
90                       No unusual behaviour detected  
180  High downtime, Frequent downtime, High sensor ...  
270                      No unusual behaviour detected  
360                               Frequent maintenance  


In [ ]:
# Selection of  the final anomaly results

final_anomaly_results = anomaly_data[
    [
        "machine_id",
        "anomaly_score",
        "anomaly_status",
        "anomaly_reason"
    ]
].copy()

# Save the results
final_anomaly_results.to_csv(
    "../data/processed/anomaly_detection_results.csv",
    index=False
)

print("Anomaly results saved successfully")
print("File: ../data/processed/anomaly_detection_results.csv")
print("Rows:", len(final_anomaly_results))
print("Columns:", final_anomaly_results.columns.tolist())

Anomaly results saved successfully
File: ../data/processed/anomaly_detection_results.csv
Rows: 5
Columns: ['machine_id', 'anomaly_score', 'anomaly_status', 'anomaly_reason']


In [ ]:
# Checking the final ML output 

import os

processed_path = "../data/processed"

files_to_check = 
[
    "feature_engineered_data.csv",
    "production_predictions.csv",
    "quality_predictions.csv",
    "failure_risk_predictions.csv",
    "anomaly_detection_results.csv",
    "production_model.pkl",
    "quality_model.pkl"
]

print("Final Data + ML files")

for file in files_to_check:
    file_path = os.path.join(processed_path, file)

    if os.path.exists(file_path):
        print(file, "- PERFECTLY OK")
    else:
        print(file, "- Missing")

Final Data + ML files
feature_engineered_data.csv - PERFECTLY OK
production_predictions.csv - PERFECTLY OK
quality_predictions.csv - PERFECTLY OK
failure_risk_predictions.csv - PERFECTLY OK
anomaly_detection_results.csv - PERFECTLY OK
production_model.pkl - PERFECTLY OK
quality_model.pkl - PERFECTLY OK


In [ ]:
print("Dataset shape:", data.shape)

AttributeError: 'dict' object has no attribute 'shape'

In [ ]:
import pandas as pd

final_dataset = pd.read_csv("../data/processed/feature_engineered_data.csv")

print("Dataset shape:", final_dataset.shape)
print("Total records:", final_dataset.shape[0])
print("Total columns:", final_dataset.shape[1])

final_dataset.head()

Dataset shape: (450, 42)
Total records: 450
Total columns: 42


,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours,target_id,target_date,target_quantity,...,completed_maintenance_count,production_day,production_month,production_day_of_week,is_weekend,shift_encoded,previous_units_produced,previous_rejection_rate,rolling_3_production_avg,rolling_3_rejection_avg
0,1,1,2026-07-20,Morning,969,59,8.0,1,2026-07-20,998,...,1,20,7,0,0,0,NaN,NaN,NaN,NaN
1,2,1,2026-07-20,Afternoon,944,48,8.0,2,2026-07-20,1012,...,1,20,7,0,0,1,969.0,5.74,969.0,5.74
2,3,1,2026-07-20,Night,940,13,8.0,3,2026-07-20,1003,...,1,20,7,0,0,2,944.0,4.84,956.5,5.29
3,16,1,2026-07-21,Morning,954,20,8.0,16,2026-07-21,974,...,1,21,7,1,0,0,940.0,1.36,951.0,3.98
4,17,1,2026-07-21,Afternoon,925,53,8.0,17,2026-07-21,978,...,1,21,7,1,0,1,954.0,2.05,946.0,2.75


In [ ]:
final_dataset.shape

NameError: name 'final_dataset' is not defined

In [ ]:
# ============================================================
# PRODUCTION PREDICTION - LINEAR REGRESSION
# ============================================================

from sklearn.linear_model import LinearRegression

print("PRODUCTION PREDICTION - LINEAR REGRESSION")
print("=" * 70)

# Create the model
production_linear_model = LinearRegression()

# Train using the new clean 365-day dataset
production_linear_model.fit(
    X_train_production_clean,
    y_train_production_clean
)

# Make predictions on the new test data
production_linear_predictions = production_linear_model.predict(
    X_test_production_clean
)

print("\nModel training completed successfully.")

print("\nNumber of predictions:")
print(len(production_linear_predictions))

print("\nFirst 10 predictions:")
print(production_linear_predictions[:10])

PRODUCTION PREDICTION - LINEAR REGRESSION

Model training completed successfully.

Number of predictions:
1095

First 10 predictions:
[949.46634496 953.058223   948.73852836 903.74064736 910.54705933
 889.80220898 797.88650012 799.34978619 785.73366783 975.76508846]


In [ ]:
# ============================================================
# EVALUATE PRODUCTION LINEAR REGRESSION
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("PRODUCTION LINEAR REGRESSION - MODEL EVALUATION")
print("=" * 70)

# Calculate evaluation metrics
mae = mean_absolute_error(
    y_test_production_clean,
    production_linear_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_production_clean,
        production_linear_predictions
    )
)

r2 = r2_score(
    y_test_production_clean,
    production_linear_predictions
)

print("\nModel Performance:")
print("-" * 50)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

PRODUCTION LINEAR REGRESSION - MODEL EVALUATION

Model Performance:
--------------------------------------------------
MAE : 33.22
RMSE: 41.63
R²  : 0.6865


In [ ]:
# ============================================================
# PRODUCTION PREDICTION - RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestRegressor

print("PRODUCTION PREDICTION - RANDOM FOREST")
print("=" * 70)

# Create the Random Forest model
production_rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model
production_rf_model.fit(
    X_train_production_clean,
    y_train_production_clean
)

# Make predictions
production_rf_predictions = production_rf_model.predict(
    X_test_production_clean
)

print("\nModel training completed successfully.")

print("\nNumber of predictions:")
print(len(production_rf_predictions))

print("\nFirst 10 predictions:")
print(production_rf_predictions[:10])

PRODUCTION PREDICTION - RANDOM FOREST

Model training completed successfully.

Number of predictions:
1095

First 10 predictions:
[945.155 940.715 934.61  913.565 904.475 907.78  804.875 787.975 791.335
 977.025]


In [ ]:
# ============================================================
# EVALUATE PRODUCTION RANDOM FOREST
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("PRODUCTION RANDOM FOREST - MODEL EVALUATION")
print("=" * 70)

# Calculate evaluation metrics
rf_mae = mean_absolute_error(
    y_test_production_clean,
    production_rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_production_clean,
        production_rf_predictions
    )
)

rf_r2 = r2_score(
    y_test_production_clean,
    production_rf_predictions
)

print("\nModel Performance:")
print("-" * 50)

print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 4))

PRODUCTION RANDOM FOREST - MODEL EVALUATION

Model Performance:
--------------------------------------------------
MAE : 32.06
RMSE: 40.04
R²  : 0.71


In [ ]:
# ============================================================
# SAVE BEST PRODUCTION MODEL
# ============================================================

import joblib
import pandas as pd

print("SAVING BEST PRODUCTION MODEL")
print("=" * 70)

# Save the Random Forest model
model_path = "../data/processed/production_model_365_rf.pkl"

joblib.dump(
    production_rf_model,
    model_path
)

# Create prediction results
production_predictions_df = test_data[
    [
        "production_id",
        "machine_id",
        "production_date",
        "shift",
        "units_produced"
    ]
].copy()

production_predictions_df["predicted_units_produced"] = (
    production_rf_predictions
)

# Calculate prediction error
production_predictions_df["prediction_error"] = (
    production_predictions_df["units_produced"]
    - production_predictions_df["predicted_units_produced"]
)

# Save predictions
prediction_path = (
    "../data/processed/production_predictions_365_rf.csv"
)

production_predictions_df.to_csv(
    prediction_path,
    index=False
)

print("\nModel saved:")
print(model_path)

print("\nPredictions saved:")
print(prediction_path)

print("\nPrediction file shape:")
print(production_predictions_df.shape)

print("\nFirst 5 prediction results:")
display(production_predictions_df.head())

SAVING BEST PRODUCTION MODEL

Model saved:
../data/processed/production_model_365_rf.pkl

Predictions saved:
../data/processed/production_predictions_365_rf.csv

Prediction file shape:
(1095, 7)

First 5 prediction results:


,production_id,machine_id,production_date,shift,units_produced,predicted_units_produced,prediction_error
4380,4381,1,2026-06-12,Morning,912,945.155,-33.155
4381,4382,1,2026-06-12,Afternoon,931,940.715,-9.715
4382,4383,1,2026-06-12,Night,896,934.610,-38.610
4383,4384,2,2026-06-12,Morning,887,913.565,-26.565
4384,4385,2,2026-06-12,Afternoon,905,904.475,0.525


In [ ]:
# ============================================================
# PRODUCTION - FEATURE IMPORTANCE
# ============================================================

print("PRODUCTION - FEATURE IMPORTANCE")
print("=" * 70)

feature_importance = pd.DataFrame({
    "feature": production_safe_features,
    "importance": production_rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\nFeature importance ranking:")
display(feature_importance)

PRODUCTION - FEATURE IMPORTANCE

Feature importance ranking:


,feature,importance
0,target_quantity,0.694875
1,rolling_3_production_avg,0.048353
2,previous_rejection_rate,0.046953
3,rolling_3_rejection_avg,0.044294
4,machine_id,0.043544
5,previous_units_produced,0.039818
6,production_day,0.033284
7,production_month,0.021432
8,production_day_of_week,0.016187
9,shift_encoded,0.009110


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - TARGET CHECK
# ============================================================

quality_data = data["quality"].copy()

print("QUALITY / DEFECT PREDICTION")
print("=" * 70)

print("\nQuality status counts:")
print(quality_data["quality_status"].value_counts())

print("\nQuality status percentages:")
print(
    quality_data["quality_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTotal quality records:")
print(len(quality_data))

QUALITY / DEFECT PREDICTION

Quality status counts:
quality_status
Pass    4070
Fail    1405
Name: count, dtype: int64

Quality status percentages:
quality_status
Pass    74.34
Fail    25.66
Name: proportion, dtype: float64

Total quality records:
5475


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - FEATURE SETUP
# ============================================================

print("SETTING UP QUALITY ML DATASET")
print("=" * 70)

quality_target = "quality_status"

quality_features = [
    "machine_id",
    "target_quantity",
    "production_time_hours",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

print("\nQuality target:")
print(quality_target)

print("\nNumber of quality features:")
print(len(quality_features))

print("\nQuality features:")
print(quality_features)

SETTING UP QUALITY ML DATASET

Quality target:
quality_status

Number of quality features:
26

Quality features:
['machine_id', 'target_quantity', 'production_time_hours', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - CREATE ML DATASET
# ============================================================

X_quality = feature_data[quality_features].copy()
y_quality = quality_data[quality_target].copy()

print("QUALITY ML DATASET")
print("=" * 70)

print("\nX_quality shape:")
print(X_quality.shape)

print("\ny_quality shape:")
print(y_quality.shape)

print("\nMissing values in X:")
print(X_quality.isnull().sum().sum())

print("\nMissing values in y:")
print(y_quality.isnull().sum())

print("\nTarget distribution:")
print(y_quality.value_counts())

QUALITY ML DATASET

X_quality shape:
(5475, 26)

y_quality shape:
(5475,)

Missing values in X:
0

Missing values in y:
0

Target distribution:
quality_status
Pass    4070
Fail    1405
Name: count, dtype: int64


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - LEAKAGE CHECK
# ============================================================

print("QUALITY LEAKAGE CHECK")
print("=" * 70)

# Check whether the target is inside the feature list
print("\nTarget inside features:")
print(quality_target in quality_features)

# Check whether any quality-result columns are being used
quality_result_columns = [
    "units_rejected",
    "total_defects",
    "defect_type_count",
    "quality_defect_rate_percent",
    "rejection_rate_percent",
    "total_output"
]

used_result_columns = [
    column for column in quality_result_columns
    if column in quality_features
]

print("\nCurrent-result columns used as features:")
print(used_result_columns)

# Check dataset shapes
print("\nX shape:")
print(X_quality.shape)

print("\ny shape:")
print(y_quality.shape)

# Check missing values
print("\nMissing values in X:")
print(X_quality.isnull().sum().sum())

print("\nMissing values in y:")
print(y_quality.isnull().sum())

QUALITY LEAKAGE CHECK

Target inside features:
False

Current-result columns used as features:
[]

X shape:
(5475, 26)

y shape:
(5475,)

Missing values in X:
0

Missing values in y:
0


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - TRAIN TEST SPLIT
# ============================================================

print("QUALITY TRAIN / TEST SPLIT")
print("=" * 70)

# Sort quality data in chronological order
quality_ml_data = feature_data.copy()

quality_ml_data = quality_ml_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

# 80% training and 20% testing
quality_split_index = int(len(quality_ml_data) * 0.80)

quality_train_data = quality_ml_data.iloc[:quality_split_index].copy()
quality_test_data = quality_ml_data.iloc[quality_split_index:].copy()

# Create X and y
X_train_quality = quality_train_data[quality_features].copy()
X_test_quality = quality_test_data[quality_features].copy()

# Get quality status from the original quality data
quality_status_map = quality_data[
    ["production_id", "quality_status"]
].copy()

# Match the target using production_id
quality_train_data = quality_train_data.merge(
    quality_status_map,
    on="production_id",
    how="left"
)

quality_test_data = quality_test_data.merge(
    quality_status_map,
    on="production_id",
    how="left"
)

y_train_quality = quality_train_data["quality_status"].copy()
y_test_quality = quality_test_data["quality_status"].copy()

print("\nTraining shape:")
print(X_train_quality.shape)

print("\nTesting shape:")
print(X_test_quality.shape)

print("\nTraining dates:")
print(
    quality_train_data["production_date"].min(),
    "to",
    quality_train_data["production_date"].max()
)

print("\nTesting dates:")
print(
    quality_test_data["production_date"].min(),
    "to",
    quality_test_data["production_date"].max()
)

print("\nTraining target distribution:")
print(y_train_quality.value_counts())

print("\nTesting target distribution:")
print(y_test_quality.value_counts())

QUALITY TRAIN / TEST SPLIT

Training shape:
(4380, 26)

Testing shape:
(1095, 26)

Training dates:
2025-08-24 to 2026-06-11

Testing dates:
2026-06-12 to 2026-08-23

Training target distribution:
quality_status
Pass    3259
Fail    1121
Name: count, dtype: int64

Testing target distribution:
quality_status
Pass    811
Fail    284
Name: count, dtype: int64


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression

print("TRAINING QUALITY LOGISTIC REGRESSION")
print("=" * 70)

quality_lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

quality_lr_model.fit(
    X_train_quality,
    y_train_quality
)

quality_lr_predictions = quality_lr_model.predict(
    X_test_quality
)

print("\nModel training completed.")
print("Predictions generated:", len(quality_lr_predictions))

print("\nFirst 10 predictions:")
print(quality_lr_predictions[:10])

TRAINING QUALITY LOGISTIC REGRESSION

Model training completed.
Predictions generated: 1095

First 10 predictions:
['Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass']


e:\Projects\BEL\BEL_PROJECT\AI_Powered_Electronics_Manufacturing_Intelligence\data-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# ============================================================
# QUALITY - SCALED LOGISTIC REGRESSION
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("TRAINING SCALED QUALITY LOGISTIC REGRESSION")
print("=" * 70)

# Scale the features
quality_scaler = StandardScaler()

X_train_quality_scaled = quality_scaler.fit_transform(
    X_train_quality
)

X_test_quality_scaled = quality_scaler.transform(
    X_test_quality
)

# Train Logistic Regression
quality_lr_model_scaled = LogisticRegression(
    max_iter=2000,
    random_state=42
)

quality_lr_model_scaled.fit(
    X_train_quality_scaled,
    y_train_quality
)

# Generate predictions
quality_lr_predictions_scaled = (
    quality_lr_model_scaled.predict(
        X_test_quality_scaled
    )
)

print("\nModel training completed.")
print(
    "Predictions generated:",
    len(quality_lr_predictions_scaled)
)

print("\nFirst 10 predictions:")
print(quality_lr_predictions_scaled[:10])

TRAINING SCALED QUALITY LOGISTIC REGRESSION

Model training completed.
Predictions generated: 1095

First 10 predictions:
['Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass']


In [ ]:
# ============================================================
# QUALITY - LOGISTIC REGRESSION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("QUALITY LOGISTIC REGRESSION - EVALUATION")
print("=" * 70)

accuracy = accuracy_score(
    y_test_quality,
    quality_lr_predictions_scaled
)

precision = precision_score(
    y_test_quality,
    quality_lr_predictions_scaled,
    pos_label="Fail"
)

recall = recall_score(
    y_test_quality,
    quality_lr_predictions_scaled,
    pos_label="Fail"
)

f1 = f1_score(
    y_test_quality,
    quality_lr_predictions_scaled,
    pos_label="Fail"
)

print("\nAccuracy:", round(accuracy, 4))
print("Precision (Fail):", round(precision, 4))
print("Recall (Fail):", round(recall, 4))
print("F1-score (Fail):", round(f1, 4))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_quality,
        quality_lr_predictions_scaled,
        labels=["Pass", "Fail"]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_quality,
        quality_lr_predictions_scaled
    )
)

QUALITY LOGISTIC REGRESSION - EVALUATION

Accuracy: 0.7406
Precision (Fail): 0.0
Recall (Fail): 0.0
F1-score (Fail): 0.0

Confusion Matrix:
[[811   0]
 [284   0]]

Classification Report:
              precision    recall  f1-score   support

        Fail       0.00      0.00      0.00       284
        Pass       0.74      1.00      0.85       811

    accuracy                           0.74      1095
   macro avg       0.37      0.50      0.43      1095
weighted avg       0.55      0.74      0.63      1095



e:\Projects\BEL\BEL_PROJECT\AI_Powered_Electronics_Manufacturing_Intelligence\data-ml\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Projects\BEL\BEL_PROJECT\AI_Powered_Electronics_Manufacturing_Intelligence\data-ml\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Projects\BEL\BEL_PROJECT\AI_Powered_Electronics_Manufacturing_Intelligence\data-ml\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [ ]:
# ============================================================
# QUALITY - CHECK FEATURE DIFFERENCES
# ============================================================

print("QUALITY FEATURE CHECK")
print("=" * 70)

# Add the target temporarily for analysis only
quality_analysis = X_quality.copy()
quality_analysis["quality_status"] = y_quality.values

# Compare average feature values for Pass and Fail
quality_feature_means = (
    quality_analysis
    .groupby("quality_status")[quality_features]
    .mean()
    .T
)

quality_feature_means["difference"] = (
    quality_feature_means["Fail"]
    - quality_feature_means["Pass"]
)

quality_feature_means = quality_feature_means.sort_values(
    "difference",
    key=abs,
    ascending=False
)

print("\nFeature averages by quality status:")
display(quality_feature_means)

QUALITY FEATURE CHECK

Feature averages by quality status:


quality_status,Fail,Pass,difference
total_maintenance_cost,167648.255544,177878.480641,-10230.225097
target_quantity,939.358007,926.176904,13.181103
total_downtime_hours,184.705167,195.771975,-11.066808
rolling_3_production_avg,919.604114,909.563057,10.041057
previous_units_produced,918.735231,909.854300,8.880932
downtime_event_count,82.158719,87.595086,-5.436367
total_maintenance_downtime,71.526206,76.473607,-4.947400
maintenance_event_count,33.170819,35.093366,-1.922548
completed_maintenance_count,23.511744,24.975676,-1.463932
sensor_value_max,79.672093,80.516759,-0.844667


In [ ]:
# ============================================================
# QUALITY / DEFECT PREDICTION - RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

print("TRAINING QUALITY RANDOM FOREST")
print("=" * 70)

quality_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

quality_rf_model.fit(
    X_train_quality,
    y_train_quality
)

quality_rf_predictions = quality_rf_model.predict(
    X_test_quality
)

print("\nModel training completed.")
print(
    "Predictions generated:",
    len(quality_rf_predictions)
)

print("\nPrediction distribution:")
print(
    pd.Series(quality_rf_predictions)
    .value_counts()
)

print("\nFirst 10 predictions:")
print(quality_rf_predictions[:10])

TRAINING QUALITY RANDOM FOREST

Model training completed.
Predictions generated: 1095

Prediction distribution:
Pass    966
Fail    129
Name: count, dtype: int64

First 10 predictions:
['Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Fail' 'Pass' 'Pass' 'Pass' 'Pass']


In [ ]:
# ============================================================
# QUALITY - RANDOM FOREST EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("QUALITY RANDOM FOREST - EVALUATION")
print("=" * 70)

accuracy = accuracy_score(
    y_test_quality,
    quality_rf_predictions
)

precision = precision_score(
    y_test_quality,
    quality_rf_predictions,
    pos_label="Fail",
    zero_division=0
)

recall = recall_score(
    y_test_quality,
    quality_rf_predictions,
    pos_label="Fail",
    zero_division=0
)

f1 = f1_score(
    y_test_quality,
    quality_rf_predictions,
    pos_label="Fail",
    zero_division=0
)

print("\nAccuracy:", round(accuracy, 4))
print("Precision (Fail):", round(precision, 4))
print("Recall (Fail):", round(recall, 4))
print("F1-score (Fail):", round(f1, 4))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_quality,
        quality_rf_predictions,
        labels=["Pass", "Fail"]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_quality,
        quality_rf_predictions,
        zero_division=0
    )
)

QUALITY RANDOM FOREST - EVALUATION

Accuracy: 0.6868
Precision (Fail): 0.2713
Recall (Fail): 0.1232
F1-score (Fail): 0.1695

Confusion Matrix:
[[717  94]
 [249  35]]

Classification Report:
              precision    recall  f1-score   support

        Fail       0.27      0.12      0.17       284
        Pass       0.74      0.88      0.81       811

    accuracy                           0.69      1095
   macro avg       0.51      0.50      0.49      1095
weighted avg       0.62      0.69      0.64      1095



In [ ]:
# Create separate temperature and vibration features

sensor_data = data["sensors"].copy()

temperature_data = sensor_data[
    sensor_data["sensor_type"] == "Temperature"
].copy()

vibration_data = sensor_data[
    sensor_data["sensor_type"] == "Vibration"
].copy()

print("Temperature records:", len(temperature_data))
print("Vibration records:", len(vibration_data))

print("\nTemperature statistics:")
print(temperature_data["sensor_value"].describe())

print("\nVibration statistics:")
print(vibration_data["sensor_value"].describe())

Temperature records: 5475
Vibration records: 5475

Temperature statistics:
count    5475.000000
mean       70.400424
std         6.768219
min        54.640000
25%        65.700000
50%        68.570000
75%        72.990000
max        91.030000
Name: sensor_value, dtype: float64

Vibration statistics:
count    5475.000000
mean        3.194088
std         1.044645
min         0.500000
25%         2.480000
50%         2.910000
75%         3.550000
max         6.570000
Name: sensor_value, dtype: float64


In [ ]:
# Create separate temperature and vibration features

temperature_features = temperature_data[
    ["machine_id", "recorded_at", "sensor_value"]
].copy()

temperature_features = temperature_features.rename(
    columns={"sensor_value": "temperature"}
)

vibration_features = vibration_data[
    ["machine_id", "recorded_at", "sensor_value"]
].copy()

vibration_features = vibration_features.rename(
    columns={"sensor_value": "vibration"}
)

print("Temperature features:")
print(temperature_features.head())

print("\nVibration features:")
print(vibration_features.head())

Temperature features:
   machine_id         recorded_at  temperature
0           1 2025-08-24 08:00:00        66.49
2           1 2025-08-24 14:00:00        62.62
4           1 2025-08-24 22:00:00        65.77
6           2 2025-08-24 08:00:00        64.42
8           2 2025-08-24 14:00:00        69.73

Vibration features:
   machine_id         recorded_at  vibration
1           1 2025-08-24 08:00:00       3.13
3           1 2025-08-24 14:00:00       3.21
5           1 2025-08-24 22:00:00       2.95
7           2 2025-08-24 08:00:00       2.80
9           2 2025-08-24 14:00:00       3.55


In [ ]:
# Combine temperature and vibration readings

sensor_features = temperature_features.merge(
    vibration_features,
    on=["machine_id", "recorded_at"],
    how="inner"
)

print("Sensor feature shape:", sensor_features.shape)

print("\nSensor feature columns:")
print(sensor_features.columns.tolist())

print("\nFirst 10 rows:")
print(sensor_features.head(10))

Sensor feature shape: (5475, 4)

Sensor feature columns:
['machine_id', 'recorded_at', 'temperature', 'vibration']

First 10 rows:
   machine_id         recorded_at  temperature  vibration
0           1 2025-08-24 08:00:00        66.49       3.13
1           1 2025-08-24 14:00:00        62.62       3.21
2           1 2025-08-24 22:00:00        65.77       2.95
3           2 2025-08-24 08:00:00        64.42       2.80
4           2 2025-08-24 14:00:00        69.73       3.55
5           2 2025-08-24 22:00:00        63.38       3.29
6           3 2025-08-24 08:00:00        82.39       4.77
7           3 2025-08-24 14:00:00        84.28       4.88
8           3 2025-08-24 22:00:00        84.15       4.50
9           4 2025-08-24 08:00:00        63.10       2.97


In [ ]:
# Prepare sensor data for matching with production records

sensor_features["recorded_at"] = pd.to_datetime(
    sensor_features["recorded_at"]
)

sensor_features["production_date"] = (
    sensor_features["recorded_at"].dt.date
)

sensor_features["shift"] = sensor_features["recorded_at"].dt.hour.map({
    8: "Morning",
    14: "Afternoon",
    22: "Night"
})

print(sensor_features.head(10))

print("\nShift counts:")
print(sensor_features["shift"].value_counts())

print("\nMissing values:")
print(sensor_features[["machine_id", "production_date", "shift",
                       "temperature", "vibration"]].isnull().sum())

   machine_id         recorded_at  temperature  vibration production_date  \
0           1 2025-08-24 08:00:00        66.49       3.13      2025-08-24   
1           1 2025-08-24 14:00:00        62.62       3.21      2025-08-24   
2           1 2025-08-24 22:00:00        65.77       2.95      2025-08-24   
3           2 2025-08-24 08:00:00        64.42       2.80      2025-08-24   
4           2 2025-08-24 14:00:00        69.73       3.55      2025-08-24   
5           2 2025-08-24 22:00:00        63.38       3.29      2025-08-24   
6           3 2025-08-24 08:00:00        82.39       4.77      2025-08-24   
7           3 2025-08-24 14:00:00        84.28       4.88      2025-08-24   
8           3 2025-08-24 22:00:00        84.15       4.50      2025-08-24   
9           4 2025-08-24 08:00:00        63.10       2.97      2025-08-24   

       shift  
0    Morning  
1  Afternoon  
2      Night  
3    Morning  
4  Afternoon  
5      Night  
6    Morning  
7  Afternoon  
8      Night  
9 

In [ ]:
# Match sensor readings with production records

feature_data["production_date"] = pd.to_datetime(
    feature_data["production_date"]
).dt.date

sensor_features["production_date"] = pd.to_datetime(
    sensor_features["production_date"]
).dt.date

feature_data = feature_data.merge(
    sensor_features[
        ["machine_id", "production_date", "shift", "temperature", "vibration"]
    ],
    on=["machine_id", "production_date", "shift"],
    how="left"
)

print("Updated feature data shape:", feature_data.shape)

print("\nNew sensor columns:")
print(feature_data[["temperature", "vibration"]].head(10))

print("\nMissing sensor values:")
print(feature_data[["temperature", "vibration"]].isnull().sum())

NameError: name 'feature_data' is not defined

In [ ]:
# Reload the feature engineered dataset

feature_data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

feature_data["production_date"] = pd.to_datetime(
    feature_data["production_date"]
)

print("Feature data shape:", feature_data.shape)
print("Missing values:", feature_data.isnull().sum().sum())

print("\nFirst 5 rows:")
print(feature_data.head())

Feature data shape: (5475, 42)
Missing values: 0

First 5 rows:
   production_id  machine_id production_date      shift  units_produced  \
0              1           1      2025-08-24    Morning             969   
1              2           1      2025-08-24  Afternoon             944   
2              3           1      2025-08-24      Night             940   
3             16           1      2025-08-25    Morning             954   
4             17           1      2025-08-25  Afternoon             925   

   units_rejected  production_time_hours  target_id target_date  \
0              59                    8.0          1  2025-08-24   
1              48                    8.0          2  2025-08-24   
2              13                    8.0          3  2025-08-24   
3              20                    8.0         16  2025-08-25   
4              53                    8.0         17  2025-08-25   

   target_quantity  ...  completed_maintenance_count  production_day  \
0         

In [ ]:
# Match temperature and vibration with production records

sensor_features["production_date"] = pd.to_datetime(
    sensor_features["production_date"]
).dt.date

feature_data["production_date_key"] = feature_data["production_date"].dt.date

feature_data = feature_data.merge(
    sensor_features[
        ["machine_id", "production_date", "shift", "temperature", "vibration"]
    ],
    left_on=["machine_id", "production_date_key", "shift"],
    right_on=["machine_id", "production_date", "shift"],
    how="left"
)

feature_data = feature_data.drop(columns=["production_date_key"])

print("Updated feature data shape:", feature_data.shape)

print("\nSensor columns:")
print(feature_data[["temperature", "vibration"]].head(10))

print("\nMissing sensor values:")
print(feature_data[["temperature", "vibration"]].isnull().sum())

Updated feature data shape: (5475, 45)

Sensor columns:
   temperature  vibration
0        66.49       3.13
1        62.62       3.21
2        65.77       2.95
3        73.71       3.78
4        67.26       2.54
5        71.23       2.41
6        64.05       3.48
7        69.30       3.03
8        71.67       1.89
9        75.02       3.57

Missing sensor values:
temperature    0
vibration      0
dtype: int64


In [ ]:
# Compare temperature and vibration between Pass and Fail records

quality_status_map = data["quality"][
    ["production_id", "quality_status"]
].copy()

quality_sensor_analysis = feature_data[
    ["production_id", "temperature", "vibration"]
].merge(
    quality_status_map,
    on="production_id",
    how="left"
)

sensor_quality_means = (
    quality_sensor_analysis
    .groupby("quality_status")[["temperature", "vibration"]]
    .mean()
)

sensor_quality_means["difference"] = (
    sensor_quality_means.loc["Fail"]
    - sensor_quality_means.loc["Pass"]
)

print("Average sensor values by quality status:")
print(sensor_quality_means)

Average sensor values by quality status:
                temperature  vibration  difference
quality_status                                    
Fail              69.682527   3.087964         NaN
Pass              70.648248   3.230722         NaN


In [ ]:
# Calculate sensor differences correctly

fail_temperature = sensor_quality_means.loc["Fail", "temperature"]
pass_temperature = sensor_quality_means.loc["Pass", "temperature"]

fail_vibration = sensor_quality_means.loc["Fail", "vibration"]
pass_vibration = sensor_quality_means.loc["Pass", "vibration"]

print("Temperature difference (Fail - Pass):",
      round(fail_temperature - pass_temperature, 3))

print("Vibration difference (Fail - Pass):",
      round(fail_vibration - pass_vibration, 3))

print("\nSensor averages by machine:")

machine_sensor_means = (
    feature_data
    .groupby("machine_id")[["temperature", "vibration"]]
    .mean()
)

print(machine_sensor_means)

Temperature difference (Fail - Pass): -0.966
Vibration difference (Fail - Pass): -0.143

Sensor averages by machine:
            temperature  vibration
machine_id                        
1             70.153726   3.001361
2             67.907342   2.802393
3             82.032420   4.998219
4             64.878685   2.461041
5             67.029945   2.707425


In [ ]:
# Add separate temperature and vibration features to the quality model

quality_features_with_sensors = quality_features + [
    "temperature",
    "vibration"
]

X_quality_sensor = feature_data[
    quality_features_with_sensors
].copy()

print("Quality features with sensors:", X_quality_sensor.shape)

print("\nNew features:")
print(quality_features_with_sensors)

print("\nMissing values:")
print(X_quality_sensor.isnull().sum().sum())

NameError: name 'quality_features' is not defined

In [ ]:
# Recreate the quality model feature list

quality_features = [
    "machine_id",
    "target_quantity",
    "production_time_hours",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

print("Number of quality features:", len(quality_features))
print(quality_features)

Number of quality features: 26
['machine_id', 'target_quantity', 'production_time_hours', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# Add separate temperature and vibration features

quality_features_with_sensors = quality_features + [
    "temperature",
    "vibration"
]

X_quality_sensor = feature_data[
    quality_features_with_sensors
].copy()

print("Quality features with sensors:", X_quality_sensor.shape)

print("\nNumber of features:", len(quality_features_with_sensors))

print("\nMissing values:",
      X_quality_sensor.isnull().sum().sum())

Quality features with sensors: (5475, 28)

Number of features: 28

Missing values: 0


In [ ]:
# Create train and test data for the sensor-enhanced quality model

quality_sensor_ml_data = feature_data.copy()

quality_sensor_ml_data = quality_sensor_ml_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

quality_sensor_split_index = int(len(quality_sensor_ml_data) * 0.80)

quality_sensor_train_data = quality_sensor_ml_data.iloc[
    :quality_sensor_split_index
].copy()

quality_sensor_test_data = quality_sensor_ml_data.iloc[
    quality_sensor_split_index:
].copy()

# Add quality status to train and test data

quality_status_map = data["quality"][
    ["production_id", "quality_status"]
].copy()

quality_sensor_train_data = quality_sensor_train_data.merge(
    quality_status_map,
    on="production_id",
    how="left"
)

quality_sensor_test_data = quality_sensor_test_data.merge(
    quality_status_map,
    on="production_id",
    how="left"
)

X_train_quality_sensor = quality_sensor_train_data[
    quality_features_with_sensors
].copy()

X_test_quality_sensor = quality_sensor_test_data[
    quality_features_with_sensors
].copy()

y_train_quality_sensor = quality_sensor_train_data[
    "quality_status"
].copy()

y_test_quality_sensor = quality_sensor_test_data[
    "quality_status"
].copy()

print("Training shape:", X_train_quality_sensor.shape)
print("Testing shape:", X_test_quality_sensor.shape)

print("\nTraining target:")
print(y_train_quality_sensor.value_counts())

print("\nTesting target:")
print(y_test_quality_sensor.value_counts())

KeyError: 'production_date'

In [ ]:
# Check the current feature data columns

print("Number of columns:", len(feature_data.columns))

print("\nColumn names:")
print(feature_data.columns.tolist())

Number of columns: 45

Column names:
['production_id', 'machine_id', 'production_date_x', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg', 'production_date_y', 'temperature'

In [ ]:
# Fix the production date column after sensor merge

feature_data["production_date"] = feature_data["production_date_x"]

# Convert it to proper datetime format
feature_data["production_date"] = pd.to_datetime(
    feature_data["production_date"]
)

print("Production date column fixed.")
print("Missing production dates:", feature_data["production_date"].isna().sum())
print("Date range:")
print(feature_data["production_date"].min(), "to", feature_data["production_date"].max())

Production date column fixed.
Missing production dates: 0
Date range:
2025-08-24 00:00:00 to 2026-08-23 00:00:00


In [ ]:
# Sort the data in time order

quality_sensor_ml_data = feature_data.copy()

quality_sensor_ml_data = quality_sensor_ml_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

# Find the date where training should end
unique_dates = quality_sensor_ml_data["production_date"].drop_duplicates().sort_values()

split_index = int(len(unique_dates) * 0.80)
split_date = unique_dates.iloc[split_index]

# Create train and test data
train_quality_sensor = quality_sensor_ml_data[
    quality_sensor_ml_data["production_date"] < split_date
].copy()

test_quality_sensor = quality_sensor_ml_data[
    quality_sensor_ml_data["production_date"] >= split_date
].copy()

print("Training data shape:", train_quality_sensor.shape)
print("Testing data shape:", test_quality_sensor.shape)

print("\nTraining date range:")
print(train_quality_sensor["production_date"].min(), "to", train_quality_sensor["production_date"].max())

print("\nTesting date range:")
print(test_quality_sensor["production_date"].min(), "to", test_quality_sensor["production_date"].max())

print("\nSplit date:", split_date)

Training data shape: (4380, 46)
Testing data shape: (1095, 46)

Training date range:
2025-08-24 00:00:00 to 2026-06-11 00:00:00

Testing date range:
2026-06-12 00:00:00 to 2026-08-23 00:00:00

Split date: 2026-06-12 00:00:00


In [ ]:
# Create features (X) and target (y) for the quality model

quality_target = "quality_defect_rate_percent"

X_train_quality_sensor = train_quality_sensor[quality_features_with_sensors].copy()
y_train_quality_sensor = train_quality_sensor[quality_target].copy()

X_test_quality_sensor = test_quality_sensor[quality_features_with_sensors].copy()
y_test_quality_sensor = test_quality_sensor[quality_target].copy()

print("X_train shape:", X_train_quality_sensor.shape)
print("y_train shape:", y_train_quality_sensor.shape)

print("X_test shape:", X_test_quality_sensor.shape)
print("y_test shape:", y_test_quality_sensor.shape)

print("\nFeatures used:")
print(quality_features_with_sensors)

print("\nMissing values:")
print("X_train:", X_train_quality_sensor.isna().sum().sum())
print("X_test:", X_test_quality_sensor.isna().sum().sum())

X_train shape: (4380, 28)
y_train shape: (4380,)
X_test shape: (1095, 28)
y_test shape: (1095,)

Features used:
['machine_id', 'target_quantity', 'production_time_hours', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg', 'temperature', 'vibration']

Missing values:
X_train: 0
X_test: 0


In [ ]:
# Train a Linear Regression model for quality prediction

from sklearn.linear_model import LinearRegression

quality_linear_model = LinearRegression()

quality_linear_model.fit(
    X_train_quality_sensor,
    y_train_quality_sensor
)

# Make predictions on the test data
quality_linear_predictions = quality_linear_model.predict(
    X_test_quality_sensor
)

print("Linear Regression model trained successfully.")
print("Number of predictions:", len(quality_linear_predictions))

Linear Regression model trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate the Linear Regression model

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

quality_linear_mae = mean_absolute_error(
    y_test_quality_sensor,
    quality_linear_predictions
)

quality_linear_rmse = np.sqrt(
    mean_squared_error(
        y_test_quality_sensor,
        quality_linear_predictions
    )
)

quality_linear_r2 = r2_score(
    y_test_quality_sensor,
    quality_linear_predictions
)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", round(quality_linear_mae, 4))
print("RMSE:", round(quality_linear_rmse, 4))
print("R²  :", round(quality_linear_r2, 4))

Linear Regression Results
-------------------------
MAE : 0.5093
RMSE: 0.6217
R²  : -0.0072


In [ ]:
# Train Random Forest for quality prediction

from sklearn.ensemble import RandomForestRegressor

quality_rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=12,
    min_samples_split=5,
    n_jobs=-1
)

quality_rf_model.fit(
    X_train_quality_sensor,
    y_train_quality_sensor
)

# Make predictions on the test data
quality_rf_predictions = quality_rf_model.predict(
    X_test_quality_sensor
)

print("Random Forest model trained successfully.")
print("Number of predictions:", len(quality_rf_predictions))

Random Forest model trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate the Random Forest model

quality_rf_mae = mean_absolute_error(
    y_test_quality_sensor,
    quality_rf_predictions
)

quality_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_quality_sensor,
        quality_rf_predictions
    )
)

quality_rf_r2 = r2_score(
    y_test_quality_sensor,
    quality_rf_predictions
)

print("Random Forest Results")
print("---------------------")
print("MAE :", round(quality_rf_mae, 4))
print("RMSE:", round(quality_rf_rmse, 4))
print("R²  :", round(quality_rf_r2, 4))

Random Forest Results
---------------------
MAE : 0.5092
RMSE: 0.623
R²  : -0.0115


In [ ]:
# Check the quality target distribution

print("Training target statistics:")
print(y_train_quality_sensor.describe())

print("\nTesting target statistics:")
print(y_test_quality_sensor.describe())

print("\nUnique target values:")
print("Train:", y_train_quality_sensor.nunique())
print("Test :", y_test_quality_sensor.nunique())

Training target statistics:
count    4380.000000
mean        1.094612
std         0.613758
min         0.100000
25%         0.600000
50%         1.010000
75%         1.510000
max         2.860000
Name: quality_defect_rate_percent, dtype: float64

Testing target statistics:
count    1095.000000
mean        1.109288
std         0.619760
min         0.100000
25%         0.620000
50%         1.020000
75%         1.515000
max         2.800000
Name: quality_defect_rate_percent, dtype: float64

Unique target values:
Train: 265
Test : 239


In [ ]:
# Check feature correlations with quality defect rate

correlation_data = quality_sensor_ml_data[
    quality_features_with_sensors + ["quality_defect_rate_percent"]
].corr(numeric_only=True)

quality_correlations = (
    correlation_data["quality_defect_rate_percent"]
    .drop("quality_defect_rate_percent")
    .sort_values(key=abs, ascending=False)
)

print("Feature correlations with quality defect rate:")
print(quality_correlations)

Feature correlations with quality defect rate:
rolling_3_rejection_avg        -0.019747
production_month                0.018972
sensor_value_max               -0.018482
completed_maintenance_count    -0.018012
production_day                  0.017918
maintenance_event_count        -0.017580
sensor_value_min               -0.017440
sensor_value_std               -0.016924
sensor_value_mean              -0.016828
target_quantity                 0.016175
temperature                    -0.016099
total_maintenance_downtime     -0.014017
average_downtime_hours          0.013520
total_maintenance_cost         -0.012679
downtime_event_count           -0.012497
production_day_of_week         -0.011512
utilization_percent             0.011301
total_downtime_hours           -0.011287
previous_rejection_rate        -0.011202
is_weekend                     -0.010182
vibration                      -0.009581
rolling_3_production_avg        0.007110
machine_id                     -0.006364
shift_enco

In [ ]:
# Check which features are related to rejected units

rejection_correlations = (
    quality_sensor_ml_data[
        quality_features_with_sensors + ["units_rejected"]
    ]
    .corr(numeric_only=True)["units_rejected"]
    .drop("units_rejected")
    .sort_values(key=abs, ascending=False)
)

print("Feature correlations with units rejected:")
print(rejection_correlations)

Feature correlations with units rejected:
target_quantity                 0.155169
downtime_event_count           -0.154911
total_maintenance_downtime     -0.154859
maintenance_event_count        -0.154181
utilization_percent             0.151465
total_downtime_hours           -0.151403
sensor_value_min               -0.151155
sensor_value_mean              -0.147386
total_maintenance_cost         -0.146798
completed_maintenance_count    -0.146678
sensor_value_std               -0.146640
sensor_value_max               -0.143390
rolling_3_production_avg        0.141935
temperature                    -0.137476
previous_units_produced         0.126714
vibration                      -0.123987
average_maintenance_downtime   -0.107903
average_downtime_hours          0.086667
production_month                0.017256
rolling_3_rejection_avg        -0.014865
production_day                  0.012485
previous_rejection_rate        -0.011081
production_day_of_week         -0.006432
machine_id     

In [ ]:
# Create X and y for predicting rejected units

quality_rejection_target = "units_rejected"

X_train_rejection = train_quality_sensor[
    quality_features_with_sensors
].copy()

y_train_rejection = train_quality_sensor[
    quality_rejection_target
].copy()

X_test_rejection = test_quality_sensor[
    quality_features_with_sensors
].copy()

y_test_rejection = test_quality_sensor[
    quality_rejection_target
].copy()

print("X_train:", X_train_rejection.shape)
print("y_train:", y_train_rejection.shape)
print("X_test :", X_test_rejection.shape)
print("y_test :", y_test_rejection.shape)

print("\nMissing values:")
print("X_train:", X_train_rejection.isna().sum().sum())
print("X_test :", X_test_rejection.isna().sum().sum())

X_train: (4380, 28)
y_train: (4380,)
X_test : (1095, 28)
y_test : (1095,)

Missing values:
X_train: 0
X_test : 0


In [ ]:
# Train the final Random Forest quality model

from sklearn.ensemble import RandomForestRegressor

final_quality_rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

final_quality_rf_model.fit(
    X_train_rejection,
    y_train_rejection
)

# Predict rejected units for the test data
final_quality_predictions = final_quality_rf_model.predict(
    X_test_rejection
)

print("Final Quality Random Forest trained successfully.")
print("Number of predictions:", len(final_quality_predictions))

Final Quality Random Forest trained successfully.
Number of predictions: 1095


In [ ]:
# Train the final Random Forest quality model

from sklearn.ensemble import RandomForestRegressor

final_quality_rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

final_quality_rf_model.fit(
    X_train_rejection,
    y_train_rejection
)

# Predict rejected units for the test data
final_quality_predictions = final_quality_rf_model.predict(
    X_test_rejection
)

print("Final Quality Random Forest trained successfully.")
print("Number of predictions:", len(final_quality_predictions))

Final Quality Random Forest trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate the final quality model

final_quality_mae = mean_absolute_error(
    y_test_rejection,
    final_quality_predictions
)

final_quality_rmse = np.sqrt(
    mean_squared_error(
        y_test_rejection,
        final_quality_predictions
    )
)

final_quality_r2 = r2_score(
    y_test_rejection,
    final_quality_predictions
)

print("Final Quality Random Forest Results")
print("-----------------------------------")
print("MAE :", round(final_quality_mae, 4))
print("RMSE:", round(final_quality_rmse, 4))
print("R²  :", round(final_quality_r2, 4))

Final Quality Random Forest Results
-----------------------------------
MAE : 16.0375
RMSE: 18.7429
R²  : 0.0112


In [ ]:
# Check which features are most important for quality prediction

feature_importance = pd.DataFrame({
    "feature": X_train_rejection.columns,
    "importance": final_quality_rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Top quality prediction features:")
print(feature_importance.to_string(index=False))

Top quality prediction features:
                     feature  importance
             target_quantity    0.133549
                 temperature    0.116889
     previous_rejection_rate    0.116414
    rolling_3_production_avg    0.109624
     rolling_3_rejection_avg    0.108345
                   vibration    0.108085
     previous_units_produced    0.104898
              production_day    0.074576
            production_month    0.051467
      production_day_of_week    0.035153
               shift_encoded    0.022503
                  is_weekend    0.003642
      average_downtime_hours    0.002587
                  machine_id    0.002299
            sensor_value_max    0.001936
            sensor_value_std    0.000888
        total_downtime_hours    0.000880
average_maintenance_downtime    0.000856
 completed_maintenance_count    0.000832
  total_maintenance_downtime    0.000737
            sensor_value_min    0.000720
           sensor_value_mean    0.000717
        downtime_event_c

In [ ]:
# Save the final quality model and its predictions

import joblib
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")
MODEL_PATH = Path("../models")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Save the trained model
joblib.dump(
    final_quality_rf_model,
    MODEL_PATH / "quality_rejection_model.pkl"
)

# Create prediction output
quality_predictions_output = test_quality_sensor[
    ["production_id", "machine_id", "production_date", "shift"]
].copy()

quality_predictions_output["actual_units_rejected"] = y_test_rejection.values
quality_predictions_output["predicted_units_rejected"] = final_quality_predictions

quality_predictions_output["prediction_error"] = (
    quality_predictions_output["actual_units_rejected"]
    - quality_predictions_output["predicted_units_rejected"]
)

# Save predictions
quality_predictions_output.to_csv(
    PROCESSED_PATH / "quality_rejection_predictions.csv",
    index=False
)

print("Quality model saved:")
print(MODEL_PATH / "quality_rejection_model.pkl")

print("\nPrediction file saved:")
print(PROCESSED_PATH / "quality_rejection_predictions.csv")

print("\nPrediction rows:", len(quality_predictions_output))

Quality model saved:
..\models\quality_rejection_model.pkl

Prediction file saved:
..\data\processed\quality_rejection_predictions.csv

Prediction rows: 1095


In [ ]:
# Create X and y for production prediction

production_target = "units_produced"

X_train_production = train_quality_sensor[
    quality_features_with_sensors
].copy()

y_train_production = train_quality_sensor[
    production_target
].copy()

X_test_production = test_quality_sensor[
    quality_features_with_sensors
].copy()

y_test_production = test_quality_sensor[
    production_target
].copy()

print("X_train:", X_train_production.shape)
print("y_train:", y_train_production.shape)
print("X_test :", X_test_production.shape)
print("y_test :", y_test_production.shape)

print("\nMissing values:")
print("X_train:", X_train_production.isna().sum().sum())
print("X_test :", X_test_production.isna().sum().sum())

X_train: (4380, 28)
y_train: (4380,)
X_test : (1095, 28)
y_test : (1095,)

Missing values:
X_train: 0
X_test : 0


In [ ]:
# Train Linear Regression for production prediction

from sklearn.linear_model import LinearRegression

production_linear_model = LinearRegression()

production_linear_model.fit(
    X_train_production,
    y_train_production
)

production_linear_predictions = production_linear_model.predict(
    X_test_production
)

print("Production Linear Regression trained successfully.")
print("Number of predictions:", len(production_linear_predictions))

Production Linear Regression trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate Production Linear Regression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

production_lr_mae = mean_absolute_error(
    y_test_production,
    production_linear_predictions
)

production_lr_rmse = np.sqrt(
    mean_squared_error(
        y_test_production,
        production_linear_predictions
    )
)

production_lr_r2 = r2_score(
    y_test_production,
    production_linear_predictions
)

print("Production Linear Regression Results")
print("-----------------------------------")
print("MAE :", round(production_lr_mae, 4))
print("RMSE:", round(production_lr_rmse, 4))
print("R²  :", round(production_lr_r2, 4))

Production Linear Regression Results
-----------------------------------
MAE : 31.1659
RMSE: 39.0438
R²  : 0.7242


In [ ]:
# Train Random Forest for production prediction

from sklearn.ensemble import RandomForestRegressor

production_rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

production_rf_model.fit(
    X_train_production,
    y_train_production
)

production_rf_predictions = production_rf_model.predict(
    X_test_production
)

print("Production Random Forest trained successfully.")
print("Number of predictions:", len(production_rf_predictions))

Production Random Forest trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate Production Random Forest

production_rf_mae = mean_absolute_error(
    y_test_production,
    production_rf_predictions
)

production_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_production,
        production_rf_predictions
    )
)

production_rf_r2 = r2_score(
    y_test_production,
    production_rf_predictions
)

print("Production Random Forest Results")
print("--------------------------------")
print("MAE :", round(production_rf_mae, 4))
print("RMSE:", round(production_rf_rmse, 4))
print("R²  :", round(production_rf_r2, 4))

Production Random Forest Results
--------------------------------
MAE : 31.6915
RMSE: 39.5965
R²  : 0.7163


In [ ]:
# Save the final production prediction model

import joblib
from pathlib import Path

MODEL_PATH = Path("../models/production_model.pkl")

joblib.dump(
    production_linear_model,
    MODEL_PATH
)

print("Production model saved successfully.")
print("Model path:", MODEL_PATH)

Production model saved successfully.
Model path: ..\models\production_model.pkl


In [ ]:
# Save production prediction results

production_predictions_df = test_quality_sensor[
    ["production_id", "machine_id", "production_date", "shift", "units_produced"]
].copy()

production_predictions_df["predicted_units_produced"] = production_linear_predictions

production_predictions_df.to_csv(
    "../data/processed/production_predictions.csv",
    index=False
)

print("Production predictions saved successfully.")
print("Rows saved:", len(production_predictions_df))
print("\nPreview:")
print(production_predictions_df.head())

Production predictions saved successfully.
Rows saved: 1095

Preview:
      production_id  machine_id production_date      shift  units_produced  \
4380           4381           1      2026-06-12    Morning             912   
4381           4382           1      2026-06-12  Afternoon             931   
4382           4383           1      2026-06-12      Night             896   
4383           4384           2      2026-06-12    Morning             887   
4384           4385           2      2026-06-12  Afternoon             905   

      predicted_units_produced  
4380                951.986692  
4381                953.344187  
4382                949.577730  
4383                901.811969  
4384                900.846464  


In [ ]:
# Create failure risk target

failure_data = feature_data.copy()

failure_data["production_date"] = pd.to_datetime(
    failure_data["production_date"]
)

print("Failure dataset shape:", failure_data.shape)
print("Date range:")
print(failure_data["production_date"].min(), "to", failure_data["production_date"].max())

Failure dataset shape: (5475, 46)
Date range:
2025-08-24 00:00:00 to 2026-08-23 00:00:00


In [ ]:
# Create failure risk target from downtime events

downtime_data = pd.read_csv("../data/generated/downtime.csv")

downtime_data["downtime_start"] = pd.to_datetime(
    downtime_data["downtime_start"]
)

downtime_data["downtime_date"] = downtime_data["downtime_start"].dt.date

failure_data["failure_risk"] = (
    failure_data.apply(
        lambda row: int(
            (
                (downtime_data["machine_id"] == row["machine_id"]) &
                (downtime_data["downtime_date"] == row["production_date"].date())
            ).any()
        ),
        axis=1
    )
)

print("Failure risk target created.")
print("\nFailure risk distribution:")
print(failure_data["failure_risk"].value_counts())

print("\nFailure risk percentage:")
print(
    failure_data["failure_risk"].value_counts(normalize=True).mul(100).round(2)
)

Failure risk target created.

Failure risk distribution:
failure_risk
0    4182
1    1293
Name: count, dtype: int64

Failure risk percentage:
failure_risk
0    76.38
1    23.62
Name: proportion, dtype: float64


In [ ]:
# Prepare features for failure risk prediction

failure_features = [
    "machine_id",
    "units_produced",
    "units_rejected",
    "target_quantity",
    "production_time_hours",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count",
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg",
    "temperature",
    "vibration"
]

X_failure = failure_data[failure_features].copy()
y_failure = failure_data["failure_risk"].copy()

print("Failure features:", X_failure.shape)
print("Failure target:", y_failure.shape)
print("Missing values:", X_failure.isna().sum().sum())

Failure features: (5475, 30)
Failure target: (5475,)
Missing values: 0


In [ ]:
# Time-based train/test split for failure risk

failure_data = failure_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

unique_failure_dates = (
    failure_data["production_date"]
    .drop_duplicates()
    .sort_values()
)

failure_split_index = int(len(unique_failure_dates) * 0.80)

failure_split_date = unique_failure_dates.iloc[failure_split_index]

train_failure = failure_data[
    failure_data["production_date"] < failure_split_date
].copy()

test_failure = failure_data[
    failure_data["production_date"] >= failure_split_date
].copy()

X_train_failure = train_failure[failure_features].copy()
y_train_failure = train_failure["failure_risk"].copy()

X_test_failure = test_failure[failure_features].copy()
y_test_failure = test_failure["failure_risk"].copy()

print("Failure Risk Train:", X_train_failure.shape)
print("Failure Risk Test :", X_test_failure.shape)

print("\nTraining dates:")
print(train_failure["production_date"].min(), "to", train_failure["production_date"].max())

print("\nTesting dates:")
print(test_failure["production_date"].min(), "to", test_failure["production_date"].max())

print("\nTarget distribution:")
print("Train:")
print(y_train_failure.value_counts())
print("\nTest:")
print(y_test_failure.value_counts())

Failure Risk Train: (4380, 30)
Failure Risk Test : (1095, 30)

Training dates:
2025-08-24 00:00:00 to 2026-06-11 00:00:00

Testing dates:
2026-06-12 00:00:00 to 2026-08-23 00:00:00

Target distribution:
Train:
failure_risk
0    3363
1    1017
Name: count, dtype: int64

Test:
failure_risk
0    819
1    276
Name: count, dtype: int64


In [ ]:
# Train Random Forest for failure risk prediction

from sklearn.ensemble import RandomForestClassifier

failure_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

failure_rf_model.fit(
    X_train_failure,
    y_train_failure
)

failure_predictions = failure_rf_model.predict(
    X_test_failure
)

failure_probabilities = failure_rf_model.predict_proba(
    X_test_failure
)[:, 1]

print("Failure Risk Random Forest trained successfully.")
print("Number of predictions:", len(failure_predictions))

Failure Risk Random Forest trained successfully.
Number of predictions: 1095


In [ ]:
# Evaluate Failure Risk model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

failure_accuracy = accuracy_score(
    y_test_failure,
    failure_predictions
)

failure_precision = precision_score(
    y_test_failure,
    failure_predictions
)

failure_recall = recall_score(
    y_test_failure,
    failure_predictions
)

failure_f1 = f1_score(
    y_test_failure,
    failure_predictions
)

failure_cm = confusion_matrix(
    y_test_failure,
    failure_predictions
)

print("Failure Risk Model Results")
print("--------------------------")
print("Accuracy :", round(failure_accuracy, 4))
print("Precision:", round(failure_precision, 4))
print("Recall   :", round(failure_recall, 4))
print("F1 Score :", round(failure_f1, 4))

print("\nConfusion Matrix:")
print(failure_cm)

Failure Risk Model Results
--------------------------
Accuracy : 0.7105
Precision: 0.4105
Recall   : 0.3406
F1 Score : 0.3723

Confusion Matrix:
[[684 135]
 [182  94]]


In [ ]:
# Check the most important features for failure risk

failure_feature_importance = pd.DataFrame({
    "feature": failure_features,
    "importance": failure_rf_model.feature_importances_
})

failure_feature_importance = failure_feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Top Failure Risk Features:")
print(failure_feature_importance.head(10))

Top Failure Risk Features:
                    feature  importance
0            production_day    0.088942
1  rolling_3_production_avg    0.088847
2               temperature    0.087430
3           target_quantity    0.085085
4   rolling_3_rejection_avg    0.082246
5                 vibration    0.075100
6   previous_rejection_rate    0.074837
7   previous_units_produced    0.074239
8            units_produced    0.072906
9            units_rejected    0.069911


In [ ]:
# Save the final failure risk model and predictions

import joblib

# Save model
joblib.dump(
    failure_rf_model,
    "../models/failure_risk_model.pkl"
)

# Create prediction results
failure_risk_predictions_df = test_failure[
    [
        "production_id",
        "machine_id",
        "production_date",
        "shift"
    ]
].copy()

failure_risk_predictions_df["actual_failure_risk"] = y_test_failure.values
failure_risk_predictions_df["predicted_failure_risk"] = failure_predictions
failure_risk_predictions_df["failure_probability"] = failure_probabilities

failure_risk_predictions_df.to_csv(
    "../data/processed/failure_risk_predictions.csv",
    index=False
)

print("Failure Risk model saved successfully.")
print("Failure Risk predictions saved successfully.")
print("Rows saved:", len(failure_risk_predictions_df))

print("\nPreview:")
print(failure_risk_predictions_df.head())

Failure Risk model saved successfully.
Failure Risk predictions saved successfully.
Rows saved: 1095

Preview:
      production_id  machine_id production_date      shift  \
4380           4381           1      2026-06-12    Morning   
4381           4382           1      2026-06-12  Afternoon   
4382           4383           1      2026-06-12      Night   
4383           4384           2      2026-06-12    Morning   
4384           4385           2      2026-06-12  Afternoon   

      actual_failure_risk  predicted_failure_risk  failure_probability  
4380                    1                       0             0.451319  
4381                    1                       1             0.514631  
4382                    1                       0             0.436672  
4383                    0                       0             0.496897  
4384                    0                       0             0.423872  


In [ ]:
# Prepare data for anomaly detection

anomaly_features = [
    "machine_id",
    "units_produced",
    "units_rejected",
    "target_quantity",
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "maintenance_event_count",
    "total_maintenance_downtime",
    "total_maintenance_cost",
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg",
    "temperature",
    "vibration"
]

X_anomaly = failure_data[anomaly_features].copy()

print("Anomaly dataset shape:", X_anomaly.shape)
print("Missing values:", X_anomaly.isna().sum().sum())

Anomaly dataset shape: (5475, 22)
Missing values: 0


In [ ]:
# Train Isolation Forest for anomaly detection

from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

anomaly_model.fit(X_anomaly)

anomaly_predictions = anomaly_model.predict(X_anomaly)
anomaly_scores = anomaly_model.decision_function(X_anomaly)

print("Anomaly Detection model trained successfully.")
print("Total records:", len(anomaly_predictions))

Anomaly Detection model trained successfully.
Total records: 5475


In [ ]:
# Check anomaly detection results

anomaly_count = (anomaly_predictions == -1).sum()
normal_count = (anomaly_predictions == 1).sum()

print("Anomaly Detection Results")
print("-------------------------")
print("Normal records :", normal_count)
print("Anomalous records:", anomaly_count)

print("\nAnomaly percentage:")
print(round((anomaly_count / len(anomaly_predictions)) * 100, 2), "%")

Anomaly Detection Results
-------------------------
Normal records : 5201
Anomalous records: 274

Anomaly percentage:
5.0 %


In [ ]:
# Save anomaly detection model and results

import joblib

# Save model
joblib.dump(
    anomaly_model,
    "../models/anomaly_detection_model.pkl"
)

# Create anomaly results
anomaly_results = failure_data[
    [
        "production_id",
        "machine_id",
        "production_date",
        "shift"
    ]
].copy()

anomaly_results["anomaly"] = (
    anomaly_predictions == -1
).astype(int)

anomaly_results["anomaly_score"] = anomaly_scores

anomaly_results.to_csv(
    "../data/processed/anomaly_detection_results.csv",
    index=False
)

print("Anomaly Detection model saved successfully.")
print("Anomaly results saved successfully.")
print("Rows saved:", len(anomaly_results))

print("\nPreview:")
print(anomaly_results.head())

Anomaly Detection model saved successfully.
Anomaly results saved successfully.
Rows saved: 5475

Preview:
   production_id  machine_id production_date      shift  anomaly  \
0              1           1      2025-08-24    Morning        0   
1              2           1      2025-08-24  Afternoon        0   
2              3           1      2025-08-24      Night        0   
3              4           2      2025-08-24    Morning        0   
4              5           2      2025-08-24  Afternoon        0   

   anomaly_score  
0       0.084446  
1       0.069774  
2       0.092802  
3       0.052790  
4       0.035519  


In [ ]:
# Final check of all ML models and prediction files

from pathlib import Path

models_path = Path("../models")
processed_path = Path("../data/processed")

print("ML MODELS")
print("---------")

model_files = [
    "production_model.pkl",
    "quality_rejection_model.pkl",
    "failure_risk_model.pkl",
    "anomaly_detection_model.pkl"
]

for file_name in model_files:
    file_path = models_path / file_name
    print(file_name, "->", "FOUND" if file_path.exists() else "MISSING")


print("\nML PREDICTION FILES")
print("-------------------")

prediction_files = [
    "production_predictions.csv",
    "quality_rejection_predictions.csv",
    "failure_risk_predictions.csv",
    "anomaly_detection_results.csv"
]

for file_name in prediction_files:
    file_path = processed_path / file_name
    print(file_name, "->", "FOUND" if file_path.exists() else "MISSING")

ML MODELS
---------
production_model.pkl -> FOUND
quality_rejection_model.pkl -> FOUND
failure_risk_model.pkl -> FOUND
anomaly_detection_model.pkl -> FOUND

ML PREDICTION FILES
-------------------
production_predictions.csv -> FOUND
quality_rejection_predictions.csv -> FOUND
failure_risk_predictions.csv -> FOUND
anomaly_detection_results.csv -> FOUND


In [ ]:
# Full dataset count and date validation

from pathlib import Path
import pandas as pd

data_path = Path("../data/generated")

csv_files = [
    "production.csv",
    "production_targets.csv",
    "quality.csv",
    "sensors.csv",
    "downtime.csv",
    "maintenance.csv",
    "production_logs.csv",
    "shifts.csv",
    "employees.csv",
    "machines.csv",
    "inventory.csv",
    "suppliers.csv"
]

print("FULL DATASET VALIDATION")
print("=======================")

for file_name in csv_files:
    file_path = data_path / file_name
    df = pd.read_csv(file_path)
    print(f"{file_name:<28} {len(df):>6} rows")

print("\nPRODUCTION DATE VALIDATION")
print("==========================")

production = pd.read_csv(data_path / "production.csv")
production["production_date"] = pd.to_datetime(production["production_date"])

print("Minimum date:", production["production_date"].min().date())
print("Maximum date:", production["production_date"].max().date())
print("Unique dates:", production["production_date"].nunique())

print("\nExpected:")
print("Start date : 2025-08-24")
print("End date   : 2026-08-23")
print("Days       : 365")

print("\nExpected production rows:")
print("5 machines × 3 shifts × 365 days = 5475")

print("\nFuture dates:", (production["production_date"] > pd.Timestamp.today()).sum())

FULL DATASET VALIDATION
production.csv                 5475 rows
production_targets.csv         5475 rows
quality.csv                    5475 rows
sensors.csv                   10950 rows
downtime.csv                    431 rows
maintenance.csv                 173 rows
production_logs.csv            5475 rows
shifts.csv                        3 rows
employees.csv                     5 rows
machines.csv                      5 rows
inventory.csv                     5 rows
suppliers.csv                     5 rows

PRODUCTION DATE VALIDATION
Minimum date: 2025-08-24
Maximum date: 2026-08-23
Unique dates: 365

Expected:
Start date : 2025-08-24
End date   : 2026-08-23
Days       : 365

Expected production rows:
5 machines × 3 shifts × 365 days = 5475

Future dates: 0


In [ ]:
#  Missing values and duplicate validation

from pathlib import Path
import pandas as pd

data_path = Path("../data/generated")

csv_files = [
    "production.csv",
    "production_targets.csv",
    "quality.csv",
    "sensors.csv",
    "downtime.csv",
    "maintenance.csv",
    "production_logs.csv",
    "shifts.csv",
    "employees.csv",
    "machines.csv",
    "inventory.csv",
    "suppliers.csv"
]

print("MISSING VALUES + DUPLICATE VALIDATION")
print("=====================================")

all_passed = True

for file_name in csv_files:

    df = pd.read_csv(data_path / file_name)

    missing_values = int(df.isnull().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    missing_status = "PASS" if missing_values == 0 else "CHECK"
    duplicate_status = "PASS" if duplicate_rows == 0 else "CHECK"

    if missing_values != 0 or duplicate_rows != 0:
        all_passed = False

    print(f"\n{file_name}")
    print(f"  Missing values : {missing_values} -> {missing_status}")
    print(f"  Duplicate rows : {duplicate_rows} -> {duplicate_status}")

print("\n=====================================")

if all_passed:
    print("RESULT: ALL DATASETS PASSED")
else:
    print("RESULT: SOME DATASETS NEED CHECKING")

MISSING VALUES + DUPLICATE VALIDATION

production.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

production_targets.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

quality.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

sensors.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

downtime.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

maintenance.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

production_logs.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

shifts.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

employees.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

machines.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

inventory.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

suppliers.csv
  Missing values : 0 -> PASS
  Duplicate rows : 0 -> PASS

RESULT: ALL DATASETS PASSED


In [251]:
# Data relationships and business-rule validation

from pathlib import Path
import pandas as pd

data_path = Path("../data/generated")

production = pd.read_csv(data_path / "production.csv")
targets = pd.read_csv(data_path / "production_targets.csv")
quality = pd.read_csv(data_path / "quality.csv")
sensors = pd.read_csv(data_path / "sensors.csv")
downtime = pd.read_csv(data_path / "downtime.csv")
maintenance = pd.read_csv(data_path / "maintenance.csv")
machines = pd.read_csv(data_path / "machines.csv")

print("DATA RELATIONSHIP + BUSINESS RULE VALIDATION")
print("============================================")

checks = []

# 1. Production machines must exist in machines table
valid_machines = set(machines["machine_id"])
invalid_production_machines = (~production["machine_id"].isin(valid_machines)).sum()
checks.append(("Production machine IDs", invalid_production_machines == 0))

# 2. Production should have 5 machines × 3 shifts × 365 days
expected_production_rows = 5 * 3 * 365
checks.append(("Production row structure", len(production) == expected_production_rows))

# 3. Every production record should have a target record
checks.append(("Target row count", len(targets) == len(production)))

# 4. Every quality record should map to production
production_ids = set(production["production_id"])
invalid_quality_ids = (~quality["production_id"].isin(production_ids)).sum()
checks.append(("Quality production IDs", invalid_quality_ids == 0))

# 5. Sensor machines must exist
invalid_sensor_machines = (~sensors["machine_id"].isin(valid_machines)).sum()
checks.append(("Sensor machine IDs", invalid_sensor_machines == 0))

# 6. Downtime machines must exist
invalid_downtime_machines = (~downtime["machine_id"].isin(valid_machines)).sum()
checks.append(("Downtime machine IDs", invalid_downtime_machines == 0))

# 7. Maintenance machines must exist
invalid_maintenance_machines = (~maintenance["equipment_id"].isin(valid_machines)).sum()
checks.append(("Maintenance machine IDs", invalid_maintenance_machines == 0))

# 8. Rejected units cannot exceed produced units
invalid_rejections = (
    production["units_rejected"] > production["units_produced"]
).sum()
checks.append(("Rejected <= Produced", invalid_rejections == 0))

# 9. Production cannot be negative
invalid_production = (production["units_produced"] < 0).sum()
checks.append(("Production >= 0", invalid_production == 0))

# 10. Rejected units cannot be negative
invalid_rejected = (production["units_rejected"] < 0).sum()
checks.append(("Rejected units >= 0", invalid_rejected == 0))

# 11. Target quantity cannot be negative
invalid_targets = (targets["target_quantity"] < 0).sum()
checks.append(("Target quantity >= 0", invalid_targets == 0))

# 12. Downtime hours cannot be negative
invalid_downtime = (downtime["downtime_hours"] < 0).sum()
checks.append(("Downtime hours >= 0", invalid_downtime == 0))

# Print results
for check_name, passed in checks:
    print(f"{check_name:<35} -> {'PASS' if passed else 'CHECK'}")

print("\n============================================")

if all(passed for _, passed in checks):
    print("RESULT: ALL BUSINESS RULES PASSED")
else:
    print("RESULT: SOME BUSINESS RULES NEED CHECKING")

DATA RELATIONSHIP + BUSINESS RULE VALIDATION
Production machine IDs              -> PASS
Production row structure            -> PASS
Target row count                    -> PASS
Quality production IDs              -> PASS
Sensor machine IDs                  -> PASS
Downtime machine IDs                -> PASS
Maintenance machine IDs             -> PASS
Rejected <= Produced                -> PASS
Production >= 0                     -> PASS
Rejected units >= 0                 -> PASS
Target quantity >= 0                -> PASS
Downtime hours >= 0                 -> PASS

RESULT: ALL BUSINESS RULES PASSED


In [252]:
# Feature engineering validation

from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed")

feature_file = processed_path / "feature_engineered_data.csv"

feature_data_check = pd.read_csv(feature_file)

print("FEATURE ENGINEERING VALIDATION")
print("==============================")

print("Dataset shape:", feature_data_check.shape)

print("\nExpected:")
print("Rows   : 5475")
print("Columns: 42")

print("\nMissing values:", feature_data_check.isnull().sum().sum())
print("Duplicate rows:", feature_data_check.duplicated().sum())

print("\nDate range:")
feature_data_check["production_date"] = pd.to_datetime(
    feature_data_check["production_date"]
)
print("Minimum date:", feature_data_check["production_date"].min().date())
print("Maximum date:", feature_data_check["production_date"].max().date())

print("\nShift values:")
print(feature_data_check["shift"].value_counts())

print("\nMachine values:")
print(sorted(feature_data_check["machine_id"].unique()))

print("\nFeature columns:")
print("Total columns:", len(feature_data_check.columns))
print(feature_data_check.columns.tolist())

print("\n==============================")

if (
    feature_data_check.shape == (5475, 42)
    and feature_data_check.isnull().sum().sum() == 0
    and feature_data_check.duplicated().sum() == 0
):
    print("RESULT: FEATURE ENGINEERING PASSED")
else:
    print("RESULT: FEATURE ENGINEERING NEEDS CHECKING")

FEATURE ENGINEERING VALIDATION
Dataset shape: (5475, 42)

Expected:
Rows   : 5475
Columns: 42

Missing values: 0
Duplicate rows: 0

Date range:
Minimum date: 2025-08-24
Maximum date: 2026-08-23

Shift values:
shift
Morning      1825
Afternoon    1825
Night        1825
Name: count, dtype: int64

Machine values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Feature columns:
Total columns: 42
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintena

In [253]:
#  Validate lag and rolling features

feature_check = feature_data_check.copy()

feature_check["production_date"] = pd.to_datetime(
    feature_check["production_date"]
)

feature_check = feature_check.sort_values(
    ["machine_id", "production_date", "shift_encoded"]
).reset_index(drop=True)

print("LAG + ROLLING FEATURE VALIDATION")
print("================================")

# Check that the first record for each machine has no previous value
first_machine_rows = (
    feature_check.groupby("machine_id", as_index=False)
    .first()
)

print("\nFirst record of each machine:")
print(
    first_machine_rows[
        [
            "machine_id",
            "production_date",
            "previous_units_produced",
            "previous_rejection_rate"
        ]
    ]
)

# Check rolling features exist after the initial records
rolling_missing = feature_check[
    [
        "previous_units_produced",
        "previous_rejection_rate",
        "rolling_3_production_avg",
        "rolling_3_rejection_avg"
    ]
].isnull().sum()

print("\nMissing values in lag/rolling features:")
print(rolling_missing)

# Check whether previous production values are within a sensible range
invalid_previous_production = (
    feature_check["previous_units_produced"] < 0
).sum()

invalid_rolling_production = (
    feature_check["rolling_3_production_avg"] < 0
).sum()

# Check rejection rates
invalid_previous_rejection = (
    (feature_check["previous_rejection_rate"] < 0) |
    (feature_check["previous_rejection_rate"] > 100)
).sum()

invalid_rolling_rejection = (
    (feature_check["rolling_3_rejection_avg"] < 0) |
    (feature_check["rolling_3_rejection_avg"] > 100)
).sum()

print("\nRange checks:")
print(
    "Negative previous production :",
    invalid_previous_production
)
print(
    "Negative rolling production  :",
    invalid_rolling_production
)
print(
    "Invalid previous rejection % :",
    invalid_previous_rejection
)
print(
    "Invalid rolling rejection %  :",
    invalid_rolling_rejection
)

print("\n================================")

if (
    invalid_previous_production == 0
    and invalid_rolling_production == 0
    and invalid_previous_rejection == 0
    and invalid_rolling_rejection == 0
):
    print("RESULT: LAG + ROLLING FEATURES PASSED")
else:
    print("RESULT: LAG + ROLLING FEATURES NEED CHECKING")

LAG + ROLLING FEATURE VALIDATION

First record of each machine:
   machine_id production_date  previous_units_produced  \
0           1      2025-08-24                    969.0   
1           2      2025-08-24                    890.0   
2           3      2025-08-24                    781.0   
3           4      2025-08-24                    903.0   
4           5      2025-08-24                    988.0   

   previous_rejection_rate  
0                     5.74  
1                     6.51  
2                     2.13  
3                     2.90  
4                     2.27  

Missing values in lag/rolling features:
previous_units_produced     0
previous_rejection_rate     0
rolling_3_production_avg    0
rolling_3_rejection_avg     0
dtype: int64

Range checks:
Negative previous production : 0
Negative rolling production  : 0
Invalid previous rejection % : 0
Invalid rolling rejection %  : 0

RESULT: LAG + ROLLING FEATURES PASSED


In [254]:
# ML dataset and target leakage validation

print("ML DATASET + LEAKAGE VALIDATION")
print("================================")

# --------------------------------------------------
# Production model
# --------------------------------------------------

print("\n1. PRODUCTION MODEL")

print("Train shape:", X_train_production.shape)
print("Test shape :", X_test_production.shape)

print("Train date range:",
      train_quality_sensor["production_date"].min().date(),
      "to",
      train_quality_sensor["production_date"].max().date())

print("Test date range :",
      test_quality_sensor["production_date"].min().date(),
      "to",
      test_quality_sensor["production_date"].max().date())

print("Target:", "units_produced")
print("Target in features:",
      "units_produced" in X_train_production.columns)

print("Missing values:",
      X_train_production.isnull().sum().sum()
      + X_test_production.isnull().sum().sum())


# --------------------------------------------------
# Quality model
# --------------------------------------------------

print("\n2. QUALITY / REJECTION MODEL")

print("Train shape:", X_train_rejection.shape)
print("Test shape :", X_test_rejection.shape)

print("Target:", quality_rejection_target)

print("Target in features:",
      quality_rejection_target in X_train_rejection.columns)

print("Missing values:",
      X_train_rejection.isnull().sum().sum()
      + X_test_rejection.isnull().sum().sum())


# --------------------------------------------------
# Failure-risk model
# --------------------------------------------------

print("\n3. FAILURE-RISK MODEL")

print("Train shape:", X_train_failure.shape)
print("Test shape :", X_test_failure.shape)

print("Target:", "failure_risk")

print("Target in features:",
      "failure_risk" in X_train_failure.columns)

print("Missing values:",
      X_train_failure.isnull().sum().sum()
      + X_test_failure.isnull().sum().sum())


# --------------------------------------------------
# Time split validation
# --------------------------------------------------

print("\n4. TIME SPLIT VALIDATION")

production_split_valid = (
    train_quality_sensor["production_date"].max()
    < test_quality_sensor["production_date"].min()
)

failure_split_valid = (
    test_failure["production_date"].min()
    > train_failure["production_date"].max()
)

print("Production train/test separated:",
      production_split_valid)

print("Failure train/test separated:",
      failure_split_valid)


# --------------------------------------------------
# Result
# --------------------------------------------------

print("\n================================")

if (
    "units_produced" not in X_train_production.columns
    and quality_rejection_target not in X_train_rejection.columns
    and "failure_risk" not in X_train_failure.columns
    and production_split_valid
    and failure_split_valid
):
    print("RESULT: BASIC ML DATASET CHECK PASSED")
else:
    print("RESULT: ML DATASET NEEDS CHECKING")

ML DATASET + LEAKAGE VALIDATION

1. PRODUCTION MODEL
Train shape: (4380, 28)
Test shape : (1095, 28)
Train date range: 2025-08-24 to 2026-06-11
Test date range : 2026-06-12 to 2026-08-23
Target: units_produced
Target in features: False
Missing values: 0

2. QUALITY / REJECTION MODEL
Train shape: (4380, 28)
Test shape : (1095, 28)
Target: units_rejected
Target in features: False
Missing values: 0

3. FAILURE-RISK MODEL
Train shape: (4380, 30)
Test shape : (1095, 30)
Target: failure_risk
Target in features: False
Missing values: 0

4. TIME SPLIT VALIDATION
Production train/test separated: True
Failure train/test separated: True

RESULT: BASIC ML DATASET CHECK PASSED


In [255]:
#  Deeper failure-risk leakage check

print("FAILURE-RISK DEEP LEAKAGE CHECK")
print("================================")

outcome_related_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "total_maintenance_downtime",
    "maintenance_event_count",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
]

print("\nFeatures that may contain same-day outcome information:")
for feature in outcome_related_features:
    print(
        f"{feature:<35} -> "
        f"{'PRESENT' if feature in X_train_failure.columns else 'NOT PRESENT'}"
    )

print("\nFailure target definition:")
print("failure_risk = 1 when a downtime event exists")
print("for the same machine and production date.")

print("\n================================")

if any(feature in X_train_failure.columns for feature in outcome_related_features):
    print("RESULT: POTENTIAL DATA LEAKAGE IDENTIFIED")
    print("NOTE: Current failure model should be treated as a baseline,")
    print("not as a true pre-failure prediction model.")
else:
    print("RESULT: NO OBVIOUS SAME-DAY DOWNTIME FEATURES FOUND")

FAILURE-RISK DEEP LEAKAGE CHECK

Features that may contain same-day outcome information:
total_downtime_hours                -> PRESENT
downtime_event_count                -> PRESENT
average_downtime_hours              -> PRESENT
total_maintenance_downtime          -> PRESENT
maintenance_event_count             -> PRESENT
average_maintenance_downtime        -> PRESENT
total_maintenance_cost              -> PRESENT
completed_maintenance_count         -> PRESENT

Failure target definition:
failure_risk = 1 when a downtime event exists
for the same machine and production date.

RESULT: POTENTIAL DATA LEAKAGE IDENTIFIED
NOTE: Current failure model should be treated as a baseline,
not as a true pre-failure prediction model.


In [256]:
# Cross-check current ML model performance

print("CURRENT ML MODEL PERFORMANCE")
print("============================")

print("\n1. PRODUCTION PREDICTION")
print("MAE :", round(
    abs(y_test_production - production_linear_predictions).mean(), 4
))
print("RMSE:", round(
    ((y_test_production - production_linear_predictions) ** 2).mean() ** 0.5,
    4
))
print("R2  :", round(
    production_linear_model.score(
        X_test_production,
        y_test_production
    ),
    4
))

print("\n2. QUALITY / REJECTION PREDICTION")
print("MAE :", round(
    abs(y_test_rejection - final_quality_predictions).mean(), 4
))
print("RMSE:", round(
    ((y_test_rejection - final_quality_predictions) ** 2).mean() ** 0.5,
    4
))
print("R2  :", round(
    final_quality_rf_model.score(
        X_test_rejection,
        y_test_rejection
    ),
    4
))

print("\n3. FAILURE-RISK PREDICTION")
print("Accuracy :", round(
    (failure_predictions == y_test_failure).mean(),
    4
))
print("Precision:", round(
    ((failure_predictions == 1) & (y_test_failure == 1)).sum()
    / max((failure_predictions == 1).sum(), 1),
    4
))
print("Recall   :", round(
    ((failure_predictions == 1) & (y_test_failure == 1)).sum()
    / max((y_test_failure == 1).sum(), 1),
    4
))

print("\n4. ANOMALY DETECTION")
print("Total records :", len(anomaly_predictions))
print("Normal        :", (anomaly_predictions == 1).sum())
print("Anomalous     :", (anomaly_predictions == -1).sum())
print("Anomaly rate  :", round(
    (anomaly_predictions == -1).mean() * 100,
    2
), "%")

print("\n============================")
print("RESULT: PERFORMANCE BASELINE RECORDED")

CURRENT ML MODEL PERFORMANCE

1. PRODUCTION PREDICTION
MAE : 31.1659
RMSE: 39.0438
R2  : 0.7242

2. QUALITY / REJECTION PREDICTION
MAE : 16.0375
RMSE: 18.7429
R2  : 0.0112

3. FAILURE-RISK PREDICTION
Accuracy : 0.7105
Precision: 0.4105
Recall   : 0.3406

4. ANOMALY DETECTION
Total records : 5475
Normal        : 5201
Anomalous     : 274
Anomaly rate  : 5.0 %

RESULT: PERFORMANCE BASELINE RECORDED


In [257]:
# Create leakage-free failure-risk features

failure_features_clean = [
    "machine_id",
    "units_produced",
    "units_rejected",
    "target_quantity",
    "production_time_hours",

    # Historical production behaviour
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg",

    # Sensor information
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count",
    "temperature",
    "vibration",

    # Calendar / operating context
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded"
]

X_failure_clean = failure_data[failure_features_clean].copy()
y_failure_clean = failure_data["failure_risk"].copy()

print("LEAKAGE-FREE FAILURE-RISK DATASET")
print("================================")

print("Features:", len(failure_features_clean))
print("X shape :", X_failure_clean.shape)
print("y shape :", y_failure_clean.shape)

print("\nMissing values:", X_failure_clean.isnull().sum().sum())

print("\nFeatures used:")
for feature in failure_features_clean:
    print("-", feature)

print("\n================================")
print("RESULT: LEAKAGE-FREE FAILURE FEATURES CREATED")

LEAKAGE-FREE FAILURE-RISK DATASET
Features: 21
X shape : (5475, 21)
y shape : (5475,)

Missing values: 0

Features used:
- machine_id
- units_produced
- units_rejected
- target_quantity
- production_time_hours
- previous_units_produced
- previous_rejection_rate
- rolling_3_production_avg
- rolling_3_rejection_avg
- sensor_value_mean
- sensor_value_min
- sensor_value_max
- sensor_value_std
- sensor_reading_count
- temperature
- vibration
- production_day
- production_month
- production_day_of_week
- is_weekend
- shift_encoded

RESULT: LEAKAGE-FREE FAILURE FEATURES CREATED


In [258]:
#  Time-based split for leakage-free failure-risk model

failure_clean_data = failure_data.copy()

failure_clean_data["production_date"] = pd.to_datetime(
    failure_clean_data["production_date"]
)

failure_clean_data = failure_clean_data.sort_values(
    ["production_date", "machine_id", "shift_encoded"]
).reset_index(drop=True)

unique_failure_dates = (
    failure_clean_data["production_date"]
    .drop_duplicates()
    .sort_values()
)

split_index = int(len(unique_failure_dates) * 0.80)
failure_split_date = unique_failure_dates.iloc[split_index]

train_failure_clean = failure_clean_data[
    failure_clean_data["production_date"] < failure_split_date
].copy()

test_failure_clean = failure_clean_data[
    failure_clean_data["production_date"] >= failure_split_date
].copy()

X_train_failure_clean = train_failure_clean[
    failure_features_clean
].copy()

y_train_failure_clean = train_failure_clean[
    "failure_risk"
].copy()

X_test_failure_clean = test_failure_clean[
    failure_features_clean
].copy()

y_test_failure_clean = test_failure_clean[
    "failure_risk"
].copy()

print("LEAKAGE-FREE FAILURE-RISK TIME SPLIT")
print("===================================")

print("Split date:", failure_split_date.date())

print("\nTrain:")
print("Rows:", len(train_failure_clean))
print(
    "Date:",
    train_failure_clean["production_date"].min().date(),
    "to",
    train_failure_clean["production_date"].max().date()
)

print("\nTest:")
print("Rows:", len(test_failure_clean))
print(
    "Date:",
    test_failure_clean["production_date"].min().date(),
    "to",
    test_failure_clean["production_date"].max().date()
)

print("\nFeature count:", len(failure_features_clean))
print("Missing values:",
      X_train_failure_clean.isnull().sum().sum()
      + X_test_failure_clean.isnull().sum().sum())

print("\nTrain target distribution:")
print(y_train_failure_clean.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test_failure_clean.value_counts().sort_index())

print("\n===================================")

if (
    len(train_failure_clean) == 4380
    and len(test_failure_clean) == 1095
    and X_train_failure_clean.isnull().sum().sum() == 0
    and X_test_failure_clean.isnull().sum().sum() == 0
    and train_failure_clean["production_date"].max()
    < test_failure_clean["production_date"].min()
):
    print("RESULT: FAILURE-RISK TIME SPLIT PASSED")
else:
    print("RESULT: FAILURE-RISK TIME SPLIT NEEDS CHECKING")

LEAKAGE-FREE FAILURE-RISK TIME SPLIT
Split date: 2026-06-12

Train:
Rows: 4380
Date: 2025-08-24 to 2026-06-11

Test:
Rows: 1095
Date: 2026-06-12 to 2026-08-23

Feature count: 21
Missing values: 0

Train target distribution:
failure_risk
0    3363
1    1017
Name: count, dtype: int64

Test target distribution:
failure_risk
0    819
1    276
Name: count, dtype: int64

RESULT: FAILURE-RISK TIME SPLIT PASSED


In [259]:
# Train leakage-free failure-risk model

from sklearn.ensemble import RandomForestClassifier

failure_rf_clean_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

failure_rf_clean_model.fit(
    X_train_failure_clean,
    y_train_failure_clean
)

failure_clean_predictions = failure_rf_clean_model.predict(
    X_test_failure_clean
)

failure_clean_probabilities = (
    failure_rf_clean_model.predict_proba(
        X_test_failure_clean
    )[:, 1]
)

print("Leakage-free Failure Risk model trained successfully.")
print("Training rows:", len(X_train_failure_clean))
print("Test rows:", len(X_test_failure_clean))
print("Features:", len(failure_features_clean))

Leakage-free Failure Risk model trained successfully.
Training rows: 4380
Test rows: 1095
Features: 21


In [ ]:
# Evaluate leakage-free failure-risk model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

failure_clean_accuracy = accuracy_score(
    y_test_failure_clean,
    failure_clean_predictions
)

failure_clean_precision = precision_score(
    y_test_failure_clean,
    failure_clean_predictions,
    zero_division=0
)

failure_clean_recall = recall_score(
    y_test_failure_clean,
    failure_clean_predictions,
    zero_division=0
)

failure_clean_f1 = f1_score(
    y_test_failure_clean,
    failure_clean_predictions,
    zero_division=0
)

failure_clean_cm = confusion_matrix(
    y_test_failure_clean,
    failure_clean_predictions
)

print("LEAKAGE-FREE FAILURE-RISK MODEL")
print("================================")

print("Accuracy :", round(failure_clean_accuracy, 4))
print("Precision:", round(failure_clean_precision, 4))
print("Recall   :", round(failure_clean_recall, 4))
print("F1 Score :", round(failure_clean_f1, 4))

print("\nConfusion Matrix:")
print(failure_clean_cm)

print("\nOLD BASELINE")
print("Accuracy :", 0.7105)
print("Precision:", 0.4105)
print("Recall   :", 0.3406)
print("F1 Score :", 0.3723)

print("\n================================")
print("Comparison complete.")

LEAKAGE-FREE FAILURE-RISK MODEL
Accuracy : 0.7023
Precision: 0.3821
Recall   : 0.2935
F1 Score : 0.332

Confusion Matrix:
[[688 131]
 [195  81]]

OLD BASELINE
Accuracy : 0.7105
Precision: 0.4105
Recall   : 0.3406
F1 Score : 0.3723

Comparison complete.


In [261]:
# Save final leakage-free failure-risk model and predictions

import joblib

# Save the corrected model
joblib.dump(
    failure_rf_clean_model,
    "../models/failure_risk_model.pkl"
)

# Create prediction output
final_failure_predictions = test_failure_clean[
    [
        "production_id",
        "machine_id",
        "production_date",
        "shift"
    ]
].copy()

final_failure_predictions["actual_failure_risk"] = (
    y_test_failure_clean.values
)

final_failure_predictions["predicted_failure_risk"] = (
    failure_clean_predictions
)

final_failure_predictions["failure_probability"] = (
    failure_clean_probabilities
)

final_failure_predictions.to_csv(
    "../data/processed/failure_risk_predictions.csv",
    index=False
)

print("Final leakage-free Failure Risk model saved successfully.")
print("Final Failure Risk predictions saved successfully.")
print("Rows saved:", len(final_failure_predictions))

print("\nPreview:")
print(final_failure_predictions.head())

Final leakage-free Failure Risk model saved successfully.
Final Failure Risk predictions saved successfully.
Rows saved: 1095

Preview:
      production_id  machine_id production_date      shift  \
4380           4381           1      2026-06-12    Morning   
4381           4382           1      2026-06-12  Afternoon   
4382           4383           1      2026-06-12      Night   
4383           4384           2      2026-06-12    Morning   
4384           4385           2      2026-06-12  Afternoon   

      actual_failure_risk  predicted_failure_risk  failure_probability  
4380                    1                       0             0.417662  
4381                    1                       0             0.491855  
4382                    1                       0             0.466213  
4383                    0                       0             0.483818  
4384                    0                       0             0.391819  


In [262]:
# Final validation of all ML output files

from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed")

output_files = {
    "Production": "production_predictions.csv",
    "Quality": "quality_rejection_predictions.csv",
    "Failure Risk": "failure_risk_predictions.csv",
    "Anomaly Detection": "anomaly_detection_results.csv"
}

print("=" * 70)
print("FINAL ML OUTPUT VALIDATION")
print("=" * 70)

all_checks_passed = True

for model_name, file_name in output_files.items():

    file_path = processed_path / file_name

    print(f"\n{model_name}")
    print("-" * 70)

    # Check file exists
    if not file_path.exists():
        print("❌ File not found:", file_name)
        all_checks_passed = False
        continue

    print("✅ File exists")

    # Load file
    df = pd.read_csv(file_path)

    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    # Missing values
    missing_values = df.isnull().sum().sum()

    if missing_values == 0:
        print("✅ Missing values: 0")
    else:
        print("❌ Missing values:", missing_values)
        all_checks_passed = False

    # Duplicate rows
    duplicate_rows = df.duplicated().sum()

    if duplicate_rows == 0:
        print("✅ Duplicate rows: 0")
    else:
        print("❌ Duplicate rows:", duplicate_rows)
        all_checks_passed = False

    # Row count check
    if model_name != "Anomaly Detection":

        if len(df) == 1095:
            print("✅ Row count: 1095")
        else:
            print("❌ Unexpected row count:", len(df))
            all_checks_passed = False

    else:

        if len(df) == 5475:
            print("✅ Row count: 5475")
        else:
            print("❌ Unexpected row count:", len(df))
            all_checks_passed = False


print("\n" + "=" * 70)

if all_checks_passed:
    print("✅ BASIC ML OUTPUT VALIDATION PASSED")
else:
    print("❌ SOME ML OUTPUT CHECKS FAILED")

print("=" * 70)

FINAL ML OUTPUT VALIDATION

Production
----------------------------------------------------------------------
✅ File exists
Rows: 1095
Columns: ['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'predicted_units_produced']
✅ Missing values: 0
✅ Duplicate rows: 0
✅ Row count: 1095

Quality
----------------------------------------------------------------------
✅ File exists
Rows: 1095
Columns: ['production_id', 'machine_id', 'production_date', 'shift', 'actual_units_rejected', 'predicted_units_rejected', 'prediction_error']
✅ Missing values: 0
✅ Duplicate rows: 0
✅ Row count: 1095

Failure Risk
----------------------------------------------------------------------
✅ File exists
Rows: 1095
Columns: ['production_id', 'machine_id', 'production_date', 'shift', 'actual_failure_risk', 'predicted_failure_risk', 'failure_probability']
✅ Missing values: 0
✅ Duplicate rows: 0
✅ Row count: 1095

Anomaly Detection
-----------------------------------------------------------